# 05 - Week 1: one drafting head for every LLM (Hindi), kill test

**Generated by `scripts/build_kaggle_notebooks.py` (git=9f1203decd0b src+consts_sha256=d875cf2683fc). Do not edit; change `src/lgmtp/` and rebuild.**

What it does, end to end (details in `PLAN_phase2_indic_universal_mtp.md`, W1):
1. Four small LLMs with different tokenizers (Qwen2.5-1.5B, Gemma-3-1B, Llama-3.2-1B, Sarvam-1) translate FLORES+ and IITB English sentences into Hindi; for every output token we keep the model's top-20 next-token distribution.
2. Cross-LLM agreement on Hindi words: are word *starts* model-specific and word *continuations* model-independent?
3. A small head that reads only text (the committed Hindi, the prompt, and the top-20 candidates written out as strings) and drafts the next 3 words in one pass. Trained leave-one-model-out; compared with a text-only ablation, an in-model head and training-free baselines.
4. The key number: the head trained only on the small LLMs, applied **zero-shot** to the Phase-1 Llama-3-8B traces (exact simulation), against Phase-1 Entropy (1.452x).
5. A GO / NO GO verdict in `SUMMARY.md`.

**Kaggle settings:** Accelerator **GPU T4 x2** - Internet **On** - Secret **`HF_TOKEN`** (optional but recommended: FLORES+ and the official Gemma / Llama repos are gated; ungated mirrors are tried automatically).

**Strongly recommended input: the Phase-1 Llama-3-8B traces.** The notebook searches all of `/kaggle/input` for `traces_dev.npz` / `traces_devtest.npz` (or the `traces_v2_dev.zip` / `traces_v2_devtest.zip` files, with `target_outputs_*.jsonl` and `timing.json`). Two ways to attach them:
- *Add Input > Your Work > Notebooks > 01 - Traces v2*, picking the version that ran `devtest`, and if Kaggle allows it also the version that ran `dev`;
- or (works always) download `traces_v2_dev.zip` and `traces_v2_devtest.zip` from those two versions' Output tabs, upload both as one private Dataset, and attach it.

`devtest` is required for the main result; `dev` adds the in-model 8B reference heads (needed for the recovery criterion). Without any traces the notebook re-generates Llama-3-8B outputs itself (`COLLECT_8B_IF_MISSING`, adds ~2-3 h) and cannot recompute the Entropy baseline on identical sentences.

**Run:** *Save Version > Save & Run All (Commit)*. Expected time with the defaults: ~3-4 h (more without the Phase-1 input). The log prints an estimate after the first batch of each model.

**Outputs:** `/kaggle/working/week1_results.zip` (tables, `SUMMARY.md`, `verdict.json`, log) -> unzip into `results/week1/` in the repo. `/kaggle/working/week1_data.zip` (all collected outputs with top-20, trained heads) -> keep for weeks 2+.

**If the run dies:** add the failed version's output as an input (*Add Input → Your Work → this notebook's output*), or upload the downloaded output as a private Dataset (zipped or not, with or without the `week1/` folder), and run again. With `RESUME_FROM="auto"` (the default) it finds that run (the folder holding `data/sources.json`), copies it into place and skips every finished stage and every finished model split. A crash writes its traceback to `results/log.txt` and still saves `week1_results.zip`.

In [ ]:
!pip -q install -U "transformers==5.22.1" accelerate datasets huggingface_hub regex

In [ ]:
# === lgmtp source, generated by scripts/build_kaggle_notebooks.py: DO NOT EDIT ===
import os, sys, importlib
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')
PKG = '/kaggle/working/lgmtp' if os.path.isdir('/kaggle/working') else os.path.abspath('lgmtp')
os.makedirs(PKG, exist_ok=True)
LGMTP_SOURCES = {
    '__init__': '"""lgmtp: linguistically guided multi-token prediction (CS769)."""\n',
    'boundaries': '"""Token bytes, Devanagari akshara units, word boundaries and per-token cut labels.\n\nExtracted from ``notebooks/legacy/test1_hindi_boundaries.ipynb`` (cell 2). The logic of the\nlegacy functions is unchanged; only type hints / docstrings were added, plus ``token_labels``.\n\nFixes bug B4 (never decode single byte-level tokens): everything works on token *bytes* and\nbyte offsets, and text is only decoded for whole sequences.\n"""\nfrom __future__ import annotations\n\nimport json\nimport re\nimport unicodedata\nfrom collections import Counter\nfrom itertools import accumulate\nfrom typing import Iterable, Sequence\n\nimport numpy as np\n\n# --------------------------------------------------------------------------- #\n# token -> bytes\n# --------------------------------------------------------------------------- #\n\n\ndef _bytes_to_unicode() -> dict[int, str]:\n    """GPT-2 byte<->unicode table (used by Llama-3 and Qwen byte-level BPE)."""\n    bs = (list(range(ord("!"), ord("~") + 1))\n          + list(range(ord("¡"), ord("¬") + 1))\n          + list(range(ord("®"), ord("ÿ") + 1)))\n    cs = bs[:]\n    n = 0\n    for b in range(256):\n        if b not in bs:\n            bs.append(b)\n            cs.append(256 + n)\n            n += 1\n    return dict(zip(bs, [chr(c) for c in cs]))\n\n\nBYTE_DECODER: dict[str, int] = {v: k for k, v in _bytes_to_unicode().items()}\n_HEX_RE = re.compile(r"^<0x([0-9A-Fa-f]{2})>$")\n\n\ndef token_to_bytes(tok: str) -> bytes:\n    """Raw bytes of a vocabulary token string (as returned by ``convert_ids_to_tokens``)."""\n    m = _HEX_RE.match(tok)\n    if m:  # SentencePiece byte-fallback token\n        return bytes([int(m.group(1), 16)])\n    if all(ch in BYTE_DECODER for ch in tok):  # GPT-2 style byte-level BPE\n        return bytes(BYTE_DECODER[ch] for ch in tok)\n    return tok.replace("▁", " ").encode("utf-8")  # SentencePiece unicode piece\n\n\nclass HFTok:\n    """Thin wrapper: text -> list of token byte-strings."""\n\n    def __init__(self, name_or_tokenizer):\n        if isinstance(name_or_tokenizer, str):\n            from transformers import AutoTokenizer\n            self.name = name_or_tokenizer\n            self.tok = AutoTokenizer.from_pretrained(name_or_tokenizer)\n        else:\n            self.name = "custom"\n            self.tok = name_or_tokenizer\n\n    def to_bytes(self, text: str) -> list[bytes]:\n        ids = self.tok(text, add_special_tokens=False)["input_ids"]\n        return [token_to_bytes(t) for t in self.tok.convert_ids_to_tokens(ids)]\n\n    def ids_to_bytes(self, ids: Sequence[int]) -> list[bytes]:\n        """Byte-strings for a list of token ids (no decoding of individual tokens)."""\n        return [token_to_bytes(t) for t in self.tok.convert_ids_to_tokens(list(ids))]\n\n\nQWEN2_PRETOK_REGEX = (r"(?i:\'s|\'t|\'re|\'ve|\'m|\'ll|\'d)|[^\\r\\n\\p{L}\\p{N}]?\\p{L}+|\\p{N}"\n                      r"| ?[^\\s\\p{L}\\p{N}]+[\\r\\n]*|\\s*[\\r\\n]+|\\s+(?!\\S)|\\s+")\n\n\ndef legacy_qwen2_pretok_tokenizer(name: str = "Qwen/Qwen3.5-0.8B"):\n    """``name``\'s tokenizer.json with its split regex replaced by the older Qwen2 one (``\\\\p{L}+``,\n    no ``\\\\p{M}``), which splits before every combining mark.\n\n    Only used to explain the legacy Test 1 Qwen number (3.33 tokens/word): the legacy run loaded\n    the tokenizer this way. The model\'s real tokenizer (``[\\\\p{L}\\\\p{M}]+``) gives ~2.16.\n    """\n    from huggingface_hub import hf_hub_download\n    from tokenizers import Tokenizer\n    from transformers import PreTrainedTokenizerFast\n    j = json.load(open(hf_hub_download(name, "tokenizer.json"), encoding="utf-8"))\n    j["pre_tokenizer"]["pretokenizers"][0]["pattern"]["Regex"] = QWEN2_PRETOK_REGEX\n    return PreTrainedTokenizerFast(tokenizer_object=Tokenizer.from_str(json.dumps(j)))\n\n\n# --------------------------------------------------------------------------- #\n# akshara + word boundaries\n# --------------------------------------------------------------------------- #\n\n_C = "क-हक़-य़ॸ-ॿ"      # consonants\n_VIRAMA = "्"\n_NUKTA = "़"\n_ZW = "\u200c\u200d"                                 # ZWNJ / ZWJ\n_IVOW = "ऄ-औॠॡॲ-ॷ"     # independent vowels\n_MATRA = "ऺऻा-ौॎॏॕ-ॗॢॣ"\n_MOD = "ऀ-ः॑-॔"                  # candrabindu, anusvara, visarga, stress marks\n\n_AKSHARA_RE = re.compile(\n    rf"[{_C}]{_NUKTA}?(?:{_VIRAMA}[{_ZW}]?[{_C}]{_NUKTA}?)*(?:{_VIRAMA}[{_ZW}]?|[{_MATRA}]*)[{_MOD}]*"\n    rf"|[{_IVOW}]{_NUKTA}?[{_MATRA}]*[{_MOD}]*"\n)\n\n\ndef akshara_units(text: str) -> list[tuple[int, int]]:\n    """Return list of (start, end) char spans. Non-Devanagari chars are single units\n    with any following combining marks / ZW joiners attached."""\n    units, i, n = [], 0, len(text)\n    while i < n:\n        m = _AKSHARA_RE.match(text, i)\n        if m and m.end() > i:\n            j = m.end()\n        else:\n            j = i + 1\n            while j < n and (unicodedata.category(text[j]).startswith("M") or text[j] in "\u200c\u200d"):\n                j += 1\n        units.append((i, j))\n        i = j\n    return units\n\n\ndef is_word_boundary(text: str, c: int) -> bool:\n    """True if char offset ``c`` is a word boundary (text edge, whitespace or punctuation)."""\n    n = len(text)\n    if c <= 0 or c >= n:\n        return True\n    a, b = text[c - 1], text[c]\n    if a.isspace() or b.isspace():\n        return True\n    if unicodedata.category(a).startswith("P") or unicodedata.category(b).startswith("P"):\n        return True\n    return False\n\n\nEND, WORD, MIDWORD, MIDAKS, MIDUTF8 = "end", "word_boundary", "mid_word", "mid_akshara", "mid_utf8"\nCUT_CATEGORIES = (END, WORD, MIDWORD, MIDAKS, MIDUTF8)\nTAG = {END: "E", WORD: "W", MIDWORD: "w", MIDAKS: "a", MIDUTF8: "u"}\n\n\ndef classify(b: int, total: int, byte2char: dict[int, int], text: str, aks: set[int]) -> str:\n    """Category of a cut at byte offset ``b``.\n\n    byte2char maps byte offsets that start a char (plus ``total``) to char offsets;\n    aks is the set of char offsets that are akshara-unit boundaries.\n    """\n    if b >= total:\n        return END\n    c = byte2char.get(b)\n    if c is None:\n        return MIDUTF8\n    if c not in aks:\n        return MIDAKS\n    return WORD if is_word_boundary(text, c) else MIDWORD\n\n\ndef sentence_features(tb: Sequence[bytes]) -> dict | None:\n    """tb: list of token byte-strings. Returns None if bytes are not valid UTF-8."""\n    full = b"".join(tb)\n    try:\n        text = full.decode("utf-8")\n    except UnicodeDecodeError:\n        return None\n    byte2char, pos = {}, 0\n    for ci, ch in enumerate(text):\n        byte2char[pos] = ci\n        pos += len(ch.encode("utf-8"))\n    byte2char[pos] = len(text)\n    units = akshara_units(text)\n    aks = {0} | {e for _, e in units}\n    ends = list(accumulate(len(b) for b in tb))\n    starts = [0] + ends[:-1]\n    total = len(full)\n    return {\n        "text": text,\n        "units": units,\n        "cut": [classify(e, total, byte2char, text, aks) for e in ends],\n        "start": [classify(s, total, byte2char, text, aks) for s in starts],\n    }\n\n\n# --------------------------------------------------------------------------- #\n# per-token gold labels (analysis only; never read by policies)\n# --------------------------------------------------------------------------- #\n\n\ndef _valid_utf8_prefix_len(full: bytes) -> int:\n    """Length of the longest prefix of ``full`` that is valid UTF-8, allowing only a\n    truncated character at the very end (e.g. generation stopped at MAX_NEW)."""\n    for cut in range(len(full), max(len(full) - 4, -1), -1):\n        try:\n            full[:cut].decode("utf-8")\n            return cut\n        except UnicodeDecodeError:\n            continue\n    raise ValueError("token bytes are not valid UTF-8 (invalid sequence before the end)")\n\n\ndef token_labels(token_bytes: Sequence[bytes],\n                 special: Sequence[bool] | None = None) -> dict[str, np.ndarray]:\n    """Gold labels for each token t of a target sequence (analysis only, never read by policies).\n\n    end_cut[t]      : category of the cut AFTER token t (END/WORD/MIDWORD/MIDAKS/MIDUTF8)\n    start_cut[t]    : category of the cut BEFORE token t\n    word_initial[t] : token t starts a new word (start_cut in {WORD, END})\n    pos_in_word[t]  : 0 for word-initial tokens, then 1, 2, ...\n    word_id[t]      : index of the word the token belongs to\n    char_span[t]    : (start_char, end_char) in the decoded text, shape (n, 2)\n\n    ``special[t]`` marks special/control tokens (e.g. ``<|eot_id|>``). They are stripped before\n    labelling and get ``END`` for both cuts, ``word_initial=True``, ``pos_in_word=0``, a fresh\n    ``word_id`` and an empty char span at the end of the text. If ``special`` is None, no token is\n    treated as special.\n\n    A trailing incomplete UTF-8 character (output truncated mid-character) is tolerated: cuts at\n    its start or inside it are ``MIDUTF8`` (what follows is unknown). Invalid bytes anywhere else\n    raise ``ValueError``.\n    """\n    n = len(token_bytes)\n    special = [False] * n if special is None else [bool(s) for s in special]\n    if len(special) != n:\n        raise ValueError("special must have the same length as token_bytes")\n    keep = [t for t in range(n) if not special[t]]\n    tb = [token_bytes[t] for t in keep]\n\n    full = b"".join(tb)\n    valid = _valid_utf8_prefix_len(full)\n    text = full[:valid].decode("utf-8")\n    full_total = len(full)\n\n    byte2char, pos = {}, 0\n    char_of_byte = np.zeros(full_total + 1, dtype=np.int64)  # char containing byte b\n    for ci, ch in enumerate(text):\n        byte2char[pos] = ci\n        nb = len(ch.encode("utf-8"))\n        char_of_byte[pos:pos + nb] = ci\n        pos += nb\n    byte2char[pos] = len(text)\n    char_of_byte[pos:] = len(text)  # trailing partial char maps past the text\n    aks = {0} | {e for _, e in akshara_units(text)}\n\n    def cat(b: int) -> str:\n        if b >= full_total:\n            return END\n        if b >= valid:  # inside the trailing truncated char\n            return MIDUTF8\n        return classify(b, valid, byte2char, text, aks)\n\n    ends = list(accumulate(len(b) for b in tb))\n    starts = [0] + ends[:-1]\n\n    end_cut = np.full(n, END, dtype=object)\n    start_cut = np.full(n, END, dtype=object)\n    char_span = np.full((n, 2), len(text), dtype=np.int64)\n    for j, t in enumerate(keep):\n        s, e = starts[j], ends[j]\n        end_cut[t] = cat(e)\n        start_cut[t] = cat(s)\n        cs = int(char_of_byte[min(s, full_total)])\n        ce = int(char_of_byte[e - 1]) + 1 if e > s else cs\n        char_span[t] = (cs, min(ce, len(text)) if e <= valid else len(text))\n\n    word_initial = np.array([sc in (WORD, END) for sc in start_cut], dtype=bool)\n    pos_in_word = np.zeros(n, dtype=np.int64)\n    word_id = np.zeros(n, dtype=np.int64)\n    wid = -1\n    for t in range(n):\n        if word_initial[t] or t == 0:\n            wid += 1\n            pos_in_word[t] = 0\n        else:\n            pos_in_word[t] = pos_in_word[t - 1] + 1\n        word_id[t] = wid\n    return dict(end_cut=end_cut, start_cut=start_cut, word_initial=word_initial,\n                pos_in_word=pos_in_word, word_id=word_id, char_span=char_span)\n\n\n# --------------------------------------------------------------------------- #\n# corpus analysis\n# --------------------------------------------------------------------------- #\n\n\ndef analyze(token_bytes_per_sentence: Iterable[Sequence[bytes]], ks: Sequence[int]):\n    """Corpus statistics of token cuts and of fixed-k draft windows. Returns (tot, per_k)."""\n    tot = Counter()\n    per_k = {k: Counter() for k in ks}\n    for tb in token_bytes_per_sentence:\n        f = sentence_features(tb)\n        if f is None:\n            tot["skipped_invalid_utf8"] += 1\n            continue\n        n = len(tb)\n        if n < 2:\n            tot["skipped_short"] += 1\n            continue\n        text, cut, start = f["text"], f["cut"], f["start"]\n        tot["sentences"] += 1\n        tot["tokens"] += n\n        tot["words"] += len(text.split())\n        tot["chars"] += len(text)\n        tot["aksharas"] += len(f["units"])\n        tot["starts_mid_utf8"] += sum(1 for s in start if s == MIDUTF8)\n        tot["starts_mid_akshara"] += sum(1 for s in start if s == MIDAKS)\n        for j in range(n - 1):  # internal cuts only (a token follows)\n            tot["cuts"] += 1\n            tot["cut_" + cut[j]] += 1\n\n        wb = [c in (WORD, END) for c in cut]\n        pw = [0] + list(accumulate(1 if x else 0 for x in wb))\n        last_wb, lw = [], -1\n        for j in range(n):\n            if wb[j]:\n                lw = j\n            last_wb.append(lw)\n        next_wb, nw = [0] * n, n - 1\n        for j in range(n - 1, -1, -1):\n            if wb[j]:\n                nw = j\n            next_wb[j] = nw\n\n        for k in ks:\n            c = per_k[k]\n            for s in range(0, n - k):\n                e = s + k - 1\n                cat = cut[e]\n                c["windows"] += 1\n                c[cat] += 1\n                done = pw[e + 1] - pw[s]\n                c["completed_words"] += done\n                if done == 0:\n                    c["zero_word"] += 1\n                if cat not in (WORD, END):\n                    c["mid_n"] += 1\n                    c["fwd_sum"] += next_wb[e] - e\n                    c["back_sum"] += e - last_wb[e]\n    return tot, per_k\n\n\ndef pct(a: float, b: float) -> float:\n    return 100.0 * a / b if b else float("nan")\n\n\ndef summarize(name: str, tot: Counter, per_k: dict, ks: Sequence[int]) -> dict:\n    """Turn ``analyze`` counters into the Test 1 summary dict."""\n    S = {\n        "tokenizer": name,\n        "sentences": tot["sentences"],\n        "tokens": tot["tokens"],\n        "tokens_per_word": tot["tokens"] / max(tot["words"], 1),\n        "tokens_per_akshara": tot["tokens"] / max(tot["aksharas"], 1),\n        "tokens_per_char": tot["tokens"] / max(tot["chars"], 1),\n        "pct_tokens_start_mid_akshara": pct(tot["starts_mid_akshara"], tot["tokens"]),\n        "pct_tokens_start_mid_utf8": pct(tot["starts_mid_utf8"], tot["tokens"]),\n        "pct_cuts_word_boundary": pct(tot["cut_" + WORD] + tot["cut_" + END], tot["cuts"]),\n        "pct_cuts_mid_word": pct(tot["cut_" + MIDWORD], tot["cuts"]),\n        "pct_cuts_mid_akshara": pct(tot["cut_" + MIDAKS], tot["cuts"]),\n        "pct_cuts_mid_utf8": pct(tot["cut_" + MIDUTF8], tot["cuts"]),\n        "skipped_invalid_utf8": tot["skipped_invalid_utf8"],\n        "per_k": [],\n    }\n    for k in ks:\n        c = per_k[k]\n        w = c["windows"]\n        S["per_k"].append({\n            "k": k,\n            "windows": w,\n            "pct_end_at_word_boundary": pct(c[WORD] + c[END], w),\n            "pct_end_mid_word": pct(c[MIDWORD], w),\n            "pct_end_mid_akshara": pct(c[MIDAKS], w),\n            "pct_end_mid_utf8": pct(c[MIDUTF8], w),\n            "pct_end_anywhere_inside_word": 100.0 - pct(c[WORD] + c[END], w) if w else float("nan"),\n            "mean_words_completed": c["completed_words"] / w if w else float("nan"),\n            "pct_drafts_completing_zero_words": pct(c["zero_word"], w),\n            "mean_tokens_forward_to_boundary": c["fwd_sum"] / c["mid_n"] if c["mid_n"] else float("nan"),\n            "mean_tokens_back_to_boundary": c["back_sum"] / c["mid_n"] if c["mid_n"] else float("nan"),\n        })\n    return S\n\n\ndef print_summary(S: dict) -> None:\n    print(f"\\n=== {S[\'tokenizer\']} ===")\n    print(f"sentences={S[\'sentences\']}  tokens={S[\'tokens\']}  "\n          f"tokens/word={S[\'tokens_per_word\']:.2f}  tokens/akshara={S[\'tokens_per_akshara\']:.2f}  "\n          f"tokens/char={S[\'tokens_per_char\']:.2f}")\n    print(f"tokens that START mid-akshara: {S[\'pct_tokens_start_mid_akshara\']:.1f}%   "\n          f"mid-UTF-8: {S[\'pct_tokens_start_mid_utf8\']:.1f}%")\n    print(f"all token cuts -> word boundary {S[\'pct_cuts_word_boundary\']:.1f}% | mid-word {S[\'pct_cuts_mid_word\']:.1f}% | "\n          f"mid-akshara {S[\'pct_cuts_mid_akshara\']:.1f}% | mid-UTF-8 {S[\'pct_cuts_mid_utf8\']:.1f}%")\n    if S["skipped_invalid_utf8"]:\n        print(f"WARNING: skipped {S[\'skipped_invalid_utf8\']} sentences (token bytes not valid UTF-8)")\n    hdr = ("k", "@word", "mid-word", "mid-aks", "mid-utf8", "inside-any", "words/draft", "0-word%", "fwd", "back")\n    print("\\n" + " ".join(f"{h:>11}" for h in hdr))\n    for r in S["per_k"]:\n        print(" ".join(f"{v:>11}" for v in (\n            r["k"], f"{r[\'pct_end_at_word_boundary\']:.1f}", f"{r[\'pct_end_mid_word\']:.1f}",\n            f"{r[\'pct_end_mid_akshara\']:.1f}", f"{r[\'pct_end_mid_utf8\']:.1f}",\n            f"{r[\'pct_end_anywhere_inside_word\']:.1f}", f"{r[\'mean_words_completed\']:.2f}",\n            f"{r[\'pct_drafts_completing_zero_words\']:.1f}",\n            f"{r[\'mean_tokens_forward_to_boundary\']:.2f}", f"{r[\'mean_tokens_back_to_boundary\']:.2f}")))\n    print("(columns 2-6 are % of k-token drafts; fwd/back = avg tokens to move to the nearest word boundary "\n          "when the draft ends inside a word)")\n\n\n# --------------------------------------------------------------------------- #\n# examples for slides\n# --------------------------------------------------------------------------- #\n\n\ndef _show(b: bytes) -> str:\n    try:\n        return b.decode("utf-8").replace(" ", "␣")  # visible space\n    except UnicodeDecodeError:\n        return "<" + b.hex() + ">"\n\n\ndef render_example(tb: Sequence[bytes], k: int) -> str:\n    """Render a tokenised sentence with fixed-k draft ends tagged by cut category."""\n    f = sentence_features(tb)\n    if f is None:\n        return "(invalid utf-8)"\n    out = []\n    for i, b in enumerate(tb):\n        out.append(_show(b))\n        if i < len(tb) - 1:\n            out.append(f" ‖[{TAG[f[\'cut\'][i]]}] " if (i + 1) % k == 0 else " | ")\n    return "".join(out)\n\n\nEXAMPLE_LEGEND = ("Legend: | token boundary,  ‖ end of a fixed-k draft,  tag = what the draft ends on: "\n                  "W=word boundary, w=inside word, a=inside akshara, u=inside UTF-8 char, ␣ = space\\n")\n\n# --------------------------------------------------------------------------- #\n# data\n# --------------------------------------------------------------------------- #\n\nDEMO = [\n    "भारत एक विशाल और विविधताओं से भरा देश है।",\n    "विद्यार्थियों ने विद्यालय में सूर्योदय के समय प्रार्थना की।",\n    "मौसम विभाग ने कल भारी बारिश की चेतावनी जारी की है।",\n    "वैज्ञानिकों का मानना है कि जलवायु परिवर्तन का प्रभाव कृषि पर पड़ेगा।",\n    "उसने अपने दोस्तों को खाने पर बुलाया और सबने मिलकर खाना खाया।",\n]\n\n\ndef load_sentences(path: str, field: str) -> list[str]:\n    out = []\n    with open(path, encoding="utf-8") as fh:\n        if path.endswith(".jsonl"):\n            for line in fh:\n                if line.strip():\n                    out.append(json.loads(line)[field])\n        else:\n            out = [ln.rstrip("\\n") for ln in fh]\n    return [unicodedata.normalize("NFC", s.strip()) for s in out if s.strip()]\n\n\ndef load_flores_hindi(split: str = "devtest") -> list[str]:\n    """FLORES+ hin_Deva sentences (NFC, stripped), as used in Test 1."""\n    from datasets import load_dataset\n    ds = load_dataset("openlanguagedata/flores_plus", "hin_Deva", split=split)\n    return [unicodedata.normalize("NFC", s.strip()) for s in ds["text"] if s.strip()]\n',
    'vocab': '"""Boolean masks over the full vocabulary, built from token *bytes* (``boundaries.token_to_bytes``).\n\nMasks (all shape ``(len(tok),)``, dtype bool):\n\n- ``starts_space``: first byte in ``b" \\\\n\\\\t"``.\n- ``starts_punct``: first decoded char has Unicode category ``P*`` (``।``, ``,``, ``.``, ``"``, ...).\n- ``starts_mark``: first decoded char has category ``M*`` (matra, virama, nukta, anusvara,\n  candrabindu, visarga, ...) or is ZWJ / ZWNJ.\n- ``starts_cont``: first byte is a UTF-8 continuation byte (0x80-0xBF): the token starts mid-character.\n  Such a token is never ``starts_mark`` / ``starts_punct``.\n- ``is_special``: special / control tokens (all registered special tokens and eos ids).\n\nA token whose bytes start with a lead byte but do not contain a complete first character\n(e.g. ``b"\\\\xe0\\\\xa4"``) has an unknown first char and is in none of space/punct/mark/cont.\nSpecial tokens are only ``is_special``.\n"""\nfrom __future__ import annotations\n\nimport unicodedata\nfrom dataclasses import dataclass, field\nfrom typing import Iterable\n\nimport numpy as np\n\nfrom .boundaries import token_to_bytes\n\nZW = "\u200c\u200d"\nMASK_NAMES = ("starts_space", "starts_punct", "starts_mark", "starts_cont", "is_special")\n\n\ndef first_char(b: bytes) -> str:\n    """First complete character of ``b`` (\'\' if none). Continuation-byte starts give \'\'."""\n    if not b or 0x80 <= b[0] <= 0xBF:\n        return ""\n    return b[:4].decode("utf-8", errors="ignore")[:1]\n\n\ndef classify_token_bytes(b: bytes) -> dict[str, bool]:\n    """Mask flags for one non-special token given its raw bytes."""\n    out = dict(starts_space=False, starts_punct=False, starts_mark=False, starts_cont=False,\n               is_special=False)\n    if not b:\n        return out\n    if b[0] in b" \\n\\t":\n        out["starts_space"] = True\n        return out\n    if 0x80 <= b[0] <= 0xBF:\n        out["starts_cont"] = True\n        return out\n    ch = first_char(b)\n    if not ch:\n        return out\n    cat = unicodedata.category(ch)\n    if cat.startswith("P"):\n        out["starts_punct"] = True\n    elif cat.startswith("M") or ch in ZW:\n        out["starts_mark"] = True\n    return out\n\n\n@dataclass\nclass VocabMasks:\n    starts_space: np.ndarray\n    starts_punct: np.ndarray\n    starts_mark: np.ndarray\n    starts_cont: np.ndarray\n    is_special: np.ndarray\n    eos_ids: list[int] = field(default_factory=list)\n\n    @property\n    def size(self) -> int:\n        return len(self.starts_space)\n\n    @property\n    def is_eos(self) -> np.ndarray:\n        m = np.zeros(self.size, dtype=bool)\n        m[[i for i in self.eos_ids if i < self.size]] = True\n        return m\n\n    @property\n    def starts_newword(self) -> np.ndarray:\n        """Tokens that end the current word if they come next: space, punctuation or eos."""\n        return self.starts_space | self.starts_punct | self.is_eos\n\n    def padded(self, name: str, size: int) -> np.ndarray:\n        """Mask padded with False to ``size`` (model logits can be wider than ``len(tok)``)."""\n        m = getattr(self, name)\n        if size < len(m):\n            raise ValueError("size smaller than vocabulary")\n        return np.concatenate([m, np.zeros(size - len(m), dtype=bool)])\n\n    def as_dict(self) -> dict[str, np.ndarray]:\n        return {k: getattr(self, k) for k in MASK_NAMES}\n\n\ndef special_ids(tok) -> set[int]:\n    """All special/control token ids of a Hugging Face tokenizer."""\n    ids = set(getattr(tok, "all_special_ids", []) or [])\n    for i, at in (getattr(tok, "added_tokens_decoder", {}) or {}).items():\n        if getattr(at, "special", False):\n            ids.add(int(i))\n    return ids\n\n\ndef build_masks(tok, eos_ids: Iterable[int] = ()) -> VocabMasks:\n    """Build all masks for a Hugging Face tokenizer (size ``len(tok)``)."""\n    n = len(tok)\n    eos_ids = sorted(set(int(i) for i in eos_ids))\n    spec = special_ids(tok) | set(eos_ids)\n    toks = tok.convert_ids_to_tokens(list(range(n)))\n    arrs = {k: np.zeros(n, dtype=bool) for k in MASK_NAMES}\n    for i, t in enumerate(toks):\n        if i in spec or t is None:\n            arrs["is_special"][i] = True\n            continue\n        for k, v in classify_token_bytes(token_to_bytes(t)).items():\n            if v:\n                arrs[k][i] = True\n    return VocabMasks(**arrs, eos_ids=eos_ids)\n\n\ndef token_bytes_list(tok, ids) -> list[bytes]:\n    """Raw bytes of each id (special tokens give their literal text bytes)."""\n    return [token_to_bytes(t) for t in tok.convert_ids_to_tokens([int(i) for i in ids])]\n',
    'traces': '"""Trace schema (one target sentence), compressed ``.npz`` save/load, gold-label attachment.\n\nIndex convention (PLAN §3): index ``t`` refers to target token ``y[t]``; every draft feature at\n``t`` is a function of ``q_t``, the draft distribution over ``y[t]`` computed teacher-forced on\n``prompt + y[:t]``.\n\nStorage: per-position arrays of all sentences are concatenated into flat arrays and stored with\nan ``offsets`` array (sentence i occupies ``offsets[i]:offsets[i+1]``). ``meta`` is stored as a\nJSON string.\n"""\nfrom __future__ import annotations\n\nimport json\nfrom dataclasses import dataclass, field, fields\nfrom pathlib import Path\n\nimport numpy as np\n\n# per-position 1-D arrays saved to disk (in this order)\nSCALAR_FIELDS = ("target_ids", "draft_argmax", "H", "top1", "top2", "p_target",\n                 "p_space", "p_punct", "p_eos", "p_newword", "p_mark", "p_cont",\n                 "target_logp_of_draft_argmax")\n# per-position (n, K) arrays\nTOPK_FIELDS = ("draft_topk_ids", "draft_topk_logp", "target_topk_ids", "target_topk_logp")\nINT_FIELDS = {"target_ids", "draft_argmax", "draft_topk_ids", "target_topk_ids"}\nTOPK = 20\n\n\n@dataclass\nclass Trace:\n    """One sentence. All arrays have first dimension ``n``."""\n    id: int\n    n: int\n    target_ids: np.ndarray\n    draft_argmax: np.ndarray\n    H: np.ndarray            # draft entropy (nats)\n    top1: np.ndarray         # draft prob of its argmax\n    top2: np.ndarray         # draft prob of its 2nd-best token\n    p_target: np.ndarray     # q_t(y[t])\n    p_space: np.ndarray\n    p_punct: np.ndarray\n    p_eos: np.ndarray\n    p_newword: np.ndarray    # p_space + p_punct + p_eos\n    p_mark: np.ndarray\n    p_cont: np.ndarray\n    draft_topk_ids: np.ndarray\n    draft_topk_logp: np.ndarray\n    target_topk_ids: np.ndarray\n    target_topk_logp: np.ndarray\n    target_logp_of_draft_argmax: np.ndarray\n    match: np.ndarray = None  # derived: draft_argmax == target_ids\n    labels: dict = field(default_factory=dict, repr=False)  # gold labels (analysis only, not saved)\n\n    def __post_init__(self):\n        self.match = np.asarray(self.draft_argmax) == np.asarray(self.target_ids)\n        for f in SCALAR_FIELDS + TOPK_FIELDS:\n            a = getattr(self, f)\n            if len(a) != self.n:\n                raise ValueError(f"trace {self.id}: {f} has length {len(a)} != n={self.n}")\n\n\ndef save_traces(path, traces: list[Trace], meta: dict | None = None) -> None:\n    """Save traces as one compressed ``.npz`` (flat arrays + offsets)."""\n    path = Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    lens = np.array([t.n for t in traces], dtype=np.int64)\n    offsets = np.concatenate([[0], np.cumsum(lens)]).astype(np.int64)\n    out = dict(ids=np.array([t.id for t in traces], dtype=np.int64), offsets=offsets,\n               meta=np.array(json.dumps(meta or {}, ensure_ascii=False)))\n    for f in SCALAR_FIELDS + TOPK_FIELDS:\n        dt = np.int32 if f in INT_FIELDS else np.float32\n        parts = [np.asarray(getattr(t, f), dtype=dt) for t in traces]\n        if parts:\n            out[f] = np.concatenate(parts, axis=0)\n        else:\n            out[f] = np.zeros((0, TOPK) if f in TOPK_FIELDS else (0,), dtype=dt)\n    np.savez_compressed(path, **out)\n\n\ndef load_traces(path) -> tuple[list[Trace], dict]:\n    """Inverse of ``save_traces``."""\n    z = np.load(path, allow_pickle=False)\n    meta = json.loads(str(z["meta"]))\n    offsets, ids = z["offsets"], z["ids"]\n    flat = {f: z[f] for f in SCALAR_FIELDS + TOPK_FIELDS}\n    traces = []\n    for i, sid in enumerate(ids):\n        a, b = int(offsets[i]), int(offsets[i + 1])\n        traces.append(Trace(id=int(sid), n=b - a, **{f: flat[f][a:b] for f in flat}))\n    return traces, meta\n\n\ndef summary(traces: list[Trace]) -> dict:\n    """Positions, agreement rate, mean draft entropy when match / mismatch."""\n    m = np.concatenate([t.match for t in traces])\n    H = np.concatenate([t.H for t in traces])\n    return dict(sentences=len(traces), positions=int(len(m)), agreement=float(m.mean()),\n                mean_H=float(H.mean()), mean_H_match=float(H[m].mean()) if m.any() else float("nan"),\n                mean_H_mismatch=float(H[~m].mean()) if (~m).any() else float("nan"))\n\n\ndef attach_word_labels(traces: list[Trace], tok) -> None:\n    """Attach gold word/akshara labels from the *target* tokens (fixes B5). Analysis only."""\n    from .boundaries import token_labels\n    from .vocab import special_ids, token_bytes_list\n    from .boundaries import _valid_utf8_prefix_len\n    spec = special_ids(tok)\n    for t in traces:\n        tb = token_bytes_list(tok, t.target_ids)\n        sp = [int(i) in spec for i in t.target_ids]\n        t.labels.update(token_labels(tb, special=sp))\n        full = b"".join(b for b, s in zip(tb, sp) if not s)\n        t.labels["text"] = full[:_valid_utf8_prefix_len(full)].decode("utf-8")  # text the char spans refer to\n\n\ndef trace_field_names() -> list[str]:\n    return [f.name for f in fields(Trace)]\n',
    'gpu_traces': '"""Target generation, teacher-forced draft/target passes and timing (Kaggle notebook 01).\n\nPorted from the legacy Part A (``notebooks/legacy/day2-day3-llama8b-kaggle.ipynb`` cells 4-11).\nEverything here also runs on CPU (``scripts/smoke_cpu.py``) with a small model in fp32.\n\nOutputs (in ``out_dir``) for one split:\n  target_outputs_{split}.jsonl   id, src, ref, prompt_ids, out_ids, out_text   (append + resume)\n  _draft_tf_{split}.pkl          partial draft teacher-forced features         (resume)\n  _target_tf_{split}.pkl         partial target teacher-forced features        (resume)\n  traces_{split}.npz             final traces (``traces.save_traces``)\n  timing.json                    t_draft, t_target, c, verify(g)               (skipped if present)\n  run_info_{split}.json\n"""\nfrom __future__ import annotations\n\nimport gc\nimport json\nimport os\nimport pickle\nimport platform\nimport shutil\nimport time\nfrom collections import Counter\nfrom pathlib import Path\n\nimport numpy as np\n\nfrom .traces import TOPK, Trace, save_traces, summary\nfrom .vocab import VocabMasks, build_masks\n\nPROMPT_TEMPLATE = "Translate the following English sentence into Hindi. Output only the Hindi translation.\\n\\n{s}"\n\n\n# --------------------------------------------------------------------------- #\n# loading\n# --------------------------------------------------------------------------- #\n\ndef _torch():\n    import torch\n    return torch\n\n\ndef sync_all() -> None:\n    torch = _torch()\n    if torch.cuda.is_available():\n        for i in range(torch.cuda.device_count()):\n            torch.cuda.synchronize(i)\n\n\ndef gpu_report(tag: str = "") -> None:\n    torch = _torch()\n    if torch.cuda.is_available():\n        print(tag, " | ".join(f"GPU{i}: {torch.cuda.memory_allocated(i) / 2**30:.1f} GiB allocated"\n                              for i in range(torch.cuda.device_count())))\n\n\ndef from_pretrained(name: str, dtype, **kw):\n    """``AutoModelForCausalLM.from_pretrained`` across transformers versions (dtype vs torch_dtype)."""\n    from transformers import AutoModelForCausalLM\n    try:\n        return AutoModelForCausalLM.from_pretrained(name, dtype=dtype, **kw).eval()\n    except TypeError:\n        return AutoModelForCausalLM.from_pretrained(name, torch_dtype=dtype, **kw).eval()\n\n\ndef load_tokenizer(name: str):\n    from transformers import AutoTokenizer\n    tok = AutoTokenizer.from_pretrained(name)\n    tok.padding_side = "left"\n    if tok.pad_token is None:\n        tok.pad_token = tok.eos_token\n    return tok\n\n\ndef load_target(name: str, dtype, max_memory: dict | None = None, device_map="auto"):\n    """Target model, layer-split over the available GPUs (``device_map=\'auto\'``) or on CPU."""\n    torch = _torch()\n    if torch.cuda.is_available():\n        m = from_pretrained(name, dtype, device_map=device_map, max_memory=max_memory,\n                            low_cpu_mem_usage=True)\n        print("modules per device:", dict(Counter(m.hf_device_map.values())))\n    else:\n        m = from_pretrained(name, dtype)\n    return m\n\n\ndef load_draft(name: str, dtype, device: str):\n    return from_pretrained(name, dtype).to(device)\n\n\ndef get_eos_ids(model, tok) -> list[int]:\n    """Stop ids: generation-config eos plus ``<|eot_id|>`` (Llama-3 instruct end of turn)."""\n    eos = model.generation_config.eos_token_id\n    ids = set(eos if isinstance(eos, list) else [eos])\n    eot = tok.convert_tokens_to_ids("<|eot_id|>")\n    if eot is not None and eot != tok.unk_token_id:\n        ids.add(eot)\n    return sorted(int(i) for i in ids if i is not None)\n\n\ndef check_draft_target_compat(draft, target, tok_d, tok_t, probe: str) -> None:\n    """Draft and target must share a vocabulary (legacy asserts, kept)."""\n    assert draft.config.vocab_size == target.config.vocab_size, "vocab sizes differ"\n    probe = probe + " नमस्ते दुनिया. Hello world!"\n    assert (tok_d(probe, add_special_tokens=False).input_ids\n            == tok_t(probe, add_special_tokens=False).input_ids), "tokenizers disagree"\n    print("draft/target tokenizers match")\n\n\n# --------------------------------------------------------------------------- #\n# data + prompts\n# --------------------------------------------------------------------------- #\n\ndef load_flores_pairs(split: str, n: int | None = None):\n    """FLORES+ eng_Latn -> hin_Deva rows ``(id, src, ref)`` for ``split`` in {dev, devtest}."""\n    from datasets import load_dataset\n    assert split in ("dev", "devtest"), split\n    en = load_dataset("openlanguagedata/flores_plus", "eng_Latn", split=split)\n    hi = load_dataset("openlanguagedata/flores_plus", "hin_Deva", split=split)\n    hi_by_id = {r["id"]: r["text"] for r in hi}\n    rows = [(int(r["id"]), r["text"], hi_by_id[r["id"]]) for r in en]\n    return rows[:n] if n else rows\n\n\ndef build_prompt(tok, s: str) -> str:\n    msgs = [{"role": "user", "content": PROMPT_TEMPLATE.format(s=s)}]\n    return tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)\n\n\ndef read_jsonl(path) -> list[dict]:\n    path = Path(path)\n    if not path.exists():\n        return []\n    with open(path, encoding="utf-8") as fh:\n        return [json.loads(l) for l in fh if l.strip()]\n\n\n# --------------------------------------------------------------------------- #\n# 1. batched greedy target generation (resumable)\n# --------------------------------------------------------------------------- #\n\ndef generate_targets(target, tok, rows, out_jsonl, batch: int, max_new: int, eos_ids,\n                     device: str, progress: bool = True) -> list[dict]:\n    """Greedy, batched (left padding) target outputs, appended to ``out_jsonl`` after every batch.\n\n    Resumes if ``out_jsonl`` holds a prefix of ``rows`` (by id); otherwise starts over.\n    """\n    torch = _torch()\n    from tqdm.auto import tqdm\n    out_jsonl = Path(out_jsonl)\n    ids = [r[0] for r in rows]\n    prompts = [build_prompt(tok, r[1]) for r in rows]\n    records = read_jsonl(out_jsonl)\n    if [r["id"] for r in records] != ids[:len(records)]:\n        print("existing output does not match this run; starting over")\n        records = []\n    print(f"resuming: {len(records)} / {len(rows)} sentences already done")\n    eos_set = set(eos_ids)\n    with open(out_jsonl, "a" if records else "w", encoding="utf-8") as f:\n        rng = range(len(records), len(prompts), batch)\n        for i in (tqdm(rng, desc="generate") if progress else rng):\n            enc = tok(prompts[i:i + batch], return_tensors="pt", padding=True,\n                      add_special_tokens=False).to(device)\n            with torch.no_grad():\n                out = target.generate(**enc, do_sample=False, temperature=None, top_p=None,\n                                      max_new_tokens=max_new, eos_token_id=list(eos_set),\n                                      pad_token_id=tok.pad_token_id)\n            gen = out[:, enc.input_ids.shape[1]:].tolist()\n            for j, g in enumerate(gen):\n                cut = next((p + 1 for p, t in enumerate(g) if t in eos_set), len(g))  # keep eos\n                g = g[:cut]\n                pid = enc.input_ids[j][enc.attention_mask[j].bool()].tolist()      # unpadded prompt\n                rid, src, ref = rows[i + j]\n                rec = dict(id=rid, src=src, ref=ref, prompt_ids=pid, out_ids=g,\n                           out_text=tok.decode(g, skip_special_tokens=True).strip())\n                records.append(rec)\n                f.write(json.dumps(rec, ensure_ascii=False) + "\\n")\n            f.flush()\n    n_hit = sum(len(r["out_ids"]) >= max_new for r in records)\n    print(f"{len(records)} target outputs in {out_jsonl.name}; {n_hit} hit MAX_NEW={max_new}")\n    return records\n\n\ndef generate_unbatched(model, tok, prompt_ids: list[int], max_new: int, eos_ids, device: str) -> list[int]:\n    """Plain greedy generation for one prompt, no padding (reference for losslessness, B9)."""\n    torch = _torch()\n    x = torch.tensor([prompt_ids], device=device)\n    with torch.no_grad():\n        out = model.generate(input_ids=x, attention_mask=torch.ones_like(x), do_sample=False,\n                             temperature=None, top_p=None, max_new_tokens=max_new,\n                             eos_token_id=list(eos_ids), pad_token_id=tok.pad_token_id)\n    g = out[0, len(prompt_ids):].tolist()\n    eos_set = set(eos_ids)\n    cut = next((p + 1 for p, t in enumerate(g) if t in eos_set), len(g))\n    return g[:cut]\n\n\n# --------------------------------------------------------------------------- #\n# 2./3. teacher-forced passes\n# --------------------------------------------------------------------------- #\n\ndef _mask_matrix(masks: VocabMasks, vocab_size: int, device):\n    """(V, 5) float32 matrix: columns space, punct, eos, mark, cont."""\n    torch = _torch()\n    cols = [masks.padded("starts_space", vocab_size), masks.padded("starts_punct", vocab_size),\n            np.concatenate([masks.is_eos, np.zeros(vocab_size - masks.size, bool)]),\n            masks.padded("starts_mark", vocab_size), masks.padded("starts_cont", vocab_size)]\n    return torch.tensor(np.stack(cols, 1).astype(np.float32), device=device)\n\n\ndef _logits_for_output(model, prompt_ids, out_ids, device):\n    """Logits (float32) predicting each output token: row t is q_t (conditioned on prompt + y[:t])."""\n    torch = _torch()\n    x = torch.tensor([list(prompt_ids) + list(out_ids)], device=device)\n    with torch.no_grad():\n        lg = model(x).logits[0, len(prompt_ids) - 1:-1]\n    return lg.float()\n\n\ndef draft_features(draft, rec: dict, M, device: str, topk: int = TOPK) -> dict[str, np.ndarray]:\n    """Per-position draft features for one record (all float32 on host)."""\n    torch = _torch()\n    lg = _logits_for_output(draft, rec["prompt_ids"], rec["out_ids"], device)\n    lp = torch.log_softmax(lg, -1)\n    p = lp.exp()\n    H = -(p * lp).sum(-1)\n    t2 = p.topk(2, -1).values\n    tk = lp.topk(topk, -1)\n    y = torch.tensor(rec["out_ids"], device=lg.device)\n    P = p @ M.to(lg.device)                       # (n, 5): space, punct, eos, mark, cont\n    f = dict(draft_argmax=lp.argmax(-1), H=H, top1=t2[:, 0], top2=t2[:, 1],\n             p_target=p.gather(1, y[:, None]).squeeze(1),\n             p_space=P[:, 0], p_punct=P[:, 1], p_eos=P[:, 2], p_newword=P[:, 0] + P[:, 1] + P[:, 2],\n             p_mark=P[:, 3], p_cont=P[:, 4], draft_topk_ids=tk.indices, draft_topk_logp=tk.values)\n    return {k: v.cpu().numpy() for k, v in f.items()}\n\n\ndef target_features(target, rec: dict, draft_argmax: np.ndarray, device: str,\n                    topk: int = TOPK) -> dict[str, np.ndarray]:\n    torch = _torch()\n    lg = _logits_for_output(target, rec["prompt_ids"], rec["out_ids"], device)\n    lp = torch.log_softmax(lg, -1)\n    tk = lp.topk(topk, -1)\n    da = torch.tensor(np.asarray(draft_argmax), device=lg.device, dtype=torch.long)\n    f = dict(target_topk_ids=tk.indices, target_topk_logp=tk.values,\n             target_logp_of_draft_argmax=lp.gather(1, da[:, None]).squeeze(1))\n    return {k: v.cpu().numpy() for k, v in f.items()}\n\n\ndef _resumable_pass(records, fn, partial_path, desc: str, save_every: int = 50, progress=True):\n    """Run ``fn(rec)`` for every record, pickling partial results so a dead session can resume."""\n    from tqdm.auto import tqdm\n    partial_path = Path(partial_path)\n    done: dict = {}\n    if partial_path.exists():\n        with open(partial_path, "rb") as fh:\n            done = pickle.load(fh)\n        print(f"{desc}: resuming with {len(done)} / {len(records)} done")\n    it = tqdm(records, desc=desc) if progress else records\n    since = 0\n    for rec in it:\n        if rec["id"] in done:\n            continue\n        done[rec["id"]] = fn(rec)\n        since += 1\n        if since >= save_every:\n            _atomic_pickle(done, partial_path)\n            since = 0\n    _atomic_pickle(done, partial_path)\n    return done\n\n\ndef _atomic_pickle(obj, path: Path) -> None:\n    tmp = path.with_suffix(".tmp")\n    with open(tmp, "wb") as fh:\n        pickle.dump(obj, fh, protocol=pickle.HIGHEST_PROTOCOL)\n    os.replace(tmp, path)\n\n\ndef run_draft_pass(draft, records, masks: VocabMasks, out_dir, split: str, device: str, progress=True):\n    vocab = draft.get_output_embeddings().weight.shape[0]\n    M = _mask_matrix(masks, vocab, device)\n    return _resumable_pass(records, lambda r: draft_features(draft, r, M, device),\n                           Path(out_dir) / f"_draft_tf_{split}.pkl", "draft teacher-forced", progress=progress)\n\n\ndef run_target_pass(target, records, draft_feats: dict, out_dir, split: str, device: str, progress=True):\n    return _resumable_pass(records, lambda r: target_features(target, r, draft_feats[r["id"]]["draft_argmax"], device),\n                           Path(out_dir) / f"_target_tf_{split}.pkl", "target teacher-forced", progress=progress)\n\n\ndef assemble_traces(records, draft_feats: dict, target_feats: dict) -> list[Trace]:\n    out = []\n    for r in records:\n        d, t = draft_feats[r["id"]], target_feats[r["id"]]\n        out.append(Trace(id=int(r["id"]), n=len(r["out_ids"]), target_ids=np.asarray(r["out_ids"], np.int32),\n                         **d, **t))\n    return out\n\n\ndef target_argmax_check(traces: list[Trace]) -> dict:\n    """Rate at which the unpadded teacher-forced target argmax equals the generated y[t]."""\n    agree, diffs, total = 0, [], 0\n    for t in traces:\n        am = t.target_topk_ids[:, 0]\n        eq = am == t.target_ids\n        agree += int(eq.sum())\n        total += t.n\n        for pos in np.flatnonzero(~eq):\n            lp = t.target_topk_logp[pos]\n            diffs.append(dict(id=t.id, pos=int(pos), y=int(t.target_ids[pos]), argmax=int(am[pos]),\n                              margin=float(lp[0] - lp[1])))\n    return dict(rate=agree / max(total, 1), positions=total, n_diff=len(diffs), diffs=diffs)\n\n\n# --------------------------------------------------------------------------- #\n# 4. timing\n# --------------------------------------------------------------------------- #\n\ndef time_fwd(model, q: int = 1, ctx: int = 150, steps: int = 60, reps: int = 3, device: str = "cuda:0") -> float:\n    """Median latency (s) of a forward with ``q`` new tokens on top of a KV cache of ``ctx`` tokens."""\n    torch = _torch()\n    ts = []\n    with torch.no_grad():\n        for _ in range(reps):\n            x = torch.randint(1000, 20000, (1, ctx), device=device)\n            o = model(x, use_cache=True)\n            pkv = o.past_key_values\n            for s in range(steps + 10):\n                inp = torch.randint(1000, 20000, (1, q), device=device)\n                sync_all()\n                t = time.perf_counter()\n                o = model(inp, past_key_values=pkv, use_cache=True)\n                sync_all()\n                dt = time.perf_counter() - t\n                pkv = o.past_key_values\n                if s >= 10:\n                    ts.append(dt)\n    return float(np.median(ts))\n\n\ndef run_timing(draft, target, out_path, device: str, setup: dict, steps: int = 60, reps: int = 3,\n               max_g: int = 8) -> dict:\n    """Legacy timing (c and verify(g), g=1..max_g) plus draft q=1 at ctx 150 and 400."""\n    out_path = Path(out_path)\n    if out_path.exists():\n        print(f"{out_path.name} exists, timing skipped")\n        return json.loads(out_path.read_text())\n    t_d = time_fwd(draft, 1, steps=steps, reps=reps, device=device)\n    t_t = time_fwd(target, 1, steps=steps, reps=reps, device=device)\n    print(f"draft decode step: {t_d * 1000:.2f} ms | target decode step: {t_t * 1000:.2f} ms | c = {t_d / t_t:.3f}")\n    ver = {g: time_fwd(target, g + 1, steps=steps, reps=reps, device=device) for g in range(1, max_g + 1)}\n    print({g: f"{v * 1000:.1f} ms" for g, v in ver.items()})\n    extra = {str(ctx): time_fwd(draft, 1, ctx=ctx, steps=steps, reps=reps, device=device) for ctx in (150, 400)}\n    T = dict(t_draft=t_d, t_target=t_t, c=t_d / t_t, t_target_verify={str(g): v for g, v in ver.items()},\n             t_draft_by_ctx=extra, setup=setup)\n    out_path.write_text(json.dumps(T, indent=1))\n    return T\n\n\n# --------------------------------------------------------------------------- #\n# run info + zip\n# --------------------------------------------------------------------------- #\n\ndef run_info(git_hash: str, **extra) -> dict:\n    torch = _torch()\n    import transformers\n    gpus = [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())] if torch.cuda.is_available() else []\n    return dict(git_hash=git_hash, torch=torch.__version__, transformers=transformers.__version__,\n                python=platform.python_version(), gpus=gpus, time=time.strftime("%Y-%m-%d %H:%M:%S"), **extra)\n\n\ndef zip_dir(src_dir, zip_path, include=None) -> str:\n    """Zip files of ``src_dir`` (only names in ``include`` if given; flat, no folder) to ``zip_path``.\n    Never deletes or moves anything in ``src_dir``. Returns the zip path."""\n    import zipfile\n    src_dir, zip_path = Path(src_dir), Path(zip_path)\n    names = list(include) if include is not None else sorted(p.name for p in src_dir.iterdir() if p.is_file())\n    missing = [n for n in names if not (src_dir / n).exists()]\n    if missing:\n        raise FileNotFoundError(f"cannot zip, missing in {src_dir}: {missing}")\n    tmp = zip_path.with_name(zip_path.name + ".part")\n    with zipfile.ZipFile(tmp, "w", compression=zipfile.ZIP_DEFLATED) as z:\n        for n in names:\n            z.write(src_dir / n, arcname=n)\n    os.replace(tmp, zip_path)\n    return str(zip_path)\n\n\ndef free(*objs) -> None:\n    for o in objs:\n        del o\n    gc.collect()\n    torch = _torch()\n    if torch.cuda.is_available():\n        torch.cuda.empty_cache()\n\n\n# --------------------------------------------------------------------------- #\n# full pipeline for one split (used by notebook 01 and the smoke test)\n# --------------------------------------------------------------------------- #\n\ndef run_split(*, split: str, out_dir, target, draft, tok, tok_d, n: int | None, batch: int,\n              max_new: int, device: str, git_hash: str, timing_kw: dict | None = None,\n              setup_extra: dict | None = None, rows=None, progress: bool = True,\n              zip_name: str | None = None) -> dict:\n    """Steps 1-6 of PLAN T2 for one split. Returns a dict with paths and summaries.\n\n    timing_kw: passed to ``run_timing`` (steps, reps, max_g); setup_extra: added to timing.json setup\n    (e.g. max_memory, GPU names).\n    """\n    out_dir = Path(out_dir)\n    out_dir.mkdir(parents=True, exist_ok=True)\n    rows = rows if rows is not None else load_flores_pairs(split, n)\n    eos_ids = get_eos_ids(target, tok)\n    print(f"{split}: {len(rows)} sentences; eos ids {eos_ids}")\n\n    records = generate_targets(target, tok, rows, out_dir / f"target_outputs_{split}.jsonl",\n                               batch, max_new, eos_ids, device, progress=progress)\n    check_draft_target_compat(draft, target, tok_d, tok, records[0]["out_text"])\n\n    masks = build_masks(tok, eos_ids)\n    dfe = run_draft_pass(draft, records, masks, out_dir, split, device, progress=progress)\n    tfe = run_target_pass(target, records, dfe, out_dir, split, device, progress=progress)\n    traces = assemble_traces(records, dfe, tfe)\n    chk = target_argmax_check(traces)\n    print(f"target teacher-forced argmax == y[t]: {chk[\'rate\']:.4f} ({chk[\'n_diff\']} positions differ)")\n    for d in chk["diffs"][:20]:\n        print("   ", d)\n    if chk["rate"] < 0.995:\n        print("WARNING: target argmax agreement below 0.995")\n\n    import transformers\n    meta = dict(split=split, target=getattr(target.config, "_name_or_path", ""),\n                draft=getattr(draft.config, "_name_or_path", ""), max_new=max_new, batch=batch,\n                eos_ids=eos_ids, git_hash=git_hash, transformers=transformers.__version__,\n                target_argmax_check={k: v for k, v in chk.items() if k != "diffs"},\n                target_argmax_diffs=chk["diffs"][:500])\n    npz = out_dir / f"traces_{split}.npz"\n    save_traces(npz, traces, meta)\n    S = summary(traces)\n    print(f"positions: {S[\'positions\']} | agreement: {S[\'agreement\']:.3f} | mean H: {S[\'mean_H\']:.3f} "\n          f"(match {S[\'mean_H_match\']:.3f}, mismatch {S[\'mean_H_mismatch\']:.3f})")\n\n    setup = dict(target=meta["target"], draft=meta["draft"], dtype=str(next(target.parameters()).dtype),\n                 device=device, **(setup_extra or {}))\n    T = run_timing(draft, target, out_dir / "timing.json", device, setup, **(timing_kw or {}))\n\n    info = run_info(git_hash, split=split, n_sentences=len(rows), summary=S,\n                    target=meta["target"], draft=meta["draft"])\n    (out_dir / f"run_info_{split}.json").write_text(json.dumps(info, indent=1, ensure_ascii=False))\n    zip_name = zip_name or f"traces_v2_{split}"\n    zp = zip_dir(out_dir, out_dir.parent / f"{zip_name}.zip",\n                 include=[npz.name, f"target_outputs_{split}.jsonl", "timing.json", f"run_info_{split}.json"])\n    return dict(zip=zp, npz=str(npz), summary=S, timing=T, target_check=chk)\n\n\n# --------------------------------------------------------------------------- #\n# notebook 01 entry point (also called by scripts/smoke_cpu.py)\n# --------------------------------------------------------------------------- #\n\nDEFAULT_CONFIG_01 = dict(\n    SPLIT="dev",                                    # "dev" or "devtest": one split per run\n    N=None,                                         # None = all sentences of the split\n    BATCH=16, MAX_NEW=256,\n    SMOKE=False,                                    # True: CPU, fp32, one model as target+draft\n    TARGET="meta-llama/Meta-Llama-3-8B-Instruct",\n    DRAFT="meta-llama/Llama-3.2-1B-Instruct",\n    MAX_MEMORY={0: "8GiB", 1: "10GiB"},\n    OUT_ROOT="/kaggle/working",\n    GIT_HASH="unknown",\n    TIMING=dict(steps=60, reps=3, max_g=8),\n    RESUME_FROM=None,   # folder with partial outputs of an earlier (dead) run, e.g. /kaggle/input/<...>\n)\n\n\ndef _prepare_out_dir(out_dir: Path, out_root: Path, resume_from) -> None:\n    """Copy partial outputs from ``resume_from`` (searched recursively) and reuse a timing.json\n    measured by another split\'s run in this session."""\n    out_dir.mkdir(parents=True, exist_ok=True)\n    if resume_from:\n        src = Path(resume_from)\n        for f in src.rglob("*"):\n            if f.is_file() and f.parent.name == out_dir.name and not (out_dir / f.name).exists():\n                shutil.copy2(f, out_dir / f.name)\n                print("resume: copied", f)\n    if not (out_dir / "timing.json").exists():\n        for other in sorted(out_root.glob("traces_v2_*/timing.json")):\n            shutil.copy2(other, out_dir / "timing.json")\n            print("reusing timing from", other)\n            break\n\n\ndef notebook01_main(cfg: dict) -> dict:\n    """Run PLAN T2 steps 1-6 for ``cfg[\'SPLIT\']``. Writes ``{OUT_ROOT}/traces_v2_{split}/`` and\n    ``{OUT_ROOT}/traces_v2_{split}.zip``. With ``SMOKE=True`` the draft model is used as both\n    target and draft (fp32, CPU, loaded once) and only 3 sentences are processed."""\n    torch = _torch()\n    cfg = {**DEFAULT_CONFIG_01, **cfg}\n    split = cfg["SPLIT"]\n    out_dir = Path(cfg["OUT_ROOT"]) / f"traces_v2_{split}"\n    _prepare_out_dir(out_dir, Path(cfg["OUT_ROOT"]), cfg["RESUME_FROM"])\n    if cfg["SMOKE"]:\n        device, dtype = "cpu", torch.float32\n        tok = load_tokenizer(cfg["DRAFT"])\n        draft = target = load_draft(cfg["DRAFT"], dtype, device)\n        tok_d = tok\n        n = cfg["N"] or 3\n        setup_extra = dict(smoke=True)\n    else:\n        assert torch.cuda.device_count() >= 2, "Need 2 GPUs: Settings > Accelerator > GPU T4 x2"\n        for i in range(torch.cuda.device_count()):\n            p = torch.cuda.get_device_properties(i)\n            print(f"GPU{i}: {p.name}, {p.total_memory / 2**30:.1f} GiB")\n        device, dtype = "cuda:0", torch.float16\n        tok = load_tokenizer(cfg["TARGET"])\n        tok_d = load_tokenizer(cfg["DRAFT"])\n        target = load_target(cfg["TARGET"], dtype, max_memory=cfg["MAX_MEMORY"])\n        gpu_report("after loading target:")\n        draft = load_draft(cfg["DRAFT"], dtype, device)\n        gpu_report("after loading draft:")\n        n = cfg["N"]\n        setup_extra = dict(max_memory={str(k): v for k, v in cfg["MAX_MEMORY"].items()},\n                           gpus=[torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])\n    res = run_split(split=split, out_dir=out_dir, target=target, draft=draft, tok=tok, tok_d=tok_d,\n                    n=n, batch=cfg["BATCH"], max_new=cfg["MAX_NEW"], device=device,\n                    git_hash=cfg["GIT_HASH"], timing_kw=cfg["TIMING"], setup_extra=setup_extra)\n    print("\\n" + "=" * 70)\n    print(f"DONE. Download  {Path(res[\'zip\']).name}  from the notebook\'s Output tab")\n    print(f"and unzip it into  data/traces_v2/  in the repo (files land directly in that folder).")\n    print("=" * 70)\n    return res\n',
    'simulator': '"""Trace simulator for greedy speculative decoding (PLAN T4, fixes B3, B8).\n\nPorted from ``notebooks/legacy/day2-day3-llama8b-kaggle.ipynb`` (cells 14-17), with:\n  1. the ``Policy`` interface ``stop(view, p, g)`` called with g = 1..cap-1 after each drafted token\n     (dynamic policies always draft >= 1 token; NoDraft / FixedK(0) / Oracle decide g = 0 via\n     ``initial_stop(view, p)``);\n  2. a causality guard (``TraceView``) - see PLAN §3;\n  3. lookahead cost: a lookahead-1 policy that stops by itself pays one extra draft forward ``c``;\n  4. per-sentence emitted / cost arrays and round records;\n  5. measured verification cost as the default.\n\nExactness for greedy decoding: within a round the draft\'s own tokens coincide with the target\'s up\nto the first mismatch, and everything after it is rejected, so accepted = leading matches.\nOnly the stopping decision after a mismatch can differ from a real loop (teacher-forcing\napproximation, PLAN §3).\n"""\nfrom __future__ import annotations\n\nimport json\nfrom dataclasses import dataclass, field\nfrom pathlib import Path\nfrom typing import Callable, Sequence\n\nimport numpy as np\n\nfrom .traces import Trace\n\nGAMMA_MAX = 8\n\n# trace fields a policy may read through ``view.feat`` (functions of q_t only)\nDRAFT_FEATURES = {"H", "top1", "top2", "p_space", "p_punct", "p_eos", "p_newword", "p_mark", "p_cont",\n                  "draft_argmax", "draft_topk_ids", "draft_topk_logp"}\n# never readable by non-oracle policies\nGOLD_FIELDS = {"match", "target_ids", "p_target", "target_topk_ids", "target_topk_logp",\n               "target_logp_of_draft_argmax"}\n\n\nclass CausalityError(RuntimeError):\n    """A policy read information a real decoder would not have at that moment."""\n\n\nclass TraceView:\n    """Read-only, causality-checked view of one trace for policies.\n\n    The simulator sets ``limit = p + g - 1 + policy.lookahead`` and ``p`` before each call.\n      feat(name, idx): draft feature at index idx (idx <= limit).\n      token(idx):      committed target token for idx < p, the draft\'s own token for p <= idx <= p+g-1\n                       (idx <= limit; for lookahead policies idx = p+g gives the draft argmax at p+g).\n      gold(name, idx): any field incl. match / target ids / labels; oracle policies only.\n    """\n\n    __slots__ = ("trace", "limit", "p", "is_oracle", "n", "_labels")\n\n    def __init__(self, trace: Trace, is_oracle: bool = False):\n        self.trace, self.is_oracle, self.n = trace, is_oracle, trace.n\n        self.limit = -1\n        self.p = 0\n        self._labels = trace.labels\n\n    def _check(self, idx: int) -> None:\n        if idx > self.limit and not self.is_oracle:\n            raise CausalityError(f"read index {idx} > limit {self.limit} (p={self.p})")\n        if idx < 0 or idx >= self.n:\n            raise IndexError(idx)\n\n    def feat(self, name: str, idx: int):\n        if name not in DRAFT_FEATURES and not self.is_oracle:\n            raise CausalityError(f"policy may not read \'{name}\' via feat()")\n        self._check(idx)\n        return getattr(self.trace, name)[idx]\n\n    def token(self, idx: int) -> int:\n        self._check(idx)\n        if idx < self.p:\n            return int(self.trace.target_ids[idx])\n        return int(self.trace.draft_argmax[idx])\n\n    def gold(self, name: str, idx: int):\n        if not self.is_oracle:\n            raise CausalityError(f"non-oracle policy read gold field \'{name}\'")\n        if name in self._labels:\n            return self._labels[name][idx]\n        return getattr(self.trace, name)[idx]\n\n\nclass Policy:\n    name: str = "policy"\n    lookahead: int = 0          # 0 or 1\n    is_oracle: bool = False\n\n    def initial_stop(self, view: TraceView, p: int) -> bool:\n        """Decision at g = 0 (before drafting; view.limit = p - 1). Only NoDraft / FixedK(0) and the\n        oracle can return True: every other dynamic policy drafts at least one token."""\n        return False\n\n    def stop(self, view: TraceView, p: int, g: int) -> bool:  # g >= 1\n        raise NotImplementedError\n\n    def reset(self, view: TraceView) -> None:\n        """Called once per sentence (policies with per-sentence caches)."""\n\n    def params(self) -> dict:\n        return {}\n\n    def __repr__(self):\n        ps = ", ".join(f"{k}={v}" for k, v in self.params().items())\n        return f"{self.name}({ps})"\n\n\n# --------------------------------------------------------------------------- #\n# cost model\n# --------------------------------------------------------------------------- #\n\ndef load_timing(path) -> dict:\n    return json.loads(Path(path).read_text())\n\n\nclass MeasuredVerify:\n    """Measured verification cost for g drafted tokens (g+1 target tokens) relative to one decode\n    step, linearly interpolated, clipped below at 1 (dips are timing noise). verify(0) = 1.\n    A class (not a closure) so it can be sent to worker processes."""\n\n    def __init__(self, timing: dict):\n        self.gs = sorted(int(g) for g in timing["t_target_verify"])\n        self.vs = [max(1.0, timing["t_target_verify"][str(g)] / timing["t_target"]) for g in self.gs]\n        self.table = {0: 1.0}\n        for g in range(1, max(self.gs) + 1):\n            self.table[g] = float(np.interp(g, self.gs, self.vs))\n\n    def __call__(self, g: int) -> float:\n        return self.table[g] if g in self.table else float(np.interp(g, self.gs, self.vs))\n\n\ndef make_verify(timing: dict) -> MeasuredVerify:\n    return MeasuredVerify(timing)\n\n\ndef unit_verify(g: int) -> float:\n    return 1.0\n\n\ndef formula_speedup(alpha: float, gamma: int, c: float) -> float:\n    """f = (1 - a^(g+1)) / ((1 - a)(g c + 1)), i.i.d. acceptance assumption (legacy)."""\n    exp_tokens = (gamma + 1) if alpha >= 1 else (1 - alpha ** (gamma + 1)) / (1 - alpha)\n    return exp_tokens / (gamma * c + 1)\n\n\n# --------------------------------------------------------------------------- #\n# simulator\n# --------------------------------------------------------------------------- #\n\n@dataclass\nclass SimResult:\n    rounds: int = 0\n    drafted: int = 0\n    accepted: int = 0\n    rejections: int = 0\n    emitted: int = 0\n    cost: float = 0.0\n    lookahead_charges: int = 0\n    policy_stops: int = 0\n    draft_lens: list = field(default_factory=list)\n    sent_ids: np.ndarray = None\n    sent_emitted: np.ndarray = None\n    sent_cost: np.ndarray = None\n    records: list = field(default_factory=list)   # (sentence_id, p, g, accepted, stopped_by_policy)\n\n\ndef simulate(traces: Sequence[Trace], policy: Policy, gamma_max: int = GAMMA_MAX, c: float = 0.334,\n             verify: Callable[[int], float] | None = None, charge_lookahead: bool = True,\n             record_rounds: bool = True) -> SimResult:\n    """Simulate greedy speculative decoding of every trace with ``policy``.\n\n    One round at committed position p: draft g tokens (cap = min(gamma_max, n - p)), accept the\n    leading matches a, emit min(a + 1, n - p) tokens, cost g*c + verify(g)\n    (+ c if the policy has lookahead 1, stopped by itself before the cap, and charge_lookahead).\n    """\n    verify = verify or unit_verify\n    R = SimResult()\n    ns = len(traces)\n    R.sent_ids = np.array([t.id for t in traces])\n    R.sent_emitted = np.zeros(ns, dtype=np.int64)\n    R.sent_cost = np.zeros(ns, dtype=np.float64)\n    la = int(policy.lookahead)\n    for si, tr in enumerate(traces):\n        n, match = tr.n, tr.match\n        view = TraceView(tr, policy.is_oracle)\n        policy.reset(view)\n        p = 0\n        s_emit, s_cost = 0, 0.0\n        while p < n:\n            r = n - p\n            cap = min(gamma_max, r)\n            view.p = p\n            g = 0\n            by_policy = False\n            view.limit = p - 1\n            if cap > 0 and not policy.initial_stop(view, p):\n                g = 1\n                while g < cap:\n                    view.limit = p + g - 1 + la\n                    if policy.stop(view, p, g):\n                        by_policy = True\n                        break\n                    g += 1\n            elif cap > 0:\n                by_policy = True\n            a = 0\n            while a < g and match[p + a]:\n                a += 1\n            gained = min(a + 1, r)\n            cost = g * c + verify(g)\n            if la and by_policy and g > 0 and charge_lookahead:\n                cost += c\n                R.lookahead_charges += 1\n            R.rounds += 1\n            R.drafted += g\n            R.accepted += a\n            R.rejections += int(a < g)\n            R.emitted += gained\n            R.cost += cost\n            R.policy_stops += int(by_policy)\n            R.draft_lens.append(g)\n            if record_rounds:\n                R.records.append((tr.id, p, g, a, by_policy))\n            s_emit += gained\n            s_cost += cost\n            p += gained\n        R.sent_emitted[si] = s_emit\n        R.sent_cost[si] = s_cost\n    return R\n\n\ndef summarize(res: SimResult) -> dict:\n    """Legacy summary metrics."""\n    return dict(\n        acceptance_rate=res.accepted / max(res.drafted, 1),                    # accepted / drafted\n        alpha_hat=res.accepted / max(res.accepted + res.rejections, 1),        # per-token accept prob (geometric MLE)\n        tokens_per_verif=res.emitted / max(res.rounds, 1),\n        mean_draft_len=res.drafted / max(res.rounds, 1),\n        speedup=res.emitted / res.cost if res.cost else float("nan"),\n    )\n\n\n# --------------------------------------------------------------------------- #\n# legacy protocol (sanity check against the PDF numbers, PLAN T4)\n# --------------------------------------------------------------------------- #\n\ndef legacy_protocol(traces: Sequence[Trace], c: float, seed: int = 0) -> dict:\n    """Reproduce the old evaluation: random half split of devtest (default_rng(0).permutation),\n    entropy grid lam = linspace(-1.5, 0.95, 40), beta in {0.5, 1, 2}, unit verify, test on the 2nd half.\n    Expected (PDF): fixed k=2 ~ 1.36, entropy ~ 1.45 (within ~0.03 after regeneration)."""\n    from .policies import Entropy, FixedK, Oracle\n    idx = np.random.default_rng(seed).permutation(len(traces))\n    dev = [traces[i] for i in idx[: len(idx) // 2]]\n    test = [traces[i] for i in idx[len(idx) // 2:]]\n    ks = range(2, 9)\n    best_k = max(ks, key=lambda k: summarize(simulate(dev, FixedK(k), k, c, record_rounds=False))["speedup"])\n    rows = []\n    for beta in (0.5, 1.0, 2.0):\n        for lam in np.round(np.linspace(-1.5, 0.95, 40), 3):\n            m = summarize(simulate(dev, Entropy(float(lam), beta), GAMMA_MAX, c, record_rounds=False))\n            rows.append((m["speedup"], float(lam), beta))\n    _, lam, beta = max(rows)\n    fixed_all = {k: summarize(simulate(traces, FixedK(k), k, c, record_rounds=False)) for k in ks}\n    return dict(\n        best_k_dev=best_k,\n        fixed_test=summarize(simulate(test, FixedK(best_k), best_k, c, record_rounds=False)),\n        fixed_k2_all=fixed_all[2],\n        entropy_params=dict(lam=lam, beta=beta),\n        entropy_test=summarize(simulate(test, Entropy(lam, beta), GAMMA_MAX, c, record_rounds=False)),\n        oracle_test=summarize(simulate(test, Oracle(), GAMMA_MAX, c, record_rounds=False)),\n        expected=dict(fixed_k2=1.361, entropy=1.452, oracle=1.905),\n    )\n',
    'policies': '"""Stopping policies (PLAN T5, T6). Each reads only through ``simulator.TraceView``.\n\nCausal (lookahead 0) policies read draft features at indices <= p+g-1. Lookahead-1 policies also read\nindex p+g (the next draft distribution q_{p+g}, one extra draft forward; the simulator charges it\nwhen the policy stops by itself). Oracles (``is_oracle``) may read gold fields; they are upper bounds.\n\n"Extend" semantics (AksharaSafeEntropy, Hybrid, MorphEntropy): let g0 be the first g at which the\nentropy rule fired in this round (recomputed from H[p..p+g-1], so there is no hidden state). At every\ng >= g0 the policy stops if its linguistic condition holds at g, or once g - g0 >= max_ext.\n"""\nfrom __future__ import annotations\n\nimport math\nfrom dataclasses import dataclass\n\nimport numpy as np\n\nfrom .boundaries import END, WORD\nfrom .simulator import Policy, TraceView\n\n\ndef entropy_fires(H: float, lam: float, beta: float) -> bool:\n    """AdaEDL/SVIP-style rule: stop when 1 - sqrt(beta * H) < lam."""\n    return 1.0 - math.sqrt(beta * max(float(H), 0.0)) < lam\n\n\ndef first_fire(view: TraceView, p: int, g: int, lam: float, beta: float) -> int | None:\n    """Smallest g\' in 1..g at which the entropy rule fired (reads H[p..p+g-1] only)."""\n    for k in range(1, g + 1):\n        if entropy_fires(view.feat("H", p + k - 1), lam, beta):\n            return k\n    return None\n\n\nclass NoDraft(Policy):\n    name = "NoDraft"\n\n    def initial_stop(self, view, p):\n        return True\n\n    def stop(self, view, p, g):\n        return True\n\n\nclass FixedK(Policy):\n    name = "FixedK"\n\n    def __init__(self, k: int):\n        self.k = int(k)\n\n    def initial_stop(self, view, p):\n        return self.k <= 0\n\n    def stop(self, view, p, g):\n        return g >= self.k\n\n    def params(self):\n        return dict(k=self.k)\n\n\nclass Entropy(Policy):\n    name = "Entropy"\n\n    def __init__(self, lam: float, beta: float = 1.0):\n        self.lam, self.beta = float(lam), float(beta)\n\n    def stop(self, view, p, g):\n        return entropy_fires(view.feat("H", p + g - 1), self.lam, self.beta)\n\n    def params(self):\n        return dict(lam=self.lam, beta=self.beta)\n\n\nclass WordBoundarySoft(Policy):\n    """Stop when g >= min_g and the next draft distribution puts > tau mass on new-word tokens."""\n    name = "WordBoundarySoft"\n    lookahead = 1\n\n    def __init__(self, tau: float, min_g: int = 1):\n        self.tau, self.min_g = float(tau), int(min_g)\n\n    def stop(self, view, p, g):\n        return g >= self.min_g and view.feat("p_newword", p + g) > self.tau\n\n    def params(self):\n        return dict(tau=self.tau, min_g=self.min_g)\n\n\nclass WordBoundaryHard(Policy):\n    """Corrected legacy word-boundary policy: stop when g >= min_g and the draft\'s next token\n    (argmax of q_{p+g}) starts with space / punctuation or is eos. Needs the vocabulary mask."""\n    name = "WordBoundaryHard"\n    lookahead = 1\n\n    def __init__(self, min_g: int, newword_mask: np.ndarray):\n        self.min_g = int(min_g)\n        self.mask = np.asarray(newword_mask, dtype=bool)\n\n    def stop(self, view, p, g):\n        if g < self.min_g:\n            return False\n        t = view.token(p + g)\n        return bool(t < len(self.mask) and self.mask[t])\n\n    def params(self):\n        return dict(min_g=self.min_g)\n\n\nclass _EntropyExtend(Policy):\n    """Entropy gate + linguistic condition at the lookahead position, with up to max_ext extensions."""\n    lookahead = 1\n\n    def __init__(self, lam: float, beta: float, max_ext: int):\n        self.lam, self.beta, self.max_ext = float(lam), float(beta), int(max_ext)\n\n    def cond(self, view: TraceView, p: int, g: int) -> bool:\n        raise NotImplementedError\n\n    def stop(self, view, p, g):\n        g0 = first_fire(view, p, g, self.lam, self.beta)\n        if g0 is None:\n            return False\n        return self.cond(view, p, g) or (g - g0) >= self.max_ext\n\n\nclass AksharaSafeEntropy(_EntropyExtend):\n    """Entropy rule, but only stop where the next token is unlikely to be a combining mark\n    (p_mark[p+g] < tau_m), i.e. the last akshara is probably complete."""\n    name = "AksharaSafeEntropy"\n\n    def __init__(self, lam: float, beta: float, tau_m: float, max_ext: int):\n        super().__init__(lam, beta, max_ext)\n        self.tau_m = float(tau_m)\n\n    def cond(self, view, p, g):\n        return view.feat("p_mark", p + g) < self.tau_m\n\n    def params(self):\n        return dict(lam=self.lam, beta=self.beta, tau_m=self.tau_m, max_ext=self.max_ext)\n\n\nclass Hybrid(_EntropyExtend):\n    """Entropy rule, but only stop where the current word is probably complete (p_newword[p+g] > tau_w)."""\n    name = "Hybrid"\n\n    def __init__(self, lam: float, beta: float, tau_w: float, max_ext: int):\n        super().__init__(lam, beta, max_ext)\n        self.tau_w = float(tau_w)\n\n    def cond(self, view, p, g):\n        return view.feat("p_newword", p + g) > self.tau_w\n\n    def params(self):\n        return dict(lam=self.lam, beta=self.beta, tau_w=self.tau_w, max_ext=self.max_ext)\n\n\nclass Oracle(Policy):\n    """Upper bound: drafts exactly the leading run of matches (can draft 0), never rejects."""\n    name = "Oracle"\n    is_oracle = True\n\n    def initial_stop(self, view, p):\n        return not view.gold("match", p)\n\n    def stop(self, view, p, g):\n        return not view.gold("match", p + g)\n\n\nclass GoldWordOracle(Policy):\n    """Upper bound for word-boundary stopping: stop (g >= min_g) when the *gold* cut after token\n    p+g-1 of the target is a word boundary. Free lookahead; needs ``attach_word_labels``."""\n    name = "GoldWordOracle"\n    is_oracle = True\n\n    def __init__(self, min_g: int = 1):\n        self.min_g = int(min_g)\n\n    def stop(self, view, p, g):\n        return g >= self.min_g and view.gold("end_cut", p + g - 1) in (WORD, END)\n\n    def params(self):\n        return dict(min_g=self.min_g)\n\n\n# --------------------------------------------------------------------------- #\n# T6: morpheme policies\n# --------------------------------------------------------------------------- #\n\n@dataclass\nclass TokenTable:\n    """Per-vocabulary-id raw bytes and a word-start mask (space / punctuation / special)."""\n    token_bytes: list\n    word_start: np.ndarray\n\n    @classmethod\n    def from_tokenizer(cls, tok, masks=None) -> "TokenTable":\n        from .vocab import build_masks, token_bytes_list\n        masks = masks or build_masks(tok)\n        tb = token_bytes_list(tok, range(len(tok)))\n        return cls(tb, masks.starts_space | masks.starts_punct | masks.is_special)\n\n\ndef _complete_utf8(b: bytes) -> str:\n    for cut in range(len(b), max(len(b) - 4, -1), -1):\n        try:\n            return b[:cut].decode("utf-8")\n        except UnicodeDecodeError:\n            continue\n    return b.decode("utf-8", errors="ignore")\n\n\ndef current_word_prefix(view: TraceView, p: int, g: int, table: TokenTable, max_back: int = 32) -> str:\n    """Text of the current word after drafting g tokens: committed target tokens before p plus drafted\n    tokens p..p+g-1, scanning back to the last token that starts a word (that token\'s leading space\n    is stripped). Only complete UTF-8 characters are decoded."""\n    end = p + g - 1\n    parts = []\n    lo = max(0, end - max_back)\n    for idx in range(end, lo - 1, -1):\n        t = view.token(idx)\n        b = table.token_bytes[t] if t < len(table.token_bytes) else b""\n        parts.append(b)\n        if t < len(table.word_start) and table.word_start[t]:\n            break\n    raw = b"".join(reversed(parts)).lstrip(b" \\n\\t")\n    return _complete_utf8(raw)\n\n\ndef morph_prob(view, p: int, g: int, pm, table: TokenTable) -> float:\n    """pm(current word prefix) after drafting g tokens from p.\n\n    In the simulator the value at (trace, p, g) is the same for every config of every morph policy, so it\n    is cached on the trace object (keyed by the prefix model\'s unique ``uid``). It is computed the first time through the\n    causality-checked view, and (p, g) fixes exactly what the view allows, so reuse cannot leak\n    information. Live views (no ``trace``) are never cached."""\n    tr = getattr(view, "trace", None)\n    if tr is None:\n        return pm.prob(current_word_prefix(view, p, g, table))\n    cache = tr.__dict__.setdefault("_pm_cache", {})\n    key = (getattr(pm, "uid", id(pm)), p, g)\n    v = cache.get(key)\n    if v is None:\n        v = cache[key] = pm.prob(current_word_prefix(view, p, g, table))\n    return v\n\n\nclass MorphEntropy(_EntropyExtend):\n    """Entropy rule; stop where the causal prefix model predicts a morpheme boundary (pm > tau_morph).\n    If use_akshara (lookahead 1): the condition also holds when the word is complete\n    (p_newword[p+g] > 0.5), and additionally requires p_mark[p+g] < tau_m.\n    Without use_akshara the policy is causal (lookahead 0). Extends up to max_ext tokens."""\n    name = "MorphEntropy"\n\n    def __init__(self, lam: float, beta: float, tau_morph: float, use_akshara: bool, tau_m: float,\n                 max_ext: int, prefix_model, table: TokenTable):\n        super().__init__(lam, beta, max_ext)\n        self.tau_morph, self.use_akshara, self.tau_m = float(tau_morph), bool(use_akshara), float(tau_m)\n        self.pm, self.table = prefix_model, table\n        self.lookahead = 1 if self.use_akshara else 0\n\n    def cond(self, view, p, g):\n        ok = morph_prob(view, p, g, self.pm, self.table) > self.tau_morph\n        if self.use_akshara:\n            ok = (ok or view.feat("p_newword", p + g) > 0.5) and view.feat("p_mark", p + g) < self.tau_m\n        return ok\n\n    def params(self):\n        return dict(lam=self.lam, beta=self.beta, tau_morph=self.tau_morph, use_akshara=self.use_akshara,\n                    tau_m=self.tau_m, max_ext=self.max_ext)\n\n\nclass MorphOnly(Policy):\n    """Pure morpheme policy (no entropy, causal): stop when g >= min_g and pm > tau_morph."""\n    name = "MorphOnly"\n\n    def __init__(self, tau_morph: float, min_g: int, prefix_model, table: TokenTable):\n        self.tau_morph, self.min_g = float(tau_morph), int(min_g)\n        self.pm, self.table = prefix_model, table\n\n    def stop(self, view, p, g):\n        return g >= self.min_g and morph_prob(view, p, g, self.pm, self.table) > self.tau_morph\n\n    def params(self):\n        return dict(tau_morph=self.tau_morph, min_g=self.min_g)\n',
    'stats': '"""Bootstrap statistics over sentences (PLAN T7.4).\n\nSpeed-up on a resample = sum(emitted) / sum(cost) over the resampled sentences (ratio of sums, the\nsame estimator as the point value). Paired comparisons use the same resampled indices for both\npolicies, which must have been simulated on the same sentence list.\n"""\nfrom __future__ import annotations\n\nimport numpy as np\n\n\ndef _idx(n: int, n_boot: int, seed: int) -> np.ndarray:\n    return np.random.default_rng(seed).integers(0, n, size=(n_boot, n))\n\n\ndef bootstrap_ci(emitted, cost, n_boot: int = 1000, seed: int = 0, level: float = 0.95) -> dict:\n    e, c = np.asarray(emitted, float), np.asarray(cost, float)\n    I = _idx(len(e), n_boot, seed)\n    s = e[I].sum(1) / c[I].sum(1)\n    a = (1 - level) / 2\n    return dict(point=float(e.sum() / c.sum()), lo=float(np.quantile(s, a)), hi=float(np.quantile(s, 1 - a)),\n                n_boot=n_boot, seed=seed)\n\n\ndef paired_bootstrap(emitted_a, cost_a, emitted_b, cost_b, n_boot: int = 1000, seed: int = 0,\n                     level: float = 0.95) -> dict:\n    """Difference speedup(a) - speedup(b) with a paired bootstrap over sentences.\n    p_better = fraction of resamples where the difference is > 0."""\n    ea, ca = np.asarray(emitted_a, float), np.asarray(cost_a, float)\n    eb, cb = np.asarray(emitted_b, float), np.asarray(cost_b, float)\n    if not (len(ea) == len(ca) == len(eb) == len(cb)):\n        raise ValueError("paired bootstrap needs per-sentence arrays of equal length")\n    I = _idx(len(ea), n_boot, seed)\n    d = ea[I].sum(1) / ca[I].sum(1) - eb[I].sum(1) / cb[I].sum(1)\n    a = (1 - level) / 2\n    return dict(diff=float(ea.sum() / ca.sum() - eb.sum() / cb.sum()), lo=float(np.quantile(d, a)),\n                hi=float(np.quantile(d, 1 - a)), p_better=float((d > 0).mean()), n_boot=n_boot, seed=seed)\n\n\ndef bootstrap_stat_by_sentence(groups: list, fn, n_boot: int = 1000, seed: int = 0, level: float = 0.95) -> dict:\n    """Generic sentence-level bootstrap: ``groups`` is a list of per-sentence objects, ``fn`` maps a\n    list of them to a scalar."""\n    I = _idx(len(groups), n_boot, seed)\n    vals = np.array([fn([groups[i] for i in row]) for row in I], float)\n    n_valid = int(np.isfinite(vals).sum())\n    vals = vals[np.isfinite(vals)]\n    a = (1 - level) / 2\n    lo = float(np.quantile(vals, a)) if n_valid else float("nan")      # e.g. constant variable\n    hi = float(np.quantile(vals, 1 - a)) if n_valid else float("nan")\n    return dict(point=float(fn(groups)), lo=lo, hi=hi, n_boot=n_boot, n_valid=n_valid, seed=seed)\n',
    'unichar': '"""Tokenizer-independent text layer for the universal MTP head (PLAN_phase2_indic_universal_mtp, W1).\n\nThree pieces, none of which depends on any LLM tokenizer:\n\n1. ``CharCodec``: characters -> (unit id, script id). The nine major Brahmic scripts share one Unicode\n   layout (each block is 128 code points with the same offsets for the same letters), so a char in any\n   of them maps to ``BRAHMIC_BASE + (codepoint - block_start)`` plus a script id. A head trained on\n   Devanagari therefore sees Bengali / Gujarati / ... letters as the *same* units.\n2. ``segment_units``: text -> "units" = optional leading spaces + one word (letters, marks, digits,\n   ZWJ/ZWNJ) or one punctuation/symbol char; newline runs are separate units. Units concatenate back to\n   the text exactly. Leading spaces live inside the unit (" के"), which matches how every\n   space-splitting tokenizer (Llama-3, Qwen, Gemma, ...) attaches the space to the following word.\n3. ``UnitVocab``: the head\'s output vocabulary, built from public Hindi text (never from an LLM), sorted\n   by UTF-8 bytes so that "all units that start with these bytes" is one contiguous id range\n   (``prefix_range``). The last id is the end-of-output unit ``<eos>``.\n"""\nfrom __future__ import annotations\n\nimport bisect\nimport json\nimport unicodedata\nfrom collections import Counter\nfrom pathlib import Path\nfrom typing import Iterable, Sequence\n\nimport numpy as np\n\ntry:\n    import regex as _re   # supports \\p{..}; installed with transformers\nexcept ImportError:  # pragma: no cover\n    _re = None\n\n# --------------------------------------------------------------------------- #\n# 1. character codec\n# --------------------------------------------------------------------------- #\n\nSCRIPTS = ("none", "deva", "beng", "guru", "gujr", "orya", "taml", "telu", "knda", "mlym")\nBLOCK_START = {"deva": 0x0900, "beng": 0x0980, "guru": 0x0A00, "gujr": 0x0A80, "orya": 0x0B00,\n               "taml": 0x0B80, "telu": 0x0C00, "knda": 0x0C80, "mlym": 0x0D00}\nSCRIPT_ID = {s: i for i, s in enumerate(SCRIPTS)}\n\nPAD, UNK, BOS, EOS, SEP, PEND, NL = 0, 1, 2, 3, 4, 5, 6\nN_SPECIAL = 7\nASCII_BASE = N_SPECIAL                       # printable ASCII 32..126 -> 7..101\nBRAHMIC_BASE = ASCII_BASE + 95               # 102..229: offset within a Brahmic block\nEXTRA_CHARS = ("\u200c\u200d\xa0\\t“”‘’–—…«»·•°₹€£×÷।॥"\n               "¡¿§¶©®™µ‘’‚„′″")\nEXTRA_BASE = BRAHMIC_BASE + 128\nN_UNITS = EXTRA_BASE + len(EXTRA_CHARS)\n_EXTRA = {ch: EXTRA_BASE + i for i, ch in enumerate(EXTRA_CHARS)}\n_BLOCKS = sorted((v, k) for k, v in BLOCK_START.items())\n\n\ndef char_unit(ch: str) -> tuple[int, int]:\n    """(unit id, script id) of one character. Danda / double danda (U+0964/5) are used by several\n    scripts and get one shared unit."""\n    if ch == "\\n" or ch == "\\r":\n        return NL, 0\n    if ch in _EXTRA:\n        return _EXTRA[ch], 0\n    cp = ord(ch)\n    if 32 <= cp <= 126:\n        return ASCII_BASE + cp - 32, 0\n    if 0x0900 <= cp < 0x0D80:\n        start, script = _BLOCKS[bisect.bisect_right([b for b, _ in _BLOCKS], cp) - 1]\n        return BRAHMIC_BASE + cp - start, SCRIPT_ID[script]\n    return UNK, 0\n\n\nclass CharCodec:\n    """Vectorised text -> unit ids with a per-codepoint cache."""\n\n    n_units = N_UNITS\n    n_scripts = len(SCRIPTS)\n\n    def __init__(self):\n        self._cache: dict[str, tuple[int, int]] = {}\n\n    def encode(self, text: str) -> tuple[np.ndarray, np.ndarray]:\n        u = np.empty(len(text), np.int16)\n        s = np.empty(len(text), np.int8)\n        cache = self._cache\n        for i, ch in enumerate(text):\n            r = cache.get(ch)\n            if r is None:\n                r = cache[ch] = char_unit(ch)\n            u[i], s[i] = r\n        return u, s\n\n    @staticmethod\n    def decode(units: Sequence[int], script: str = "deva") -> str:\n        """Inverse of ``encode`` for one target script (unknown / special units are dropped)."""\n        out, start = [], BLOCK_START.get(script, 0x0900)\n        inv_extra = {v: k for k, v in _EXTRA.items()}\n        for u in units:\n            u = int(u)\n            if u == NL:\n                out.append("\\n")\n            elif ASCII_BASE <= u < BRAHMIC_BASE:\n                out.append(chr(u - ASCII_BASE + 32))\n            elif BRAHMIC_BASE <= u < EXTRA_BASE:\n                off = u - BRAHMIC_BASE\n                out.append("।॥"[off - 0x64] if off in (0x64, 0x65) else chr(start + off))\n            elif u in inv_extra:\n                out.append(inv_extra[u])\n        return "".join(out)\n\n\ndef transliterate(text: str, to_script: str) -> str:\n    """Map every Brahmic char to the same offset in ``to_script`` (the unified-layout identity the codec\n    relies on). Used in tests and for the zero-shot-script experiments of later weeks."""\n    start = BLOCK_START[to_script]\n    out = []\n    for ch in text:\n        cp = ord(ch)\n        if 0x0900 <= cp < 0x0D80 and ch not in "।॥":\n            b = _BLOCKS[bisect.bisect_right([x for x, _ in _BLOCKS], cp) - 1][0]\n            out.append(chr(start + cp - b))\n        else:\n            out.append(ch)\n    return "".join(out)\n\n\n# --------------------------------------------------------------------------- #\n# 2. units\n# --------------------------------------------------------------------------- #\n\nZW = "\u200c\u200d"\nif _re is not None:\n    UNIT_RE = _re.compile(r"[\\r\\n]+|[ \\t\xa0]*(?:[\\p{L}\\p{M}\\p{N}\u200c\u200d]+|[^\\s\\p{L}\\p{M}\\p{N}])"\n                          r"|[ \\t\xa0]+|.", _re.DOTALL)\n\n\ndef is_word_char(ch: str) -> bool:\n    return unicodedata.category(ch)[0] in "LMN" or ch in ZW\n\n\ndef segment_units(text: str) -> list[tuple[int, int]]:\n    """Char spans (start, end) of the units of ``text``; they tile the text exactly."""\n    if _re is None:  # pragma: no cover\n        raise ImportError("the \'regex\' package is required")\n    return [m.span() for m in UNIT_RE.finditer(text)]\n\n\ndef unit_strings(text: str) -> list[str]:\n    return [text[a:b] for a, b in segment_units(text)]\n\n\ndef continues_unit(ch: str) -> bool:\n    """Inference rule for a pending (incomplete) UTF-8 char: it continues the last unit if the last\n    complete char is a word char or a space/tab (a word can follow spaces in the same unit)."""\n    return is_word_char(ch) or ch in " \\t\xa0"\n\n\n# --------------------------------------------------------------------------- #\n# 3. unit vocabulary\n# --------------------------------------------------------------------------- #\n\nEOS_UNIT = "<eos>"\n\n\nclass UnitVocab:\n    """Sorted-by-bytes unit list; id ``len(units)`` is ``<eos>``. ``size`` includes ``<eos>``."""\n\n    def __init__(self, units: Iterable[str]):\n        uniq = sorted(set(u for u in units if u and u != EOS_UNIT), key=lambda s: s.encode("utf-8"))\n        self.units: list[str] = uniq\n        self.bytes: list[bytes] = [u.encode("utf-8") for u in uniq]\n        self.index: dict[str, int] = {u: i for i, u in enumerate(uniq)}\n        self.eos_id = len(uniq)\n        self.size = len(uniq) + 1\n\n    def id(self, unit: str) -> int:\n        """Unit id, ``-100`` (ignore index) if out of vocabulary."""\n        return self.index.get(unit, -100)\n\n    def prefix_range(self, prefix: bytes) -> tuple[int, int]:\n        """Ids [lo, hi) of all units whose UTF-8 bytes start with ``prefix`` (0xFF never occurs in UTF-8)."""\n        lo = bisect.bisect_left(self.bytes, prefix)\n        hi = bisect.bisect_left(self.bytes, prefix + b"\\xff")\n        return lo, hi\n\n    def unit_bytes(self, i: int) -> bytes:\n        return b"" if i == self.eos_id else self.bytes[i]\n\n    def save(self, path) -> None:\n        Path(path).write_text(json.dumps(dict(units=self.units), ensure_ascii=False), encoding="utf-8")\n\n    @classmethod\n    def load(cls, path) -> "UnitVocab":\n        return cls(json.loads(Path(path).read_text(encoding="utf-8"))["units"])\n\n    @classmethod\n    def build(cls, texts: Iterable[str], max_size: int = 50000, min_count: int = 2) -> tuple["UnitVocab", dict]:\n        cnt: Counter = Counter()\n        n_texts = 0\n        for t in texts:\n            cnt.update(unit_strings(t))\n            n_texts += 1\n        keep = [u for u, c in cnt.most_common(max_size) if c >= min_count]\n        total = sum(cnt.values())\n        covered = sum(cnt[u] for u in keep)\n        V = cls(keep)\n        return V, dict(texts=n_texts, unit_tokens=total, unit_types=len(cnt), vocab=V.size,\n                       train_coverage=covered / max(total, 1))\n\n    def coverage(self, texts: Iterable[str]) -> float:\n        n = hit = 0\n        for t in texts:\n            for u in unit_strings(t):\n                n += 1\n                hit += u in self.index\n        return hit / max(n, 1)\n',
    'uni_features': '"""Tokenizer-independent inputs and unit targets for the universal head (PLAN_phase2_indic_universal_mtp, W1).\n\nA *record* is one generated output of any LLM:\n    dict(id, prompt_text, out_ids [n] (incl. a final eos id if the model stopped), eos_end: bool,\n         topk_ids [n, K], topk_logp [n, K])          # row t = target distribution q_t over out_ids[t]\n\nIndex convention (as in Phase 1): position p (0 <= p < n) means "out_ids[:p] are committed". At p the\nhead may read the committed text and the distributions q_t for t < p (each committed token was\nproduced by a target pass that also gave q_t). It must predict the text that follows. Nothing here\nreads q_t for t >= p, so features at p are causal by construction (tested in tests/test_uni_features.py).\n\nEverything the head sees is text: characters (``unichar.CharCodec``) and, for every candidate token of\nthe last M distributions, its *string* relative to the committed text plus its log-probability. The\nLLM\'s token ids never reach the head, which is why one head can serve any LLM.\n"""\nfrom __future__ import annotations\n\nfrom dataclasses import dataclass, field\nfrom pathlib import Path\n\nimport numpy as np\n\nfrom .boundaries import token_to_bytes\nfrom .unichar import EOS, PAD, PEND, SEP, CharCodec, UnitVocab, continues_unit, segment_units\n\nK_CAND, S_CHARS, M_LAGS, K_HEADS, L_PROMPT, L_OUT = 20, 8, 3, 3, 160, 96\nF_STATIC = 7          # logp/10, p, prefix mass, len/8, is_eos, incomplete, valid\nF_DYN = 3             # delta/16, consistent with committed text, extends beyond it\nN_FEAT = F_STATIC + F_DYN\n\n\n# --------------------------------------------------------------------------- #\n# bytes / text helpers\n# --------------------------------------------------------------------------- #\n\ndef utf8_offsets(b: bytes) -> tuple[str, np.ndarray]:\n    """Decode ``b``; return (text, char byte offsets of length len(text)+1).\n\n    Invalid bytes become U+FFFD (one byte each). A trailing incomplete char is left out, so\n    ``offsets[-1]`` is the end of the last complete char (can be < len(b))."""\n    try:\n        text = b.decode("utf-8")\n        a = np.frombuffer(b, np.uint8)\n        starts = np.flatnonzero((a & 0xC0) != 0x80)\n        return text, np.append(starts, len(b)).astype(np.int64)\n    except UnicodeDecodeError:\n        pass\n    chars, offs, i, n = [], [], 0, len(b)\n    while i < n:\n        x = b[i]\n        L = 1 if x < 0x80 else 2 if 0xC2 <= x <= 0xDF else 3 if 0xE0 <= x <= 0xEF else 4 if 0xF0 <= x <= 0xF4 else 0\n        if L == 0 or (0x80 <= x <= 0xBF):\n            chars.append("�"); offs.append(i); i += 1\n            continue\n        if i + L > n:\n            if all(0x80 <= y <= 0xBF for y in b[i + 1:]):\n                break                                  # trailing incomplete char\n            chars.append("�"); offs.append(i); i += 1\n            continue\n        try:\n            chars.append(b[i:i + L].decode("utf-8")); offs.append(i); i += L\n        except UnicodeDecodeError:\n            chars.append("�"); offs.append(i); i += 1\n    return "".join(chars), np.array(offs + [i], np.int64)   # i = end of the last complete char\n\n\ndef special_id_set(tok) -> set[int]:\n    from .vocab import special_ids\n    return special_ids(tok)\n\n\nclass TokTable:\n    """Static per-tokenizer tables: token bytes and each token\'s chars when read on its own."""\n\n    def __init__(self, tok, eos_ids, vocab_size: int | None = None, codec: CharCodec | None = None,\n                 S: int = S_CHARS):\n        codec = codec or CharCodec()\n        n = len(tok)\n        V = max(n, int(vocab_size or 0))\n        self.size = V\n        self.eos_ids = sorted(set(int(i) for i in eos_ids))\n        spec = special_id_set(tok) | set(self.eos_ids)\n        toks = tok.convert_ids_to_tokens(list(range(n)))\n        self.bytes: list[bytes] = [b""] * V\n        self.special = np.ones(V, bool)\n        for i, t in enumerate(toks):\n            if i in spec or t is None:\n                continue\n            self.bytes[i] = token_to_bytes(t)\n            self.special[i] = False\n        self.is_eos = np.zeros(V, bool)\n        self.is_eos[[i for i in self.eos_ids if i < V]] = True\n        self.chars = np.zeros((V, S), np.int16)\n        self.scr = np.zeros((V, S), np.int8)\n        self.clen = np.zeros(V, np.int8)\n        self.cont_start = np.zeros(V, bool)\n        self.incomplete = np.zeros(V, bool)\n        for i in range(V):\n            if self.is_eos[i]:\n                self.chars[i, 0], self.clen[i] = EOS, 1\n                continue\n            b = self.bytes[i]\n            if not b:\n                continue\n            if 0x80 <= b[0] <= 0xBF:\n                self.cont_start[i] = True\n                continue\n            text, offs = utf8_offsets(b)\n            self.incomplete[i] = offs[-1] < len(b)\n            u, s = codec.encode(text[:S])\n            self.chars[i, :len(u)], self.scr[i, :len(u)], self.clen[i] = u, s, len(u)\n        self.valid = ~(self.special & ~self.is_eos) & ~self.cont_start\n\n    def seq_bytes(self, ids) -> list[bytes]:\n        return [self.bytes[int(i)] if int(i) < self.size else b"" for i in ids]\n\n\ndef make_encoder(tok):\n    """``encode(list[str]) -> list[list[int]]`` without special tokens and without a SentencePiece dummy\n    prefix space (detected once; then the text is encoded after a newline that is stripped again)."""\n    def raw(texts):\n        return tok(list(texts), add_special_tokens=False)["input_ids"]\n\n    probe = raw(["x"])[0]\n    pb = b"".join(token_to_bytes(t) for t in tok.convert_ids_to_tokens(probe))\n    if pb == b"x":\n        return raw\n\n    def strip_nl(ids):\n        acc = b""\n        for j, t in enumerate(tok.convert_ids_to_tokens(ids)):\n            acc += token_to_bytes(t)\n            if acc.strip(b" ") == b"\\n":\n                return ids[j + 1:]\n            if len(acc) > 2:\n                break\n        return None\n\n    def enc(texts):\n        texts = list(texts)\n        out = raw(["\\n" + t for t in texts])\n        res = []\n        for t, ids in zip(texts, out):\n            s = strip_nl(ids)\n            res.append(s if s is not None else raw([t])[0])\n        return res\n    return enc\n\n\n# --------------------------------------------------------------------------- #\n# per-sequence positions and unit targets\n# --------------------------------------------------------------------------- #\n\n@dataclass\nclass SeqText:\n    """Committed-text bookkeeping of one output sequence."""\n    out_bytes: bytes\n    text: str\n    coff: np.ndarray          # char byte offsets, len(text)+1\n    b_tok: np.ndarray         # committed bytes after p tokens, p = 0..n\n    c: np.ndarray             # complete chars committed after p tokens\n    pend: np.ndarray          # bytes of an incomplete char committed after p tokens\n    unit_start: np.ndarray    # char start of each unit\n    unit_of_char: np.ndarray\n    unit_strs: list = field(default_factory=list)\n\n\ndef seq_text(table: TokTable, out_ids) -> SeqText:\n    bl = table.seq_bytes(out_ids)\n    b_tok = np.zeros(len(bl) + 1, np.int64)\n    b_tok[1:] = np.cumsum([len(b) for b in bl])\n    out_bytes = b"".join(bl)\n    return text_positions(out_bytes, b_tok)\n\n\ndef text_positions(out_bytes: bytes, byte_cuts: np.ndarray) -> SeqText:\n    text, coff = utf8_offsets(out_bytes)\n    c = np.searchsorted(coff[1:], byte_cuts, side="right")\n    pend = byte_cuts - coff[c]\n    spans = segment_units(text)\n    us = np.array([a for a, _ in spans], np.int64)\n    uoc = np.repeat(np.arange(len(spans)), [b - a for a, b in spans]) if spans else np.zeros(0, np.int64)\n    return SeqText(out_bytes, text, coff, byte_cuts.astype(np.int64), c, pend, us, uoc,\n                   [text[a:b] for a, b in spans])\n\n\ndef partial_start(st: SeqText, i: int) -> int:\n    """Char index where the unit being written at cut i starts (inference rule, needs no future text)."""\n    cp, pl = int(st.c[i]), int(st.pend[i])\n    if cp == 0:\n        return 0\n    if pl and not continues_unit(st.text[cp - 1]):\n        return cp\n    return int(st.unit_start[st.unit_of_char[cp - 1]])\n\n\ndef position_targets(st: SeqText, vocab: UnitVocab, eos_end: bool, k: int = K_HEADS, cuts=None):\n    """For each cut i (default: all but the last): partial-unit start byte, head-1 prefix range\n    [lo, hi) and the k unit targets (-100 = ignore). Head 1 targets the full unit being written (the\n    committed part is a prefix of it); heads 2..k target the following units, then <eos>."""\n    cuts = range(len(st.b_tok) - 1) if cuts is None else cuts\n    uid = [vocab.id(u) for u in st.unit_strs]\n    start2unit = {int(a): j for j, a in enumerate(st.unit_start)}\n    nU = len(uid)\n    ps_b, lo, hi, tg = [], [], [], []\n    for i in cuts:\n        ps = partial_start(st, i)\n        pb = int(st.coff[ps])\n        part = st.out_bytes[pb:int(st.b_tok[i])]\n        a, b = vocab.prefix_range(part)\n        j = start2unit.get(ps)\n        row = [-100] * k\n        if j is not None:\n            for h in range(k):\n                jj = j + h\n                if jj < nU:\n                    row[h] = uid[jj]\n                elif jj == nU and eos_end:\n                    row[h] = vocab.eos_id\n                    break\n                else:\n                    break\n            if not (a <= row[0] < b):\n                row[0] = -100\n        ps_b.append(pb); lo.append(a); hi.append(b); tg.append(row)\n    return (np.array(ps_b, np.int64), np.array(lo, np.int32), np.array(hi, np.int32),\n            np.array(tg, np.int32).reshape(-1, k))\n\n\n# --------------------------------------------------------------------------- #\n# candidates (the target\'s top-K, re-expressed as strings)\n# --------------------------------------------------------------------------- #\n\ndef candidate_arrays(st: SeqText, table: TokTable, topk_ids, topk_logp, text_u, codec: CharCodec,\n                     eos_end: bool, S: int = S_CHARS):\n    """Per token t: chars of each candidate relative to the committed text before t, its length,\n    common-prefix length with the actual text from the anchor (only compared up to the committed end\n    when used), static features, validity."""\n    n, K = topk_ids.shape\n    ids = np.asarray(topk_ids, np.int64)\n    ids = np.where(ids < table.size, ids, 0)\n    cu = table.chars[ids].copy()                 # (n, K, S)\n    cs = table.scr[ids].copy()\n    cl = table.clen[ids].astype(np.int16)\n    inc = table.incomplete[ids].copy()\n    val = table.valid[ids].copy()\n    val &= np.asarray(topk_ids) < table.size\n    for t in np.flatnonzero(st.pend[:n] > 0):    # tokens that start inside a char: decode pend + bytes\n        pb = st.out_bytes[int(st.coff[st.c[t]]):int(st.b_tok[t])]\n        for j in range(K):\n            v = int(ids[t, j])\n            cu[t, j], cs[t, j], cl[t, j], inc[t, j], val[t, j] = 0, 0, 0, False, False\n            if table.is_eos[v] or table.special[v]:\n                continue\n            txt, offs = utf8_offsets(pb + table.bytes[v])\n            if not txt or txt[0] == "�":\n                continue\n            u, s = codec.encode(txt[:S])\n            cu[t, j, :len(u)], cs[t, j, :len(u)], cl[t, j] = u, s, len(u)\n            inc[t, j], val[t, j] = offs[-1] < len(pb) + len(table.bytes[v]), True\n    # common prefix with the real text from the anchor char c[t]\n    ext = np.concatenate([text_u, np.array([EOS] if eos_end else [], np.int16), np.zeros(S, np.int16)])\n    W = ext[st.c[:n, None] + np.arange(S)[None]]                    # (n, S)\n    sidx = np.arange(S)\n    eq = (cu == W[:, None, :]) & (sidx[None, None] < cl[..., None])\n    lmatch = np.cumprod(eq, axis=2).sum(2).astype(np.int8)\n    # char-prefix mass: sum of probabilities of the candidates whose string starts with this one\n    p = np.exp(np.clip(np.asarray(topk_logp, np.float32), -30, 0)) * val\n    same = cu[:, None, :, :] == cu[:, :, None, :]                    # [n, i, j, s]\n    within = sidx[None, None, None] < cl[:, :, None, None]           # s < len_i\n    starts = np.all(same | ~within, axis=3) & (cl[:, None, :] >= cl[:, :, None]) & val[:, None, :]\n    pmass = (starts * p[:, None, :]).sum(2)\n    lp = np.clip(np.asarray(topk_logp, np.float32), -30, 0)\n    f = np.stack([lp / 10, p, pmass, cl / 8, table.is_eos[ids], inc, val], -1).astype(np.float16)\n    return cu, cs, cl.astype(np.int8), lmatch, f, val\n\n\n# --------------------------------------------------------------------------- #\n# FeatureSet: flat arrays for many sequences, vectorised batch building\n# --------------------------------------------------------------------------- #\n\nARRAYS = ("prompt_u", "prompt_s", "text_u", "text_s", "seq_char0", "seq_tok0", "pos_seq", "pos_c", "pos_pend",\n          "pos_tok", "tgt", "lo", "hi", "cand_u", "cand_s", "cand_len", "cand_match", "cand_f", "cand_valid")\n\n\n@dataclass\nclass FeatureSet:\n    prompt_u: np.ndarray\n    prompt_s: np.ndarray\n    text_u: np.ndarray\n    text_s: np.ndarray\n    seq_char0: np.ndarray\n    seq_tok0: np.ndarray\n    pos_seq: np.ndarray\n    pos_c: np.ndarray\n    pos_pend: np.ndarray\n    pos_tok: np.ndarray\n    tgt: np.ndarray\n    lo: np.ndarray\n    hi: np.ndarray\n    cand_u: np.ndarray = None\n    cand_s: np.ndarray = None\n    cand_len: np.ndarray = None\n    cand_match: np.ndarray = None\n    cand_f: np.ndarray = None\n    cand_valid: np.ndarray = None\n    seq_ids: list = field(default_factory=list)\n    meta: dict = field(default_factory=dict)\n\n    @property\n    def has_cands(self) -> bool:\n        return self.cand_u is not None\n\n    @property\n    def n_pos(self) -> int:\n        return len(self.pos_seq)\n\n    def trainable(self) -> np.ndarray:\n        return np.flatnonzero((self.tgt != -100).any(1))\n\n    def split_by_seq(self, frac: float, seed: int = 0):\n        """Position indices (train, val) with ``frac`` of the sequences in val."""\n        rng = np.random.default_rng(seed)\n        n_seq = len(self.prompt_u)\n        val_seq = rng.random(n_seq) < frac\n        tr = self.trainable()\n        m = val_seq[self.pos_seq[tr]]\n        return tr[~m], tr[m]\n\n    def save(self, path) -> None:\n        import json\n        d = {k: getattr(self, k) for k in ARRAYS if getattr(self, k) is not None}\n        np.savez(path, **d, _seq_ids=np.array(self.seq_ids, np.int64),\n                 _meta=np.array(json.dumps(self.meta, ensure_ascii=False)))\n\n    @classmethod\n    def load(cls, path) -> "FeatureSet":\n        import json\n        z = np.load(path, allow_pickle=False)\n        kw = {k: z[k] for k in ARRAYS if k in z.files}\n        return cls(**kw, seq_ids=z["_seq_ids"].tolist(), meta=json.loads(str(z["_meta"])))\n\n\ndef concat(sets: list[FeatureSet]) -> FeatureSet:\n    if len(sets) == 1:\n        return sets[0]\n    hc = {s.has_cands for s in sets}\n    if len(hc) != 1:\n        raise ValueError("cannot concatenate feature sets with and without candidates")\n    out = {k: [] for k in ARRAYS}\n    ns = nc = nt = 0\n    for s in sets:\n        for k in ("prompt_u", "prompt_s", "text_u", "text_s", "pos_pend", "tgt", "lo", "hi"):\n            out[k].append(getattr(s, k))\n        out["seq_char0"].append(s.seq_char0 + nc)\n        out["seq_tok0"].append(s.seq_tok0 + nt)\n        out["pos_seq"].append(s.pos_seq + ns)\n        out["pos_c"].append(s.pos_c + nc)\n        out["pos_tok"].append(np.where(s.pos_tok >= 0, s.pos_tok + nt, -1))\n        if s.has_cands:\n            for k in ("cand_u", "cand_s", "cand_len", "cand_match", "cand_f", "cand_valid"):\n                out[k].append(getattr(s, k))\n        ns += len(s.prompt_u)\n        nc += len(s.text_u)\n        nt += len(s.cand_u) if s.has_cands else 0\n    kw = {k: (np.concatenate(v) if v else None) for k, v in out.items()}\n    return FeatureSet(**kw, seq_ids=sum((s.seq_ids for s in sets), []),\n                      meta=dict(parts=[s.meta for s in sets]))\n\n\ndef _prompt_arrays(texts, codec: CharCodec, L: int):\n    pu = np.zeros((len(texts), L), np.int16)\n    ps = np.zeros((len(texts), L), np.int8)\n    for i, t in enumerate(texts):\n        u, s = codec.encode((t or "")[-L:])\n        if len(u):\n            pu[i, -len(u):], ps[i, -len(u):] = u, s\n    return pu, ps\n\n\ndef featurize_records(records: list[dict], table: TokTable, vocab: UnitVocab, codec: CharCodec | None = None,\n                      k: int = K_HEADS, with_cands: bool = True, meta: dict | None = None) -> FeatureSet:\n    """One example per position p = 0..n-1 of every record (positions are global token indices)."""\n    codec = codec or CharCodec()\n    text_u, text_s, cand = [], [], {k_: [] for k_ in ("cu", "cs", "cl", "lm", "f", "v")}\n    pos_c, pos_pend, tgt, lo, hi, pos_seq = [], [], [], [], [], []\n    seq_char0, seq_tok0 = [], []\n    nc = nt = 0\n    for si, r in enumerate(records):\n        out_ids = list(r["out_ids"])\n        n = len(out_ids)\n        st = seq_text(table, out_ids)\n        u, s = codec.encode(st.text)\n        _, l_, h_, t_ = position_targets(st, vocab, bool(r.get("eos_end", True)), k)\n        seq_char0.append(nc); seq_tok0.append(nt)\n        text_u.append(u); text_s.append(s)\n        pos_c.append(st.c[:n] + nc); pos_pend.append(st.pend[:n] > 0)\n        tgt.append(t_); lo.append(l_); hi.append(h_); pos_seq.append(np.full(n, si, np.int32))\n        if with_cands:\n            cu, cs, cl, lm, f, v = candidate_arrays(st, table, np.asarray(r["topk_ids"])[:n],\n                                                    np.asarray(r["topk_logp"])[:n], u, codec,\n                                                    bool(r.get("eos_end", True)))\n            for k_, a in zip(("cu", "cs", "cl", "lm", "f", "v"), (cu, cs, cl, lm, f, v)):\n                cand[k_].append(a)\n        nc += len(u); nt += n\n    pu, ps = _prompt_arrays([r.get("prompt_text", "") for r in records], codec, L_PROMPT)\n    cat = lambda xs, dt: np.concatenate(xs).astype(dt) if xs else np.zeros(0, dt)\n    fs = FeatureSet(prompt_u=pu, prompt_s=ps, text_u=cat(text_u, np.int16), text_s=cat(text_s, np.int8),\n                    seq_char0=np.array(seq_char0, np.int64), seq_tok0=np.array(seq_tok0, np.int64),\n                    pos_seq=cat(pos_seq, np.int32), pos_c=cat(pos_c, np.int64), pos_pend=cat(pos_pend, bool),\n                    pos_tok=np.arange(nt, dtype=np.int64),\n                    tgt=np.concatenate(tgt) if tgt else np.zeros((0, k), np.int32),\n                    lo=cat(lo, np.int32), hi=cat(hi, np.int32), seq_ids=[int(r["id"]) for r in records],\n                    meta=dict(meta or {}, n_seq=len(records), n_pos=nt))\n    if with_cands:\n        fs.cand_u, fs.cand_s = np.concatenate(cand["cu"]), np.concatenate(cand["cs"])\n        fs.cand_len, fs.cand_match = np.concatenate(cand["cl"]), np.concatenate(cand["lm"])\n        fs.cand_f, fs.cand_valid = np.concatenate(cand["f"]), np.concatenate(cand["v"])\n    return fs\n\n\ndef featurize_texts(texts: list[str], vocab: UnitVocab, codec: CharCodec | None = None, cuts_per_text: int = 8,\n                    seed: int = 0, k: int = K_HEADS) -> FeatureSet:\n    """Text-only examples from plain Hindi text: random char cuts, no prompt, no candidates, no <eos>."""\n    codec = codec or CharCodec()\n    rng = np.random.default_rng(seed)\n    text_u, text_s, pos_c, tgt, lo, hi, pos_seq, seq_char0 = [], [], [], [], [], [], [], []\n    nc = 0\n    kept = 0\n    for t in texts:\n        if not t:\n            continue\n        b = t.encode("utf-8")\n        _, coff = utf8_offsets(b)\n        nchar = len(coff) - 1\n        cuts_c = np.unique(rng.integers(0, nchar + 1, size=min(cuts_per_text, nchar + 1)))\n        st = text_positions(b, coff[cuts_c])\n        u, s = codec.encode(st.text)\n        _, l_, h_, t_ = position_targets(st, vocab, False, k, cuts=range(len(cuts_c)))\n        seq_char0.append(nc); text_u.append(u); text_s.append(s)\n        pos_c.append(st.c + nc); tgt.append(t_); lo.append(l_); hi.append(h_)\n        pos_seq.append(np.full(len(cuts_c), kept, np.int32))\n        nc += len(u); kept += 1\n    n = sum(len(x) for x in pos_c)\n    return FeatureSet(prompt_u=np.zeros((kept, L_PROMPT), np.int16), prompt_s=np.zeros((kept, L_PROMPT), np.int8),\n                      text_u=np.concatenate(text_u).astype(np.int16), text_s=np.concatenate(text_s).astype(np.int8),\n                      seq_char0=np.array(seq_char0, np.int64), seq_tok0=np.zeros(kept, np.int64),\n                      pos_seq=np.concatenate(pos_seq), pos_c=np.concatenate(pos_c).astype(np.int64),\n                      pos_pend=np.zeros(n, bool), pos_tok=np.full(n, -1, np.int64),\n                      tgt=np.concatenate(tgt).astype(np.int32), lo=np.concatenate(lo).astype(np.int32),\n                      hi=np.concatenate(hi).astype(np.int32), seq_ids=list(range(kept)),\n                      meta=dict(kind="mono", n_seq=kept, n_pos=n))\n\n\ndef make_batch(fs: FeatureSet, idx, use_cands: bool = True, M: int = M_LAGS, L_O: int = L_OUT) -> dict:\n    """Vectorised inputs for positions ``idx`` (numpy). Shapes: ctx (B, L_P+1+L_O+1); cands (B, M*K, S)."""\n    idx = np.asarray(idx)\n    B = len(idx)\n    seq = fs.pos_seq[idx]\n    c = fs.pos_c[idx]\n    offs = c[:, None] - L_O + np.arange(L_O)[None]\n    ok = offs >= fs.seq_char0[seq][:, None]\n    oc = np.clip(offs, 0, max(len(fs.text_u) - 1, 0))\n    out_u = np.where(ok, fs.text_u[oc] if len(fs.text_u) else 0, PAD).astype(np.int64)\n    out_s = np.where(ok, fs.text_s[oc] if len(fs.text_s) else 0, 0).astype(np.int64)\n    pend = np.where(fs.pos_pend[idx], PEND, PAD)[:, None]\n    pu = fs.prompt_u[seq].astype(np.int64)\n    L_P = pu.shape[1]\n    ctx_u = np.concatenate([pu, np.full((B, 1), SEP), out_u, pend], 1)\n    ctx_s = np.concatenate([fs.prompt_s[seq].astype(np.int64), np.zeros((B, 1), np.int64), out_s,\n                            np.zeros((B, 1), np.int64)], 1)\n    ctx_t = np.concatenate([np.ones(L_P, np.int64), [2], np.full(L_O, 3), [4]])\n    ctx_p = np.concatenate([np.arange(L_P)[::-1], [0], np.arange(L_O)[::-1], [0]])\n    ctx_t = np.where(ctx_u == PAD, 0, ctx_t[None])\n    batch = dict(ctx_u=ctx_u, ctx_s=ctx_s, ctx_t=ctx_t, ctx_p=np.broadcast_to(ctx_p, ctx_u.shape).copy(),\n                 tgt=fs.tgt[idx].astype(np.int64), lo=fs.lo[idx].astype(np.int64), hi=fs.hi[idx].astype(np.int64))\n    if not (use_cands and fs.has_cands):\n        return batch\n    K, S = fs.cand_u.shape[1], fs.cand_u.shape[2]\n    tok = fs.pos_tok[idx]\n    lags = np.arange(1, M + 1)\n    t = tok[:, None] - lags[None]\n    tv = (tok[:, None] >= 0) & (t >= fs.seq_tok0[seq][:, None])\n    tc = np.clip(t, 0, None)\n    cu = fs.cand_u[tc].astype(np.int64)                        # (B, M, K, S)\n    cl = fs.cand_len[tc].astype(np.int64)\n    d = (c[:, None] - fs.pos_c[tc])[:, :, None]                # chars committed since the anchor\n    cons = fs.cand_match[tc] >= np.minimum(d, cl)\n    extd = cons & (cl > d)\n    sidx = d[..., None] + np.arange(S)\n    ext = np.take_along_axis(cu, np.clip(sidx, 0, S - 1), axis=3)\n    ext = np.where((sidx < cl[..., None]) & extd[..., None], ext, PAD)\n    valid = tv[:, :, None] & fs.cand_valid[tc]\n    f = np.concatenate([fs.cand_f[tc].astype(np.float32),\n                        np.minimum(np.broadcast_to(d, cl.shape), 32)[..., None] / 16.0,\n                        cons[..., None], extd[..., None]], -1)\n    f = np.where(valid[..., None], f, 0)\n    MK = M * K\n    batch.update(cand_u=np.where(valid[..., None], cu, PAD).reshape(B, MK, S),\n                 cand_s=fs.cand_s[tc].astype(np.int64).reshape(B, MK, S),\n                 ext_u=ext.reshape(B, MK, S), cand_f=f.reshape(B, MK, -1).astype(np.float32),\n                 cand_lag=np.broadcast_to(np.repeat(lags, K)[None], (B, MK)).copy(),\n                 cand_rank=np.broadcast_to(np.tile(np.arange(K), M)[None], (B, MK)).copy(),\n                 cand_mask=valid.reshape(B, MK))\n    return batch\n',
    'unihead': '"""The universal head: one small network that drafts the next k word units for any LLM (W1).\n\nInput (all text, see ``uni_features``): prompt chars, committed output chars, and for each of the last\nM target distributions its top-K candidate strings with probabilities. Output: k unit distributions in\none pass. Head 1 is restricted to units that extend the partial unit being written (``lo..hi``), heads\n2..k are chained on the previous unit\'s embedding (teacher forcing in training, argmax at inference).\nThe text-only ablation is the same network without the candidate tokens.\n"""\nfrom __future__ import annotations\n\nimport json\nimport math\nimport time\nfrom pathlib import Path\n\nimport numpy as np\n\nfrom .uni_features import K_CAND, K_HEADS, L_OUT, L_PROMPT, M_LAGS, N_FEAT, S_CHARS, FeatureSet, make_batch\nfrom .unichar import N_UNITS, PAD, SCRIPTS\n\n\ndef _torch():\n    import torch\n    return torch\n\n\nDEFAULT_ARCH = dict(d=256, layers=2, nhead=4, k=K_HEADS, S=S_CHARS, M=M_LAGS, K=K_CAND, F=N_FEAT,\n                    L_max=max(L_PROMPT, L_OUT) + 2, dropout=0.1)\n\n\ndef build_model(n_vocab: int, arch: dict | None = None):\n    torch = _torch()\n    nn = torch.nn\n    a = {**DEFAULT_ARCH, **(arch or {})}\n\n    class UniHead(nn.Module):\n        def __init__(self):\n            super().__init__()\n            d = a["d"]\n            self.arch, self.n_vocab = a, n_vocab\n            self.char = nn.Embedding(N_UNITS, d, padding_idx=PAD)\n            self.script = nn.Embedding(len(SCRIPTS), d)\n            self.ctype = nn.Embedding(6, d)\n            self.pos = nn.Embedding(a["L_max"], d)\n            self.spos = nn.Embedding(a["S"], d)\n            self.epos = nn.Embedding(a["S"], d)\n            self.cfeat = nn.Linear(a["F"], d)\n            self.lag = nn.Embedding(a["M"] + 1, d)\n            self.rank = nn.Embedding(a["K"], d)\n            self.cand_type = nn.Parameter(torch.zeros(d))\n            self.queries = nn.Parameter(torch.randn(a["k"], d) * 0.02)\n            layer = nn.TransformerEncoderLayer(d, a["nhead"], 4 * d, a["dropout"], batch_first=True,\n                                               norm_first=True, activation="gelu")\n            self.enc = nn.TransformerEncoder(layer, a["layers"], enable_nested_tensor=False)\n            self.norm = nn.LayerNorm(d)\n            self.unit = nn.Embedding(n_vocab + 1, d)          # row n_vocab = unknown previous unit\n            nn.init.normal_(self.unit.weight, std=0.02)\n            self.out_bias = nn.Parameter(torch.zeros(n_vocab))\n            self.chain = nn.ModuleList([nn.Sequential(nn.Linear(2 * d, d), nn.GELU(), nn.Linear(d, d))\n                                        for _ in range(a["k"] - 1)])\n            self.proj = nn.ModuleList([nn.Sequential(nn.Linear(d, d), nn.GELU(), nn.LayerNorm(d))\n                                       for _ in range(a["k"])])\n\n        def encode(self, b: dict):\n            x = (self.char(b["ctx_u"]) + self.script(b["ctx_s"]) + self.ctype(b["ctx_t"]) + self.pos(b["ctx_p"]))\n            pad = b["ctx_u"] == PAD\n            parts, masks = [x], [pad]\n            if "cand_u" in b:\n                cu, eu = b["cand_u"], b["ext_u"]\n                S = cu.shape[-1]\n                sp = torch.arange(S, device=cu.device)\n                cm = (cu != PAD).unsqueeze(-1).float()\n                ce = ((self.char(cu) + self.script(b["cand_s"]) + self.spos(sp)) * cm).sum(2) / cm.sum(2).clamp(min=1)\n                em = (eu != PAD).unsqueeze(-1).float()\n                ee = ((self.char(eu) + self.epos(sp)) * em).sum(2) / em.sum(2).clamp(min=1)\n                v = ce + ee + self.cfeat(b["cand_f"]) + self.lag(b["cand_lag"]) + self.rank(b["cand_rank"]) + self.cand_type\n                parts.append(v)\n                masks.append(~b["cand_mask"])\n            B = x.shape[0]\n            q = self.queries.unsqueeze(0).expand(B, -1, -1) + self.ctype.weight[5]\n            parts.append(q)\n            masks.append(torch.zeros(B, q.shape[1], dtype=torch.bool, device=x.device))\n            h = self.enc(torch.cat(parts, 1), src_key_padding_mask=torch.cat(masks, 1))\n            return self.norm(h[:, -q.shape[1]:])\n\n        def head_logits(self, hq, i: int, prev=None):\n            x = hq[:, i]\n            if i > 0:\n                x = x + self.chain[i - 1](torch.cat([x, self.unit(prev)], -1))\n            h = self.proj[i](x)\n            return h @ self.unit.weight[: self.n_vocab].t() + self.out_bias\n\n    return UniHead()\n\n\ndef range_mask(lo, hi, V: int):\n    torch = _torch()\n    ar = torch.arange(V, device=lo.device)\n    return (ar[None] >= lo[:, None]) & (ar[None] < hi[:, None])\n\n\ndef to_device(batch: dict, device) -> dict:\n    torch = _torch()\n    out = {}\n    for k, v in batch.items():\n        t = torch.from_numpy(np.ascontiguousarray(v))\n        out[k] = t.to(device, non_blocking=True)\n    if "cand_mask" in out:\n        out["cand_mask"] = out["cand_mask"].bool()\n    return out\n\n\ndef loss_fn(model, b: dict):\n    """Sum of per-head cross entropies (ignore -100); head 1 restricted to its prefix range."""\n    torch = _torch()\n    F = torch.nn.functional\n    V = model.n_vocab\n    hq = model.encode(b)\n    tgt = b["tgt"]\n    total, stats = 0.0, {}\n    prev = None\n    for i in range(tgt.shape[1]):\n        lg = model.head_logits(hq, i, prev).float()\n        if i == 0:\n            lg = lg.masked_fill(~range_mask(b["lo"], b["hi"], V), -1e4)\n        t = tgt[:, i]\n        m = t != -100\n        if m.any():\n            li = F.cross_entropy(lg, t, ignore_index=-100)\n            total = total + li\n            stats[f"loss{i + 1}"] = float(li.detach())\n            stats[f"acc{i + 1}"] = float((lg.argmax(-1)[m] == t[m]).float().mean())\n        prev = torch.where(m, t, torch.full_like(t, V))\n    return total, stats\n\n\ndef train_head(model, sources: list, steps: int, batch_size: int, lr: float, device, use_cands: bool,\n               val=None, log_every: int = 200, seed: int = 0, warmup: int = 200, log=print) -> dict:\n    """``sources``: list of (FeatureSet, position index array). Batches sample positions uniformly over the\n    union. ``val``: optional list of (FeatureSet, idx) evaluated at the end. AdamW, cosine decay, fp16\n    autocast on GPU."""\n    torch = _torch()\n    rng = np.random.default_rng(seed)\n    torch.manual_seed(seed)\n    model.to(device).train()\n    sizes = np.array([len(ix) for _, ix in sources], float)\n    if sizes.sum() == 0:\n        raise ValueError("no training positions")\n    probs = sizes / sizes.sum()\n    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.01)\n    sched = torch.optim.lr_scheduler.LambdaLR(\n        opt, lambda s: min(1.0, (s + 1) / warmup) * 0.5 * (1 + math.cos(math.pi * min(s, steps) / max(steps, 1))))\n    amp = device != "cpu" and str(device).startswith("cuda")\n    scaler = torch.amp.GradScaler("cuda", enabled=amp)\n    hist, t0, run = [], time.time(), {}\n    for step in range(steps):\n        counts = rng.multinomial(batch_size, probs)\n        parts = []\n        for (fs, ix), cnt in zip(sources, counts):\n            if cnt:\n                parts.append((fs, ix[rng.integers(0, len(ix), cnt)]))\n        loss = 0.0\n        opt.zero_grad(set_to_none=True)\n        stats_acc = {}\n        for fs, ii in parts:\n            b = to_device(make_batch(fs, ii, use_cands=use_cands), device)\n            with torch.autocast("cuda", dtype=torch.float16, enabled=amp):\n                l, st = loss_fn(model, b)\n            w = len(ii) / batch_size\n            scaler.scale(l * w).backward()\n            for k, v in st.items():\n                stats_acc[k] = stats_acc.get(k, 0.0) + v * w\n            loss += float(l.detach()) * w\n        scaler.unscale_(opt)\n        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)\n        scaler.step(opt)\n        scaler.update()\n        sched.step()\n        for k, v in stats_acc.items():\n            run[k] = 0.98 * run.get(k, v) + 0.02 * v\n        if (step + 1) % log_every == 0 or step == steps - 1:\n            rec = dict(step=step + 1, loss=loss, sec=round(time.time() - t0, 1), **{k: round(v, 4) for k, v in run.items()})\n            hist.append(rec)\n            log(f"  step {step + 1}/{steps} loss {loss:.3f} " + " ".join(f"{k}={v:.3f}" for k, v in run.items())\n                + f" ({rec[\'sec\']:.0f}s)")\n    out = dict(history=hist, train_sec=time.time() - t0)\n    if val:\n        out["val"] = evaluate_heads(model, val, device, use_cands)\n        log("  val: " + json.dumps(out["val"]))\n    return out\n\n\ndef evaluate_heads(model, sets: list, device, use_cands: bool, batch_size: int = 1024) -> dict:\n    """Teacher-forced per-head accuracy and loss on (FeatureSet, idx) pairs."""\n    torch = _torch()\n    model.eval()\n    agg, n = {}, {}\n    with torch.no_grad():\n        for fs, ix in sets:\n            for s in range(0, len(ix), batch_size):\n                b = to_device(make_batch(fs, ix[s:s + batch_size], use_cands=use_cands), device)\n                with torch.autocast("cuda", dtype=torch.float16, enabled=str(device).startswith("cuda")):\n                    _, st = loss_fn(model, b)\n                w = len(ix[s:s + batch_size])\n                for k, v in st.items():\n                    agg[k] = agg.get(k, 0.0) + v * w\n                    n[k] = n.get(k, 0) + w\n    model.train()\n    return {k: round(agg[k] / n[k], 4) for k in agg}\n\n\ndef predict(model, fs: FeatureSet, device, use_cands: bool, idx=None, batch_size: int = 1024):\n    """Greedy chained prediction at every position: units ``w`` (N, k; -1 = none) and their\n    probabilities ``pw`` (head 1 renormalised over its prefix range)."""\n    torch = _torch()\n    model.eval()\n    idx = np.arange(fs.n_pos) if idx is None else np.asarray(idx)\n    V, k = model.n_vocab, model.arch["k"]\n    W = np.full((len(idx), k), -1, np.int32)\n    P = np.zeros((len(idx), k), np.float32)\n    eos = V - 1\n    with torch.no_grad():\n        for s in range(0, len(idx), batch_size):\n            ii = idx[s:s + batch_size]\n            b = to_device(make_batch(fs, ii, use_cands=use_cands), device)\n            with torch.autocast("cuda", dtype=torch.float16, enabled=str(device).startswith("cuda")):\n                hq = model.encode(b)\n                prev, alive = None, (b["hi"] > b["lo"])\n                for i in range(k):\n                    lg = model.head_logits(hq, i, prev).float()\n                    if i == 0:\n                        lg = lg.masked_fill(~range_mask(b["lo"], b["hi"], V), -1e4)\n                    pr = torch.softmax(lg, -1)\n                    p, w = pr.max(-1)\n                    W[s:s + len(ii), i] = torch.where(alive, w, torch.full_like(w, -1)).cpu().numpy()\n                    P[s:s + len(ii), i] = torch.where(alive, p, torch.zeros_like(p)).cpu().numpy()\n                    alive = alive & (w != eos)\n                    prev = w\n    return W, P\n\n\ndef time_head(model, fs: FeatureSet, device, use_cands: bool, n: int = 200, warmup: int = 20) -> dict:\n    """Batch-1 latency of one drafting call: GPU forward + chained argmax (inputs already on the device),\n    and separately the CPU time to build the inputs. Median over ``n`` positions."""\n    torch = _torch()\n    model.eval()\n    rng = np.random.default_rng(0)\n    ix = rng.integers(0, fs.n_pos, n + warmup)\n    gpu, cpu = [], []\n    V, k = model.n_vocab, model.arch["k"]\n    sync = (lambda: torch.cuda.synchronize()) if str(device).startswith("cuda") else (lambda: None)\n    with torch.no_grad():\n        for j, i in enumerate(ix):\n            t0 = time.perf_counter()\n            nb = make_batch(fs, [i], use_cands=use_cands)\n            t1 = time.perf_counter()\n            b = to_device(nb, device)\n            sync()\n            t2 = time.perf_counter()\n            with torch.autocast("cuda", dtype=torch.float16, enabled=str(device).startswith("cuda")):\n                hq = model.encode(b)\n                prev = None\n                for h in range(k):\n                    lg = model.head_logits(hq, h, prev)\n                    if h == 0:\n                        lg = lg.masked_fill(~range_mask(b["lo"], b["hi"], V), -1e4)\n                    prev = lg.argmax(-1)\n            _ = prev.cpu()\n            sync()\n            t3 = time.perf_counter()\n            if j >= warmup:\n                cpu.append(t1 - t0)\n                gpu.append(t3 - t2)\n    return dict(t_head_gpu=float(np.median(gpu)), t_inputs_cpu=float(np.median(cpu)), n=n, device=str(device),\n                params=int(sum(p.numel() for p in model.parameters())))\n\n\ndef save_head(model, path, extra: dict | None = None) -> None:\n    torch = _torch()\n    torch.save(dict(state=model.state_dict(), arch=model.arch, n_vocab=model.n_vocab, extra=extra or {}), path)\n\n\ndef load_head(path, device="cpu"):\n    torch = _torch()\n    d = torch.load(path, map_location=device, weights_only=False)\n    m = build_model(d["n_vocab"], d["arch"])\n    m.load_state_dict(d["state"])\n    return m.to(device), d.get("extra", {})\n',
    'uni_sim': '"""Exact trace simulation of the universal head on any target (W1), plus training-free baselines.\n\nFor greedy decoding the head\'s draft at committed position p depends only on out_ids[:p] and q_t for\nt < p, which are the same however p was reached, so drafts can be precomputed for every position and\nthe round-by-round loop below is exact up to the cost model (same argument as Phase 1, PLAN §3).\n\nRound at p: draft g tokens (the head\'s units converted to the *target\'s* tokens, see ``convert``),\naccept the leading matches a, emit min(a+1, n-p) tokens, cost = c_head + verify(g).\n"""\nfrom __future__ import annotations\n\nfrom dataclasses import dataclass\n\nimport numpy as np\n\nfrom .stats import bootstrap_ci, paired_bootstrap\nfrom .uni_features import TokTable, position_targets, seq_text, utf8_offsets\nfrom .unichar import UnitVocab, segment_units\n\n\n# --------------------------------------------------------------------------- #\n# cost model\n# --------------------------------------------------------------------------- #\n\nclass Verify:\n    """verify(g) in target-step units from a Phase-1 style timing dict; linear extrapolation past the\n    largest measured g (np.interp would flatten it and flatter long drafts)."""\n\n    def __init__(self, timing: dict):\n        tt = timing["t_target"]\n        gs = sorted(int(g) for g in timing["t_target_verify"])\n        vs = [max(1.0, timing["t_target_verify"][str(g)] / tt) for g in gs]\n        self.gs, self.vs = gs, vs\n        tail = [(g, v) for g, v in zip(gs, vs) if g >= max(gs) - 4]\n        self.slope = max(0.0, float(np.polyfit([g for g, _ in tail], [v for _, v in tail], 1)[0])) if len(tail) > 1 else 0.0\n\n    def __call__(self, g: int) -> float:\n        if g <= 0:\n            return 1.0\n        if g <= self.gs[-1]:\n            return float(np.interp(g, self.gs, self.vs))\n        return self.vs[-1] + self.slope * (g - self.gs[-1])\n\n    def as_dict(self):\n        return {str(g): round(self(g), 4) for g in range(0, 17)}\n\n\n# --------------------------------------------------------------------------- #\n# sequences of one target\n# --------------------------------------------------------------------------- #\n\n@dataclass\nclass SimSeq:\n    id: int\n    out_ids: np.ndarray\n    n: int\n    b_tok: np.ndarray\n    out_bytes: bytes\n    char_start: np.ndarray     # bool per byte offset (0..len): is a char boundary\n    ps_byte: np.ndarray        # partial-unit start byte at each position p\n    pend: np.ndarray\n    eos_id: int | None\n    prompt_ids: list | None\n    topk_ids: np.ndarray | None\n\n\ndef build_simseqs(records: list[dict], table: TokTable, vocab: UnitVocab) -> list[SimSeq]:\n    out = []\n    for r in records:\n        ids = np.asarray(r["out_ids"], np.int64)\n        st = seq_text(table, ids)\n        ps_b, _, _, _ = position_targets(st, vocab, bool(r.get("eos_end", True)))\n        cs = np.zeros(len(st.out_bytes) + 1, bool)\n        cs[st.coff] = True\n        eos = int(ids[-1]) if len(ids) and table.is_eos[int(ids[-1])] else (table.eos_ids[0] if table.eos_ids else None)\n        out.append(SimSeq(int(r["id"]), ids, len(ids), st.b_tok, st.out_bytes, cs, ps_b, st.pend, eos,\n                          r.get("prompt_ids"), np.asarray(r["topk_ids"]) if r.get("topk_ids") is not None else None))\n    return out\n\n\ndef draft_levels(vocab: UnitVocab, w_row, partial_len: int) -> list[tuple[bytes, bool]]:\n    """Draft bytes for levels 1..k (level j = first j units); stops at <eos> (flag) or a missing unit."""\n    levels, acc = [], b""\n    for i, w in enumerate(w_row):\n        w = int(w)\n        if w < 0:\n            break\n        if w == vocab.eos_id:\n            levels.append((acc, True))\n            break\n        ub = vocab.bytes[w]\n        acc = acc + (ub[partial_len:] if i == 0 else ub)\n        levels.append((acc, False))\n    return levels\n\n\ndef _anchor(seq: SimSeq, p: int) -> int:\n    """Largest token index q <= p that starts at or before the partial unit, on a char boundary."""\n    q = p\n    while q > 0 and (seq.b_tok[q] > seq.ps_byte[p] or not seq.char_start[seq.b_tok[q]]):\n        q -= 1\n    return q\n\n\ndef _level_prefixes(toks: list, tok_bytes, levels: list, eos_id) -> list:\n    """Level j = the longest prefix of ``toks`` (tokenization of the longest draft) whose bytes end\n    inside level j\'s text. A shorter level therefore never ends with a token that the following\n    predicted unit would have merged into (Llama-3 glues a word-final vowel sign to the punctuation\n    after it: "ी।" is one token). Only the longest level can end on such a token."""\n    cum, c = [], 0\n    for t in toks:\n        c += len(tok_bytes[t]) if t < len(tok_bytes) else 0\n        cum.append(c)\n    out = []\n    for db, eos in levels:\n        B = len(db)\n        n = 0\n        while n < len(cum) and cum[n] <= B:\n            n += 1\n        d = list(toks[:n])\n        if eos and eos_id is not None:\n            d.append(eos_id)\n        out.append(d)\n    return out\n\n\ndef convert_all(seqs: list[SimSeq], W: np.ndarray, vocab: UnitVocab, encode, tok_bytes, gmax: int = 16):\n    """Target-token drafts for every position and level. ``tok_bytes`` = ``TokTable.bytes``.\n\n    Per position: retokenise (committed text from the start of the current word) + (all predicted\n    units) once with the target tokenizer; if that reproduces the committed tokens, the remaining\n    tokens are the draft (``anchored``); otherwise, when the committed text ends on a char boundary,\n    the draft text is tokenised on its own (``fallback``, non-canonical); else no draft (``failed``).\n    Returns (drafts, stats) with drafts[s][p] = list over levels 0..L of token-id lists (level 0 = [])."""\n    jobs, meta = [], []\n    off = 0\n    for s, seq in enumerate(seqs):\n        for p in range(seq.n):\n            plen = int(seq.b_tok[p] - seq.ps_byte[p])\n            levels = draft_levels(vocab, W[off + p], plen)\n            if levels and levels[-1][0]:\n                q = _anchor(seq, p)\n                try:\n                    txt = (seq.out_bytes[seq.b_tok[q]:seq.b_tok[p]] + levels[-1][0]).decode("utf-8")\n                except UnicodeDecodeError:\n                    txt = None\n                meta.append((s, p, q, levels, txt))\n                jobs.append(txt if txt is not None else "")\n            elif levels:                                   # only <eos> / empty completions\n                drafts_e = [([seq.eos_id] if eos and seq.eos_id is not None else []) for _, eos in levels]\n                meta.append((s, p, None, levels, drafts_e))\n        off += seq.n\n    enc = iter(encode(jobs) if jobs else [])\n    drafts = [[[[]] for _ in range(seq.n)] for seq in seqs]\n    stats = dict(positions=0, levels=0, anchored=0, fallback=0, failed=0, empty=0)\n    fb_jobs, fb_meta = [], []\n    for s, p, q, levels, txt in meta:\n        seq = seqs[s]\n        stats["positions"] += 1\n        stats["levels"] += len(levels)\n        if q is None:\n            drafts[s][p] = [[]] + [d[:gmax] for d in txt]\n            stats["empty"] += 1\n            continue\n        ids = next(enc)\n        if txt is not None and list(ids[:p - q]) == seq.out_ids[q:p].tolist():\n            drafts[s][p] = [[]] + [d[:gmax] for d in _level_prefixes(list(ids[p - q:]), tok_bytes, levels, seq.eos_id)]\n            stats["anchored"] += 1\n            continue\n        stats["failed"] += 1\n        drafts[s][p] = [[] for _ in range(len(levels) + 1)]\n        if seq.pend[p] == 0:\n            try:\n                fb_jobs.append(levels[-1][0].decode("utf-8"))\n                fb_meta.append((s, p, levels))\n            except UnicodeDecodeError:\n                pass\n    for (s, p, levels), ids in zip(fb_meta, encode(fb_jobs) if fb_jobs else []):\n        seq = seqs[s]\n        drafts[s][p] = [[]] + [d[:gmax] for d in _level_prefixes(list(ids), tok_bytes, levels, seq.eos_id)]\n        stats["fallback"] += 1\n        stats["failed"] -= 1\n    return drafts, stats\n\n\n# --------------------------------------------------------------------------- #\n# simulation\n# --------------------------------------------------------------------------- #\n\ndef choose_level(probs, n_levels: int, tau: float) -> int:\n    """Largest level j <= n_levels whose cumulative unit probability is >= tau (0 if none)."""\n    c, best = 1.0, 0\n    for j in range(n_levels):\n        c *= float(probs[j])\n        if c >= tau:\n            best = j + 1\n        else:\n            break\n    return best\n\n\ndef simulate(seqs: list[SimSeq], drafts, P: np.ndarray, tau: float, c_head: float, verify, gmax: int = 16,\n             cap_to_end: bool = True, oracle_level: bool = False) -> dict:\n    ns = len(seqs)\n    em, co = np.zeros(ns), np.zeros(ns)\n    R = dict(rounds=0, drafted=0, accepted=0)\n    off = 0\n    for s, seq in enumerate(seqs):\n        p = 0\n        y = seq.out_ids\n        while p < seq.n:\n            lv_all = drafts[s][p]\n            if oracle_level:\n                best = (-1, 0)\n                for lv in range(len(lv_all)):\n                    d = lv_all[lv][: (seq.n - p) if cap_to_end else gmax]\n                    a = _accept(d, y, p)\n                    if a > best[0]:\n                        best = (a, lv)\n                lv = best[1]\n            else:\n                lv = choose_level(P[off + p], len(lv_all) - 1, tau)\n            d = lv_all[lv][:gmax]\n            if cap_to_end:\n                d = d[: seq.n - p]\n            a = _accept(d, y, p)\n            gained = min(a + 1, seq.n - p)\n            em[s] += gained\n            co[s] += c_head + verify(len(d))\n            R["rounds"] += 1; R["drafted"] += len(d); R["accepted"] += a\n            p += gained\n        off += seq.n\n    R.update(sent_emitted=em, sent_cost=co, speedup=float(em.sum() / co.sum()),\n             tokens_per_round=float(em.sum() / max(R["rounds"], 1)),\n             mean_draft_len=R["drafted"] / max(R["rounds"], 1),\n             acceptance=R["accepted"] / max(R["drafted"], 1))\n    return R\n\n\ndef _accept(d, y, p) -> int:\n    a = 0\n    for i, t in enumerate(d):\n        if p + i < len(y) and int(y[p + i]) == int(t):\n            a += 1\n        else:\n            break\n    return a\n\n\ndef tune_tau(seqs, drafts, P, taus, c_head, verify, gmax=16) -> tuple[float, list]:\n    rows = [(float(simulate(seqs, drafts, P, t, c_head, verify, gmax)["speedup"]), float(t)) for t in taus]\n    best = max(rows)[1]\n    return best, [dict(tau=t, speedup=s) for s, t in rows]\n\n\ndef summary_row(name: str, R: dict, n_boot: int = 1000, seed: int = 0, ref: dict | None = None, **extra) -> dict:\n    ci = bootstrap_ci(R["sent_emitted"], R["sent_cost"], n_boot, seed)\n    row = dict(method=name, speedup=ci["point"], ci_lo=ci["lo"], ci_hi=ci["hi"],\n               tokens_per_round=R["tokens_per_round"], mean_draft_len=R["mean_draft_len"],\n               acceptance=R["acceptance"], rounds=R["rounds"], **extra)\n    if ref is not None:\n        pb = paired_bootstrap(R["sent_emitted"], R["sent_cost"], ref["sent_emitted"], ref["sent_cost"], n_boot, seed)\n        row.update(diff_vs_ref=pb["diff"], diff_lo=pb["lo"], diff_hi=pb["hi"], p_better=pb["p_better"])\n    return row\n\n\n# --------------------------------------------------------------------------- #\n# training-free baselines (token level, draft cost ~0)\n# --------------------------------------------------------------------------- #\n\ndef _lookup(ctx: list, key: tuple, max_len: int) -> list:\n    """Continuation after the most recent earlier occurrence of ``key`` in ``ctx`` (excluding the end)."""\n    k = len(key)\n    for i in range(len(ctx) - k - 1, -1, -1):\n        if tuple(ctx[i:i + k]) == key:\n            return ctx[i + k:i + k + max_len]\n    return []\n\n\nclass _NgramIndex:\n    """Latest start of every n-gram of a growing context that has at least one following token\n    (same answer as ``_lookup``, in O(1) per query)."""\n\n    def __init__(self, ctx: list, ns):\n        self.ctx, self.ns = list(ctx), tuple(ns)\n        self.idx = {n: {} for n in self.ns}\n        for n in self.ns:\n            for i in range(len(self.ctx) - n):\n                self.idx[n][tuple(self.ctx[i:i + n])] = i\n\n    def lookup(self, key: tuple, max_len: int) -> list:\n        i = self.idx[len(key)].get(tuple(key))\n        return [] if i is None else self.ctx[i + len(key):i + len(key) + max_len]\n\n    def append(self, t) -> None:\n        for n in self.ns:\n            if len(self.ctx) >= n:\n                self.idx[n][tuple(self.ctx[-n:])] = len(self.ctx) - n\n        self.ctx.append(t)\n\n\ndef pld_drafts(seqs: list[SimSeq], max_len: int = 8, ngram: tuple = (3, 2, 1)):\n    """Prompt-lookup decoding: match the last n committed tokens in prompt + committed output."""\n    out = []\n    for seq in seqs:\n        ix = _NgramIndex(list(seq.prompt_ids or []), ngram)\n        rows = []\n        for t in seq.out_ids.tolist():\n            d = []\n            for n in ngram:\n                if len(ix.ctx) > n:\n                    d = ix.lookup(tuple(ix.ctx[-n:]), max_len)\n                    if d:\n                        break\n            rows.append([[], d])\n            ix.append(t)\n        out.append(rows)\n    return out\n\n\ndef logitspec_drafts(seqs: list[SimSeq], max_len: int = 8):\n    """LogitSpec-style chain: guess y[p] as the best candidate of q_{p-1} other than the committed token,\n    then extend by looking up (y[p-1], guess) in prompt + committed output."""\n    out = []\n    for seq in seqs:\n        y = seq.out_ids.tolist()\n        ix = _NgramIndex(list(seq.prompt_ids or []), (2,))\n        if y:\n            ix.append(y[0])\n        rows = [[[], []]]\n        for p in range(1, seq.n):\n            cands = [int(v) for v in seq.topk_ids[p - 1] if int(v) != y[p - 1]]\n            if cands:\n                g = cands[0]\n                rows.append([[], [g] + ix.lookup((y[p - 1], g), max_len - 1)])\n            else:\n                rows.append([[], []])\n            ix.append(y[p])\n        out.append(rows)\n    return out\n\n\ndef next_next_coverage(seqs: list[SimSeq], ks=(1, 2, 5, 10, 20)) -> dict:\n    """Fraction of positions p >= 1 where y[p] is among the top-k of q_{p-1} (LogitSpec\'s observation)."""\n    hits = {k: 0 for k in ks}\n    n = 0\n    for seq in seqs:\n        y = seq.out_ids\n        for p in range(1, seq.n):\n            row = seq.topk_ids[p - 1]\n            for k in ks:\n                hits[k] += int(y[p]) in set(int(v) for v in row[:k])\n            n += 1\n    return {f"top{k}": hits[k] / max(n, 1) for k in ks} | dict(positions=n)\n\n\ndef ones_like_probs(drafts) -> np.ndarray:\n    n = sum(len(r) for r in drafts)\n    return np.ones((n, 1), np.float32)\n\n\n# --------------------------------------------------------------------------- #\n# cross-LLM agreement on word units (teacher forced)\n# --------------------------------------------------------------------------- #\n\ndef unit_agreement(text: str, ids: list, argmax: list, table: TokTable) -> dict:\n    """Model B teacher-forced on ``text`` (tokenised by B as ``ids``; ``argmax[t]`` = B\'s greedy token\n    at t). Counts per unit: first token correct (unit start), later tokens correct (continuation),\n    whole unit correct, whole unit correct given its start was correct."""\n    bl = table.seq_bytes(ids)\n    starts = np.zeros(len(bl) + 1, np.int64)\n    starts[1:] = np.cumsum([len(b) for b in bl])\n    _, coff = utf8_offsets(text.encode("utf-8"))\n    spans = segment_units(text)\n    ubytes = [(int(coff[a]), int(coff[b])) for a, b in spans]\n    ok = [int(a) == int(t) for a, t in zip(argmax, ids)]\n    tok_unit = np.searchsorted([u[0] for u in ubytes], starts[:-1], side="right") - 1\n    res = dict(units=0, start_ok=0, cont_tokens=0, cont_ok=0, unit_ok=0, unit_ok_given_start=0, start_ok_units=0)\n    for u in range(len(ubytes)):\n        toks = np.flatnonzero(tok_unit == u)\n        if not len(toks):\n            continue          # unit inside a token that started earlier (merged across units)\n        res["units"] += 1\n        s_ok = ok[toks[0]]\n        all_ok = all(ok[t] for t in toks)\n        res["start_ok"] += s_ok\n        res["cont_tokens"] += len(toks) - 1\n        res["cont_ok"] += sum(ok[t] for t in toks[1:])\n        res["unit_ok"] += all_ok\n        if s_ok:\n            res["start_ok_units"] += 1\n            res["unit_ok_given_start"] += all_ok\n    return res\n\n\ndef agreement_table(counts: dict) -> dict:\n    c = counts\n    return dict(units=c["units"], p_start=c["start_ok"] / max(c["units"], 1),\n                p_cont_token=c["cont_ok"] / max(c["cont_tokens"], 1),\n                p_unit=c["unit_ok"] / max(c["units"], 1),\n                p_unit_given_start=c["unit_ok_given_start"] / max(c["start_ok_units"], 1))\n',
    'collect': '"""Data collection for W1: several LLMs\' greedy Hindi outputs with their top-K distributions (GPU).\n\nPer (model, split) it writes ``{split}.jsonl`` (id, src, prompt_text, prompt_ids, out_ids, out_text,\neos_end) and ``{split}_topk.npz`` (flat top-K ids / log-probs with offsets), both resumable. The top-K\ncomes from a separate teacher-forced pass over prompt + output (unpadded semantics via attention mask\nand explicit position ids), like the Phase-1 traces, so small-model data and the Phase-1 8B traces are\nbuilt the same way.\n"""\nfrom __future__ import annotations\n\nimport gc\nimport json\nimport os\nimport time\nfrom pathlib import Path\n\nimport numpy as np\n\nfrom .gpu_traces import PROMPT_TEMPLATE, from_pretrained, read_jsonl\n\nBASE_DEMO = ("English: The weather is very pleasant today.\\nHindi: आज मौसम बहुत सुहावना है।\\n\\n")\n\nDEFAULT_MODELS = [\n    dict(key="qwen2.5-1.5b", names=["Qwen/Qwen2.5-1.5B-Instruct"], dtype="fp16", family="qwen"),\n    dict(key="gemma3-1b", names=["google/gemma-3-1b-it", "unsloth/gemma-3-1b-it"], dtype="fp32", family="gemma"),\n    dict(key="llama3.2-1b", names=["meta-llama/Llama-3.2-1B-Instruct", "unsloth/Llama-3.2-1B-Instruct"],\n         dtype="fp16", family="llama3"),\n    dict(key="sarvam1", names=["sarvamai/sarvam-1"], dtype="fp16", family="sarvam", prompt_mode="base"),\n]\nOPTIONAL_MODELS = [\n    dict(key="qwen3-1.7b", names=["Qwen/Qwen3-1.7B"], dtype="fp16", family="qwen",\n         chat_kwargs=dict(enable_thinking=False)),\n]\n\n\ndef _torch():\n    import torch\n    return torch\n\n\ndef head_prompt_text(src: str) -> str:\n    """The prompt text the universal head reads: the same for every model (model-independent)."""\n    return PROMPT_TEMPLATE.format(s=src)\n\n\n# --------------------------------------------------------------------------- #\n# sources\n# --------------------------------------------------------------------------- #\n\ndef load_flores_english(split: str, n: int | None = None) -> list[tuple[int, str]]:\n    """(id, English sentence) of FLORES+ ``split``. Falls back to the ungated mteb/flores parquet (whose\n    values carry one extra leading quote, removed here); its Hindi column is corrupted and is not used."""\n    from datasets import load_dataset\n    try:\n        en = load_dataset("openlanguagedata/flores_plus", "eng_Latn", split=split)\n        rows = [(int(r["id"]), r["text"]) for r in en]\n        src = "openlanguagedata/flores_plus"\n    except Exception as e:  # noqa: BLE001\n        print(f"flores_plus unavailable ({type(e).__name__}: {e}); using mteb/flores English column")\n        ds = load_dataset("mteb/flores", split=split)\n        rows = []\n        for i, r in enumerate(ds):\n            t = r["eng_Latn"]\n            if t.startswith(\'"\') and t.count(\'"\') % 2 == 1:\n                t = t[1:]\n            rows.append((i, t))\n        src = "mteb/flores"\n    print(f"FLORES {split}: {len(rows)} sentences from {src}")\n    return rows[:n] if n else rows\n\n\ndef load_iitb(n_en: int, n_hi: int, seed: int = 0, min_words: int = 8, max_words: int = 40):\n    """Extra English sources (for generation) and Hindi sentences (unit vocabulary + text-only\n    pretraining) from the IITB English-Hindi corpus, disjoint sentence pairs, length-filtered."""\n    from datasets import load_dataset\n    ds = load_dataset("cfilt/iitb-english-hindi", split="train")\n    tr = ds["translation"]\n    rng = np.random.default_rng(seed)\n    order = rng.permutation(len(tr))\n    en, hi = [], []\n    for i in order:\n        e, h = tr[i]["en"], tr[i]["hi"]\n        ne, nh = len(e.split()), len(h.split())\n        if not (min_words <= ne <= max_words and min_words <= nh <= max_words * 2):\n            continue\n        if len(en) < n_en and sum(ch.isalpha() for ch in e) > 0.6 * len(e):\n            en.append((1_000_000 + int(i), e))\n        elif len(hi) < n_hi and sum("ऀ" <= ch <= "ॿ" for ch in h) > 0.5 * len(h.replace(" ", "")):\n            hi.append(h)\n        if len(en) >= n_en and len(hi) >= n_hi:\n            break\n    print(f"IITB: {len(en)} English sources, {len(hi)} Hindi sentences")\n    return en, hi\n\n\n# --------------------------------------------------------------------------- #\n# models\n# --------------------------------------------------------------------------- #\n\ndef load_model(spec: dict, device: str = "cuda:0", max_memory=None):\n    """Load the first loadable name of ``spec[\'names\']``. Returns (model, tok, name) or raises."""\n    torch = _torch()\n    from transformers import AutoTokenizer\n    dt = dict(fp16=torch.float16, fp32=torch.float32, bf16=torch.bfloat16)[spec.get("dtype", "fp16")]\n    if device == "cpu":\n        dt = torch.float32\n    last = None\n    for name in spec["names"]:\n        try:\n            tok = AutoTokenizer.from_pretrained(name)\n            if spec.get("device_map") == "auto" and device != "cpu":\n                m = from_pretrained(name, dt, device_map="auto", max_memory=max_memory, low_cpu_mem_usage=True)\n            else:\n                m = from_pretrained(name, dt).to(device)\n            if tok.pad_token is None:\n                tok.pad_token = tok.eos_token\n            print(f"loaded {name} ({spec.get(\'dtype\', \'fp16\')} on {device})")\n            return m, tok, name\n        except Exception as e:  # noqa: BLE001\n            last = e\n            print(f"  could not load {name}: {type(e).__name__}: {str(e)[:200]}")\n    raise RuntimeError(f"no loadable model for {spec[\'key\']}: {last}")\n\n\ndef model_eos_ids(model, tok, spec: dict) -> list[int]:\n    ids = set()\n    eos = model.generation_config.eos_token_id\n    for e in (eos if isinstance(eos, (list, tuple)) else [eos]):\n        if e is not None:\n            ids.add(int(e))\n    if tok.eos_token_id is not None:\n        ids.add(int(tok.eos_token_id))\n    for t in ("<|eot_id|>", "<end_of_turn>", "<|im_end|>", "<|endoftext|>"):\n        i = tok.convert_tokens_to_ids(t)\n        if isinstance(i, int) and i != tok.unk_token_id and i >= 0:\n            ids.add(i)\n    return sorted(ids)\n\n\ndef build_model_prompt(tok, spec: dict, src: str) -> str:\n    if spec.get("prompt_mode") != "base" and getattr(tok, "chat_template", None):\n        msgs = [{"role": "user", "content": PROMPT_TEMPLATE.format(s=src)}]\n        try:\n            return tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True,\n                                           **spec.get("chat_kwargs", {}))\n        except Exception as e:  # noqa: BLE001\n            print(f"chat template failed ({e}); using base prompt")\n    bos = tok.bos_token or ""\n    return f"{bos}Translate the following English sentence into Hindi.\\n\\n{BASE_DEMO}English: {src}\\nHindi:"\n\n\ndef _base_cut(ids: list, tok_bytes, eos_id: int) -> tuple[list, bool]:\n    """Base-model outputs: stop at the first newline after some text; the newline becomes the model\'s eos."""\n    acc = b""\n    for j, t in enumerate(ids):\n        b = tok_bytes[j]\n        if b"\\n" in b and acc.strip():\n            return ids[:j] + [eos_id], True\n        acc += b\n    return ids, False\n\n\n# --------------------------------------------------------------------------- #\n# generation + teacher-forced top-K\n# --------------------------------------------------------------------------- #\n\ndef generate(model, tok, spec: dict, rows: list, out_jsonl, batch: int, max_new: int, device: str,\n             eos_ids: list[int], log=print) -> list[dict]:\n    """Greedy batched generation (left padding), appended to ``out_jsonl`` after every batch, resumable."""\n    torch = _torch()\n    from .boundaries import token_to_bytes\n    out_jsonl = Path(out_jsonl)\n    done = {r["id"]: r for r in read_jsonl(out_jsonl)}\n    todo = [r for r in rows if r[0] not in done]\n    log(f"  generate: {len(done)} done, {len(todo)} to go")\n    tok.padding_side = "left"\n    order = sorted(range(len(todo)), key=lambda i: len(todo[i][1]))\n    base = spec.get("prompt_mode") == "base" or not getattr(tok, "chat_template", None)\n    eos_set = set(eos_ids)\n    t0 = time.time()\n    with open(out_jsonl, "a", encoding="utf-8") as f:\n        for bi in range(0, len(order), batch):\n            chunk = [todo[i] for i in order[bi:bi + batch]]\n            prompts = [build_model_prompt(tok, spec, s) for _, s in chunk]\n            enc = tok(prompts, return_tensors="pt", padding=True, add_special_tokens=False).to(device)\n            with torch.no_grad():\n                out = model.generate(**enc, do_sample=False, temperature=None, top_p=None, top_k=None,\n                                     repetition_penalty=1.0, max_new_tokens=max_new,\n                                     eos_token_id=sorted(eos_set), pad_token_id=tok.pad_token_id)\n            gen = out[:, enc.input_ids.shape[1]:].tolist()\n            for j, g in enumerate(gen):\n                cut = next((p + 1 for p, t in enumerate(g) if t in eos_set), len(g))\n                g = g[:cut]\n                eos_end = bool(g) and g[-1] in eos_set\n                if base:\n                    core = g[:-1] if eos_end else g\n                    tb = [token_to_bytes(t) for t in tok.convert_ids_to_tokens(core)]\n                    cut_ids, was_cut = _base_cut(core, tb, sorted(eos_set)[0])\n                    if was_cut:\n                        g, eos_end = cut_ids, True\n                pid = enc.input_ids[j][enc.attention_mask[j].bool()].tolist()\n                rid, src = chunk[j]\n                rec = dict(id=int(rid), src=src, prompt_text=head_prompt_text(src), prompt_ids=pid, out_ids=g,\n                           out_text=tok.decode(g, skip_special_tokens=True).strip(), eos_end=eos_end)\n                done[rid] = rec\n                f.write(json.dumps(rec, ensure_ascii=False) + "\\n")\n            f.flush()\n            if bi == 0:\n                el = time.time() - t0\n                log(f"  first batch {el:.1f}s -> ESTIMATE ~{el * len(order) / max(len(chunk), 1) / 60:.1f} min for this split")\n    recs = [done[r[0]] for r in rows if r[0] in done]\n    nhit = sum(len(r["out_ids"]) >= max_new for r in recs)\n    log(f"  {len(recs)} outputs; {nhit} hit MAX_NEW; mean len {np.mean([len(r[\'out_ids\']) for r in recs]):.1f} tokens")\n    return recs\n\n\ndef _logits_budget_rows(V: int, L: int, bytes_per: int, budget: float = 2.5e9) -> int:\n    """Rows per teacher-forced batch so that the kept logits (model dtype + a float32 copy) fit."""\n    return max(1, int(budget // max(V * L * (bytes_per + 4), 1)))\n\n\ndef teacher_forced_topk(model, recs: list[dict], device: str, k: int = 20, log=print,\n                        ids_key: str = "out_ids", argmax_only: bool = False):\n    """Top-k log-probs of q_t for every output position t of every record (prompt_ids + out ids),\n    batched with left padding, explicit position ids and only the output rows of the logits.\n    Returns list of (ids [n,k] int32, logp [n,k] float16) or list of argmax arrays."""\n    torch = _torch()\n    V = model.get_output_embeddings().weight.shape[0]\n    bpe = 4 if next(model.parameters()).dtype == torch.float32 else 2\n    order = sorted(range(len(recs)), key=lambda i: len(recs[i]["prompt_ids"]) + len(recs[i][ids_key]))\n    res = [None] * len(recs)\n    i = 0\n    while i < len(order):\n        L_peek = max(len(recs[c][ids_key]) for c in order[i:i + 32]) + 1\n        rows = _logits_budget_rows(V, L_peek, bpe)\n        chunk = order[i:i + min(rows, 32)]\n        L_out = max(len(recs[c][ids_key]) for c in chunk)\n        seqs = [list(recs[c]["prompt_ids"]) + list(recs[c][ids_key]) for c in chunk]\n        Lmax = max(len(s) for s in seqs)\n        x = torch.full((len(seqs), Lmax), 0, dtype=torch.long)\n        am = torch.zeros((len(seqs), Lmax), dtype=torch.long)\n        for j, s in enumerate(seqs):\n            x[j, Lmax - len(s):] = torch.tensor(s)\n            am[j, Lmax - len(s):] = 1\n        pos = (am.cumsum(1) - 1).clamp(min=0)\n        x, am, pos = x.to(device), am.to(device), pos.to(device)\n        keep = L_out + 1\n        with torch.no_grad():\n            try:\n                lg = model(input_ids=x, attention_mask=am, position_ids=pos, logits_to_keep=keep).logits\n            except TypeError:\n                lg = model(input_ids=x, attention_mask=am, position_ids=pos).logits[:, -keep:]\n            lg = lg.float()\n            for j, c in enumerate(chunk):\n                n = len(recs[c][ids_key])\n                rowlg = lg[j, keep - 1 - n: keep - 1]           # q_t for t = 0..n-1\n                if argmax_only:\n                    res[c] = rowlg.argmax(-1).cpu().numpy().astype(np.int32)\n                else:\n                    lp = torch.log_softmax(rowlg, -1)\n                    tk = lp.topk(k, -1)\n                    res[c] = (tk.indices.cpu().numpy().astype(np.int32), tk.values.cpu().numpy().astype(np.float16))\n        del lg\n        i += len(chunk)\n    return res\n\n\ndef save_topk(path, recs: list[dict], tk: list) -> None:\n    offs = np.zeros(len(recs) + 1, np.int64)\n    offs[1:] = np.cumsum([len(r["out_ids"]) for r in recs])\n    ids = np.concatenate([a for a, _ in tk]) if tk else np.zeros((0, 20), np.int32)\n    lp = np.concatenate([b for _, b in tk]) if tk else np.zeros((0, 20), np.float16)\n    tmp = str(path) + ".tmp.npz"\n    np.savez(tmp, rec_ids=np.array([r["id"] for r in recs], np.int64), offsets=offs, topk_ids=ids, topk_logp=lp)\n    os.replace(tmp, path)\n\n\ndef load_records(jsonl, topk_npz) -> list[dict]:\n    recs = read_jsonl(jsonl)\n    # Read every array exactly once: each NpzFile[key] access decompresses the whole array again, and\n    # a per-record slice of it keeps that full copy alive (records x array size: the week-1 Kaggle OOM).\n    with np.load(topk_npz) as z:\n        rec_ids, offs = z["rec_ids"], z["offsets"]\n        ids_all, lp_all = z["topk_ids"], z["topk_logp"]\n    by = {int(r): i for i, r in enumerate(rec_ids)}\n    out = []\n    for r in recs:\n        i = by.get(int(r["id"]))\n        if i is None:\n            continue\n        a, b = int(offs[i]), int(offs[i + 1])\n        if b - a != len(r["out_ids"]):\n            continue\n        r["topk_ids"], r["topk_logp"] = ids_all[a:b], lp_all[a:b].astype(np.float32)\n        out.append(r)\n    return out\n\n\ndef collect_model(spec: dict, splits: dict, out_dir, batch: int, max_new: int, device: str, k: int = 20,\n                  log=print, max_memory=None, keep_loaded: bool = False):\n    """Generate + teacher-forced top-K for every split in ``splits`` ({name: rows}). Skips finished\n    splits. Returns dict(name -> records) plus model info; the model is freed unless keep_loaded."""\n    out_dir = Path(out_dir)\n    out_dir.mkdir(parents=True, exist_ok=True)\n    info_p = out_dir / "model_info.json"\n    need = [s for s in splits if not (out_dir / f"{s}_topk.npz").exists()]\n    model = tok = None\n    if need:\n        model, tok, name = load_model(spec, device, max_memory)\n        eos_ids = model_eos_ids(model, tok, spec)\n        info = dict(key=spec["key"], name=name, eos_ids=eos_ids, vocab_size=int(model.get_output_embeddings().weight.shape[0]),\n                    len_tok=len(tok), dtype=spec.get("dtype", "fp16"), prompt_mode=spec.get("prompt_mode", "chat"),\n                    family=spec.get("family", ""))\n        info_p.write_text(json.dumps(info, indent=1))\n        for s in need:\n            t0 = time.time()\n            log(f"[{spec[\'key\']}] split {s}: {len(splits[s])} sources")\n            recs = generate(model, tok, spec, splits[s], out_dir / f"{s}.jsonl", batch, max_new, device, eos_ids, log)\n            tk = teacher_forced_topk(model, recs, device, k, log)\n            chk = np.mean([float((a[:, 0] == np.asarray(r["out_ids"])).mean()) for r, (a, _) in zip(recs, tk) if len(r["out_ids"])])\n            log(f"  teacher-forced argmax == generated token: {chk:.4f}  ({time.time() - t0:.0f}s)")\n            save_topk(out_dir / f"{s}_topk.npz", recs, tk)\n            info.setdefault("tf_argmax_agreement", {})[s] = float(chk)\n            info_p.write_text(json.dumps(info, indent=1))\n    info = json.loads(info_p.read_text())\n    res = {s: load_records(out_dir / f"{s}.jsonl", out_dir / f"{s}_topk.npz") for s in splits}\n    if keep_loaded:\n        return res, info, (model, tok)\n    del model, tok\n    free_model()\n    return res, info, None\n\n\ndef free_model(*objs) -> None:\n    for o in objs:\n        del o\n    gc.collect()\n    torch = _torch()\n    if torch.cuda.is_available():\n        torch.cuda.empty_cache()\n',
    'week1': '"""Week-1 kill test of the universal (LLM-independent) Hindi MTP head, end to end (Kaggle notebook 05).\n\nStages (each skips work whose output already exists, so a dead session can resume):\n  S1 sources      FLORES+ dev/devtest English, IITB English (extra training sources) + IITB Hindi (vocab/pretrain)\n  S2 vocab        unit vocabulary from public Hindi text only\n  S3 collect      each small LLM: greedy outputs + teacher-forced top-20 on dev, extra, devtest\n  S4 phase1       Phase-1 Llama-3-8B traces (attached Kaggle input) or a fresh collection\n  S5 agree        cross-LLM agreement on word units (unit starts vs continuations)\n  S6 features     tokenizer-independent features per model and split\n  S7 pretrain     text-only pretraining on Hindi text; head latency -> c_head\n  S8 folds        leave-one-model-out heads (zero-shot / text-only / in-model) on the small models\n  S9 llama8b      zero-shot head on the 8B traces vs Entropy (Phase 1), in-model heads, baselines\n  S10 report      CSV/JSON, SUMMARY.md with the go / no-go verdict, zips\n"""\nfrom __future__ import annotations\n\nimport json\nimport shutil\nimport time\nimport zipfile\nfrom pathlib import Path\n\nimport numpy as np\n\nfrom . import collect as C\nfrom . import uni_sim as U\nfrom .gpu_traces import read_jsonl, run_info\nfrom .stats import paired_bootstrap\nfrom .uni_features import FeatureSet, TokTable, featurize_records, featurize_texts, make_encoder, seq_text\nfrom .unichar import CharCodec, UnitVocab\nfrom .unihead import build_model, load_head, predict, save_head, time_head, train_head\n\nDEFAULT_W1 = dict(\n    OUT_ROOT="/kaggle/working", RUN_NAME="week1", SMOKE=False, GIT_HASH="unknown", SEED=0,\n    MODELS=C.DEFAULT_MODELS, MIN_MODELS=3,\n    N_DEV=None, N_DEVTEST=None, N_EXTRA=3000, MONO_N=200000,\n    MAX_NEW=256, GEN_BATCH=32, TOPK=20, VAL_FRAC=0.04,\n    VOCAB_MAX=50000, VOCAB_MIN_COUNT=3,\n    ARCH=dict(d=256, layers=2, nhead=4), PRE_STEPS=3000, FT_STEPS=4000, INMODEL_STEPS=3000,\n    BATCH=256, LR=1e-3, PRE_LR=2e-3,\n    RUN_AGREE=True, N_AGREE=300, RUN_FOLDS=True, RUN_L8=True,\n    PHASE1_DIR=None, PHASE1_SEARCH=["/kaggle/input"],\n    TARGET8B=["meta-llama/Meta-Llama-3-8B-Instruct", "NousResearch/Meta-Llama-3-8B-Instruct"],\n    COLLECT_8B_IF_MISSING=True, MAX_MEMORY={0: "8GiB", 1: "10GiB"},\n    GMAX=16, TAUS=[0.0, 0.01, 0.02, 0.05, 0.1, 0.15, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9],\n    C_HEAD=None, C_SENS=[0.0, 0.02, 0.05, 0.1, 0.2], N_BOOT=1000,\n    GO_SPEEDUP=1.6, GO_RECOVERY=0.7,\n    PHASE1_SELECTED=None, PHASE1_MAIN=None, PHASE1_TIMING_FALLBACK=None,\n    SOURCES=None,            # smoke tests: dict(dev=[(id, en)], devtest=[...], extra=[...], mono=[hi, ...])\n    RESUME_FROM=None,\n)\n\n\nPHASE1_VERIFY_REPORT = (1.020, 1.025, 1.034, 1.043, 1.060, 1.063, 1.066, 1.074)   # REPORT.md section 3\n\n\ndef phase1_constants(results_dir) -> dict:\n    """Phase-1 numbers embedded into the notebook at build time (results/phase1/ is not on Kaggle):\n    selected Entropy / FixedK settings, the devtest main-table rows, and a timing fallback (REPORT.md:\n    70.6 ms per 8B step, c = 0.3231, verify(1..8)) used only if the attached traces lack timing.json."""\n    import csv\n    rd = Path(results_dir)\n    sel = json.loads((rd / "selected_params.json").read_text(encoding="utf-8"))\n    main = {}\n    with open(rd / "main_table.csv", encoding="utf-8") as f:\n        for r in csv.DictReader(f):\n            if r["policy"] in ("Entropy", "FixedK", "Oracle", "MorphEntropy_ext1"):\n                main[r["policy"]] = dict(speedup=float(r["speedup_charged"]), ci_lo=float(r["CI_low"]),\n                                         ci_hi=float(r["CI_high"]), tokens_per_round=float(r["tokens_per_verif"]))\n    t_t = 0.0706\n    timing = dict(t_target=t_t, t_draft=t_t * float(sel["c"]), c=float(sel["c"]),\n                  t_target_verify={str(g): t_t * v for g, v in zip(range(1, 9), PHASE1_VERIFY_REPORT)},\n                  source="Phase-1 REPORT.md (fallback)")\n    return dict(PHASE1_SELECTED={k: sel["selected"][k] for k in ("Entropy", "FixedK")}, PHASE1_MAIN=main,\n                PHASE1_TIMING_FALLBACK=timing)\n\n\nclass Log:\n    def __init__(self, path: Path):\n        self.path = path\n        self.t0 = time.time()\n\n    def __call__(self, *a):\n        msg = " ".join(str(x) for x in a)\n        stamp = f"[{(time.time() - self.t0) / 60:6.1f}m] "\n        print(stamp + msg, flush=True)\n        with open(self.path, "a", encoding="utf-8") as f:\n            f.write(stamp + msg + "\\n")\n\n\ndef _device():\n    import torch\n    return "cuda:0" if torch.cuda.is_available() else "cpu"\n\n\ndef _mem() -> str:\n    """Free RAM / GPU memory, for the log (a killed kernel leaves no traceback)."""\n    parts = []\n    try:\n        import psutil\n        vm = psutil.virtual_memory()\n        parts.append(f"RAM free {vm.available / 2**30:.1f}/{vm.total / 2**30:.0f} GB")\n    except Exception:  # noqa: BLE001\n        pass\n    try:\n        import torch\n        if torch.cuda.is_available():\n            f, t = torch.cuda.mem_get_info()\n            parts.append(f"GPU0 free {f / 2**30:.1f}/{t / 2**30:.0f} GB")\n    except Exception:  # noqa: BLE001\n        pass\n    return ", ".join(parts) or "memory n/a"\n\n\ndef _md_table(rows: list[dict], cols: list[str], fmt: dict | None = None) -> str:\n    fmt = fmt or {}\n    out = ["| " + " | ".join(cols) + " |", "|" + "---|" * len(cols)]\n    for r in rows:\n        cells = []\n        for c in cols:\n            v = r.get(c, "")\n            if isinstance(v, float):\n                v = fmt.get(c, "{:.3f}").format(v)\n            cells.append(str(v))\n        out.append("| " + " | ".join(cells) + " |")\n    return "\\n".join(out)\n\n\ndef _write_csv(path: Path, rows: list[dict]) -> None:\n    import pandas as pd\n    pd.DataFrame(rows).to_csv(path, index=False)\n\n\n# --------------------------------------------------------------------------- #\n# S1-S2: sources and vocabulary\n# --------------------------------------------------------------------------- #\n\ndef stage_sources(cfg, root: Path, log) -> dict:\n    p = root / "data" / "sources.json"\n    if p.exists():\n        S = json.loads(p.read_text(encoding="utf-8"))\n        log(f"S1 sources: reused {p.name} (dev {len(S[\'dev\'])}, devtest {len(S[\'devtest\'])}, "\n            f"extra {len(S[\'extra\'])}, mono {len(S[\'mono\'])})")\n        return S\n    if cfg["SOURCES"] is not None:\n        S = {k: [list(x) if isinstance(x, (list, tuple)) else x for x in v] for k, v in cfg["SOURCES"].items()}\n    else:\n        dev = C.load_flores_english("dev", cfg["N_DEV"])\n        devtest = C.load_flores_english("devtest", cfg["N_DEVTEST"])\n        extra, mono = C.load_iitb(cfg["N_EXTRA"], cfg["MONO_N"], seed=cfg["SEED"])\n        S = dict(dev=[list(r) for r in dev], devtest=[list(r) for r in devtest], extra=[list(r) for r in extra], mono=mono)\n    p.parent.mkdir(parents=True, exist_ok=True)\n    p.write_text(json.dumps(S, ensure_ascii=False), encoding="utf-8")\n    log(f"S1 sources: dev {len(S[\'dev\'])}, devtest {len(S[\'devtest\'])}, extra {len(S[\'extra\'])}, mono {len(S[\'mono\'])}")\n    return S\n\n\ndef stage_vocab(cfg, root: Path, S: dict, log) -> UnitVocab:\n    p = root / "data" / "unit_vocab.json"\n    if p.exists():\n        V = UnitVocab.load(p)\n        log(f"S2 vocab: reused ({V.size} units)")\n        return V\n    V, st = UnitVocab.build(S["mono"], cfg["VOCAB_MAX"], cfg["VOCAB_MIN_COUNT"])\n    V.save(p)\n    (root / "results" / "vocab_stats.json").write_text(json.dumps(st, indent=1))\n    log(f"S2 vocab: {st}")\n    return V\n\n\n# --------------------------------------------------------------------------- #\n# S3: collection\n# --------------------------------------------------------------------------- #\n\ndef _rows(S, split):\n    return [(int(i), s) for i, s in S[split]]\n\n\ndef stage_collect(cfg, root: Path, S: dict, log, device) -> dict:\n    out = {}\n    summ = []\n    for spec in cfg["MODELS"]:\n        key = spec["key"]\n        d = root / "data" / key\n        log(f"S3 collect: {key} ({_mem()})")\n        try:\n            res, info, _ = C.collect_model(spec, {s: _rows(S, s) for s in ("dev", "extra", "devtest")}, d,\n                                           cfg["GEN_BATCH"], cfg["MAX_NEW"], device, cfg["TOPK"], log)\n            rows = []\n            for s, recs in res.items():\n                txt = "".join(r["out_text"] for r in recs)\n                deva = sum("ऀ" <= ch <= "ॿ" for ch in txt) / max(sum(not ch.isspace() for ch in txt), 1)\n                rows.append(dict(model=key, name=info["name"], split=s, n=len(recs), status="ok",\n                                 mean_tokens=float(np.mean([len(r["out_ids"]) for r in recs])) if recs else 0.0,\n                                 eos_rate=float(np.mean([r["eos_end"] for r in recs])) if recs else 0.0,\n                                 hit_max=int(sum(len(r["out_ids"]) >= cfg["MAX_NEW"] for r in recs)),\n                                 deva_frac=deva, tf_argmax=info.get("tf_argmax_agreement", {}).get(s, float("nan"))))\n        except Exception as e:  # noqa: BLE001\n            log(f"S3 collect: model {key} FAILED and is skipped: {type(e).__name__}: {str(e)[:300]}")\n            summ.append(dict(model=key, status=f"failed: {type(e).__name__}"))\n            continue\n        out[key] = dict(records=res, info=info, spec=spec)\n        summ.extend(rows)\n        log(f"S3 collect: {key} done ({_mem()})")\n    _write_csv(root / "results" / "collection_summary.csv", summ)\n    log(f"S3 collect: {len(out)} models ok: {list(out)}")\n    return out\n\n\n# --------------------------------------------------------------------------- #\n# S4: Phase-1 Llama-3-8B traces\n# --------------------------------------------------------------------------- #\n\ndef _find(name: str, dirs) -> Path | None:\n    for d in dirs:\n        if d and Path(d).exists():\n            hits = sorted(Path(d).rglob(name))\n            if hits:\n                return hits[0]\n    return None\n\n\ndef find_phase1(cfg, root: Path, log) -> dict:\n    dirs = [cfg["PHASE1_DIR"], cfg["RESUME_FROM"], *cfg["PHASE1_SEARCH"], str(root / "phase1")]\n    found = {}\n    for split in ("dev", "devtest"):\n        npz = _find(f"traces_{split}.npz", dirs)\n        if npz is None:\n            z = _find(f"traces_v2_{split}.zip", dirs)\n            if z is not None:\n                dst = root / "phase1" / split\n                dst.mkdir(parents=True, exist_ok=True)\n                with zipfile.ZipFile(z) as zf:\n                    zf.extractall(dst)\n                npz = dst / f"traces_{split}.npz"\n                log(f"S4: extracted {z}")\n        if npz is not None and npz.exists():\n            jl = npz.parent / f"target_outputs_{split}.jsonl"\n            jl = jl if jl.exists() else _find(f"target_outputs_{split}.jsonl", dirs)\n            tm = npz.parent / "timing.json"\n            tm = tm if tm.exists() else _find("timing.json", [str(npz.parent.parent)] + dirs)\n            found[split] = dict(npz=npz, jsonl=jl, timing=tm)\n            log(f"S4: Phase-1 {split}: {npz} (outputs: {jl}, timing: {tm})")\n    return found\n\n\ndef load_tokenizer_any(names):\n    from transformers import AutoTokenizer\n    last = None\n    for n in names:\n        try:\n            return AutoTokenizer.from_pretrained(n), n\n        except Exception as e:  # noqa: BLE001\n            last = e\n    raise RuntimeError(f"no tokenizer from {names}: {last}")\n\n\ndef stage_phase1(cfg, root: Path, S: dict, log, device) -> dict | None:\n    from .traces import load_traces\n    found = find_phase1(cfg, root, log)\n    timing = None\n    if "devtest" in found:\n        L8 = dict(source="phase1", traces={}, records={})\n        for split, f in found.items():\n            traces, meta = load_traces(f["npz"])\n            outs = {int(r["id"]): r for r in read_jsonl(f["jsonl"])} if f["jsonl"] else {}\n            eos_ids = set(meta.get("eos_ids", []))\n            recs = []\n            for t in traces:\n                o = outs.get(int(t.id), {})\n                recs.append(dict(id=int(t.id), prompt_text=C.head_prompt_text(o["src"]) if "src" in o else "",\n                                 prompt_ids=o.get("prompt_ids"), out_ids=np.asarray(t.target_ids, np.int64),\n                                 topk_ids=t.target_topk_ids, topk_logp=t.target_topk_logp.astype(np.float32),\n                                 eos_end=bool(len(t.target_ids)) and int(t.target_ids[-1]) in eos_ids))\n            L8["traces"][split], L8["records"][split] = traces, recs\n            L8["eos_ids"] = sorted(eos_ids)\n            if f["timing"] and timing is None:\n                timing = json.loads(Path(f["timing"]).read_text())\n        if "dev" not in L8["records"]:\n            log("S4: Phase-1 dev traces not found: in-model 8B heads and 8B-dev tuning are skipped")\n    elif cfg["COLLECT_8B_IF_MISSING"]:\n        log("S4: Phase-1 traces not attached -> collecting Llama-3-8B outputs here (slow, ~2-3 h on 2xT4)")\n        spec = dict(key="llama3-8b", names=cfg["TARGET8B"], dtype="fp16", device_map="auto", family="llama3")\n        d = root / "data" / "llama3-8b"\n        res, info, mt = C.collect_model(spec, {s: _rows(S, s) for s in ("dev", "devtest")}, d, 16, cfg["MAX_NEW"],\n                                        device, cfg["TOPK"], log, max_memory=cfg["MAX_MEMORY"], keep_loaded=True)\n        tp = d / "timing.json"\n        if not tp.exists() and mt is not None and mt[0] is not None:\n            from .gpu_traces import time_fwd\n            m = mt[0]\n            t_t = time_fwd(m, 1, device=device)\n            ver = {str(g): time_fwd(m, g + 1, device=device) for g in range(1, 9)}\n            tp.write_text(json.dumps(dict(t_target=t_t, t_target_verify=ver, c=None, source="week1 fresh"), indent=1))\n        if mt is not None:\n            del mt\n            C.free_model()\n        timing = json.loads(tp.read_text()) if tp.exists() else None\n        L8 = dict(source="fresh", traces={}, records=res, eos_ids=info["eos_ids"])\n    else:\n        log("S4: no Phase-1 traces and COLLECT_8B_IF_MISSING=False -> the Llama-3-8B fold is skipped")\n        return None\n    if timing is None:\n        timing = cfg["PHASE1_TIMING_FALLBACK"]\n        log("S4: timing.json not found; using the Phase-1 REPORT numbers embedded in the notebook")\n    tok, name = load_tokenizer_any(cfg["TARGET8B"])\n    L8.update(timing=timing, tok=tok, tok_name=name)\n    log(f"S4: 8B data from {L8[\'source\']}; splits {list(L8[\'records\'])}; tokenizer {name}; "\n        f"t_target {timing[\'t_target\'] * 1000:.1f} ms")\n    return L8\n\n\n# --------------------------------------------------------------------------- #\n# S5: cross-LLM agreement\n# --------------------------------------------------------------------------- #\n\ndef stage_agree(cfg, root: Path, M: dict, L8, tables: dict, log, device) -> list:\n    p = root / "results" / "agreement.csv"\n    if p.exists():\n        import pandas as pd\n        log("S5 agree: reused")\n        return pd.read_csv(p).to_dict("records")\n    texts = {}\n    for key, m in M.items():\n        recs = m["records"]["devtest"][: cfg["N_AGREE"]]\n        texts[key] = [(r["src"], seq_text(tables[key], r["out_ids"]).text) for r in recs]\n    if L8 is not None and "devtest" in L8["records"]:\n        tl = TokTable(L8["tok"], L8["eos_ids"])\n        texts["llama3-8b"] = []\n        for r in L8["records"]["devtest"][: cfg["N_AGREE"]]:\n            src = r["prompt_text"].split("\\n\\n", 1)[-1] if r["prompt_text"] else ""\n            texts["llama3-8b"].append((src, seq_text(tl, r["out_ids"]).text))\n    rows = []\n    for key, m in M.items():\n        model, tok, _ = C.load_model(m["spec"], device)\n        enc = make_encoder(tok)\n        for a, items in texts.items():\n            items = [(s, t) for s, t in items if t.strip()]\n            if not items:\n                continue\n            ids = enc([t for _, t in items])\n            recs = [dict(prompt_ids=tok(C.build_model_prompt(tok, m["spec"], s), add_special_tokens=False)["input_ids"],\n                         out_ids=i) for (s, _), i in zip(items, ids)]\n            am = C.teacher_forced_topk(model, recs, device, argmax_only=True)\n            tot = {}\n            for (s, t), i, a_ in zip(items, ids, am):\n                for k_, v in U.unit_agreement(t, i, list(a_), tables[key]).items():\n                    tot[k_] = tot.get(k_, 0) + v\n            row = dict(text_from=a, model=key, same=(a == key), **U.agreement_table(tot))\n            rows.append(row)\n            log(f"S5 agree: {key} on {a}\'s text: start {row[\'p_start\']:.3f} cont {row[\'p_cont_token\']:.3f} "\n                f"unit {row[\'p_unit\']:.3f} unit|start {row[\'p_unit_given_start\']:.3f}")\n        del model, tok\n        C.free_model()\n    _write_csv(p, rows)\n    return rows\n\n\n# --------------------------------------------------------------------------- #\n# S6: features\n# --------------------------------------------------------------------------- #\n\ndef _split_records(recs, frac, seed):\n    rng = np.random.default_rng(seed)\n    m = rng.random(len(recs)) < frac\n    if not m.any() and len(recs) > 1:\n        m[0] = True\n    return [r for r, x in zip(recs, m) if not x], [r for r, x in zip(recs, m) if x]\n\n\ndef stage_features(cfg, root: Path, M: dict, L8, V: UnitVocab, S: dict, log):\n    from transformers import AutoTokenizer\n    fd = root / "features"\n    fd.mkdir(exist_ok=True)\n    codec = CharCodec()\n    F, R, T = {}, {}, {}\n    for key, m in M.items():\n        info = m["info"]\n        tok = AutoTokenizer.from_pretrained(info["name"])\n        T[key] = dict(table=TokTable(tok, info["eos_ids"], info["vocab_size"], codec), enc=make_encoder(tok), tok=tok)\n        tr, va = _split_records(m["records"]["dev"] + m["records"]["extra"], cfg["VAL_FRAC"], cfg["SEED"])\n        R[key] = dict(train=tr, val=va, test=m["records"]["devtest"])\n        F[key] = {}\n        for part, recs in R[key].items():\n            p = fd / f"{key}_{part}.npz"\n            if p.exists():\n                F[key][part] = FeatureSet.load(p)\n            else:\n                t0 = time.time()\n                F[key][part] = featurize_records(recs, T[key]["table"], V, codec, meta=dict(model=key, part=part))\n                F[key][part].save(p)\n                log(f"S6 features {key}/{part}: {F[key][part].n_pos} positions ({time.time() - t0:.0f}s)")\n    if L8 is not None:\n        key = "llama3-8b"\n        T[key] = dict(table=TokTable(L8["tok"], L8["eos_ids"], None, codec), enc=make_encoder(L8["tok"]), tok=L8["tok"])\n        R[key], F[key] = {}, {}\n        for split, part in (("dev", "train"), ("devtest", "test")):\n            if split not in L8["records"]:\n                continue\n            R[key][part] = L8["records"][split]\n            p = fd / f"{key}_{part}.npz"\n            if p.exists():\n                F[key][part] = FeatureSet.load(p)\n            else:\n                F[key][part] = featurize_records(R[key][part], T[key]["table"], V, codec, meta=dict(model=key, part=part))\n                F[key][part].save(p)\n                log(f"S6 features {key}/{part}: {F[key][part].n_pos} positions")\n    p = fd / "mono.npz"\n    if p.exists():\n        mono = FeatureSet.load(p)\n    else:\n        mono = featurize_texts(S["mono"], V, codec, seed=cfg["SEED"])\n        mono.save(p)\n    log(f"S6 features mono: {mono.n_pos} positions")\n    # coverage of the vocabulary on each model\'s test outputs\n    cov = []\n    for key in R:\n        if "test" in R[key]:\n            texts = [seq_text(T[key]["table"], r["out_ids"]).text for r in R[key]["test"]]\n            cov.append(dict(model=key, unit_coverage=V.coverage(texts),\n                            head1_target_ok=float((F[key]["test"].tgt[:, 0] != -100).mean()) if F[key]["test"].n_pos else 0.0))\n    _write_csv(root / "results" / "vocab_coverage.csv", cov)\n    return F, R, T, mono\n\n\n# --------------------------------------------------------------------------- #\n# S7-S9: heads\n# --------------------------------------------------------------------------- #\n\ndef _train_or_load(path: Path, init: Path | None, V, cfg, sources, val, use_cands, steps, lr, device, log, tag):\n    if path.exists():\n        m, extra = load_head(path, device)\n        log(f"  {tag}: reused {path.name}")\n        return m, extra\n    if init is not None and init.exists():\n        m, _ = load_head(init, device)\n    else:\n        m = build_model(V.size, cfg["ARCH"]).to(device)\n    log(f"  {tag}: training {steps} steps on {sum(len(i) for _, i in sources)} positions (cands={use_cands})")\n    out = train_head(m, sources, steps, cfg["BATCH"], lr, device, use_cands, val=val, log=log, seed=cfg["SEED"],\n                     log_every=max(steps // 10, 1))\n    save_head(m, path, dict(tag=tag, use_cands=use_cands, val=out.get("val"), train_sec=out["train_sec"]))\n    return m, dict(tag=tag, use_cands=use_cands, val=out.get("val"), train_sec=out["train_sec"])\n\n\ndef _pred(path: Path, model, fs, device, use_cands):\n    if path.exists():\n        z = np.load(path)\n        return z["W"], z["P"]\n    W, P = predict(model, fs, device, use_cands)\n    np.savez(path, W=W, P=P)\n    return W, P\n\n\ndef _truth_W(fs) -> np.ndarray:\n    W = fs.tgt.copy()\n    W[W < 0] = -1\n    for i in range(1, W.shape[1]):          # a chain stops at the first missing unit\n        W[:, i] = np.where(W[:, i - 1] < 0, -1, W[:, i])\n    return W\n\n\ndef _head_acc(W, fs) -> dict:\n    t = fs.tgt\n    out = {}\n    for i in range(t.shape[1]):\n        m = t[:, i] != -100\n        out[f"acc{i + 1}"] = float((W[m, i] == t[m, i]).mean()) if m.any() else float("nan")\n    m = (t[:, 0] != -100)\n    out["chain2"] = float(((W[:, 0] == t[:, 0]) & (W[:, 1] == t[:, 1]))[m].mean()) if m.any() else float("nan")\n    return out\n\n\nclass Evaluator:\n    """Converts head predictions on one target\'s sequences into that target\'s tokens and simulates."""\n\n    def __init__(self, key, records, T, V, verify, c_head, cfg):\n        self.key, self.V, self.verify, self.c_head, self.cfg = key, V, verify, c_head, cfg\n        self.seqs = U.build_simseqs(records, T["table"], V)\n        self.enc = T["enc"]\n        self.table = T["table"]\n        self._cache: dict = {}\n\n    def drafts(self, W):\n        """Converted drafts for predictions W (cached: the same W is simulated under many settings)."""\n        import hashlib\n        key = hashlib.md5(np.ascontiguousarray(W).tobytes()).hexdigest()\n        if key not in self._cache:\n            if len(self._cache) >= 2:\n                self._cache.pop(next(iter(self._cache)))\n            self._cache[key] = U.convert_all(self.seqs, W, self.V, self.enc, self.table.bytes, self.cfg["GMAX"])\n        return self._cache[key]\n\n    def run(self, W, P, tau, c_head=None, oracle_level=False):\n        d, st = self.drafts(W)\n        R = U.simulate(self.seqs, d, P, tau, self.c_head if c_head is None else c_head, self.verify,\n                       self.cfg["GMAX"], oracle_level=oracle_level)\n        return R, d, st\n\n    def tau_grid(self, W, P):\n        d, _ = self.drafts(W)\n        return [dict(tau=t, speedup=U.simulate(self.seqs, d, P, t, self.c_head, self.verify, self.cfg["GMAX"])["speedup"])\n                for t in self.cfg["TAUS"]]\n\n\n_VAL_EV: dict = {}\n\n\ndef _val_tau(model, keys, F, R, T, V, verify, c_head, cfg, device, use_cands) -> float:\n    """Draft-length threshold chosen on the *training* models\' validation outputs (never the held-out one)."""\n    grids = []\n    for k in keys:\n        if not F[k]["val"].n_pos:\n            continue\n        if k not in _VAL_EV:\n            _VAL_EV[k] = Evaluator(k, R[k]["val"], T[k], V, verify, c_head, cfg)\n        Wv, Pv = predict(model, F[k]["val"], device, use_cands)\n        grids.append(_VAL_EV[k].tau_grid(Wv, Pv))\n    return _pick_tau(grids) if grids else 0.0\n\n\ndef _pick_tau(grids: list[list[dict]]) -> float:\n    taus = [g["tau"] for g in grids[0]]\n    mean = [np.mean([gr[i]["speedup"] for gr in grids]) for i in range(len(taus))]\n    return float(taus[int(np.argmax(mean))])\n\n\ndef stage_heads(cfg, root: Path, F, R, T, mono, V, L8, log, device) -> dict:\n    hd, pd_ = root / "heads", root / "preds"\n    hd.mkdir(exist_ok=True); pd_.mkdir(exist_ok=True)\n    res = dict(lomo=[], lomo_sim=[], l8=[], l8_tau=[], l8_sens=[], conv=[], nn_cov=[], timing={})\n    # ---- S7 pretrain + timing\n    pre = hd / "pretrain_text.pt"\n    tr, va = mono.split_by_seq(0.01, cfg["SEED"])\n    m_pre, _ = _train_or_load(pre, None, V, cfg, [(mono, tr)], [(mono, va)], False, cfg["PRE_STEPS"], cfg["PRE_LR"],\n                              device, log, "S7 pretrain (text only, Hindi corpus)")\n    timing = cfg["PHASE1_TIMING_FALLBACK"] if L8 is None else L8["timing"]\n    verify = U.Verify(timing)\n    any_fs = next(F[k]["test"] for k in F if "test" in F[k] and F[k]["test"].n_pos)\n    th = time_head(m_pre, any_fs, device, use_cands=True) if device != "cpu" or cfg["SMOKE"] else {}\n    tt = time_head(m_pre, any_fs, device, use_cands=False) if device != "cpu" or cfg["SMOKE"] else {}\n    c_meas = (th["t_head_gpu"] + th["t_inputs_cpu"]) / timing["t_target"] if th else 0.05\n    c_head = float(cfg["C_HEAD"]) if cfg["C_HEAD"] is not None else c_meas\n    res["timing"] = dict(head_with_cands=th, head_text_only=tt, t_target=timing["t_target"], c_head_measured=c_meas,\n                         c_head_used=c_head, verify=verify.as_dict(),\n                         note="c_head = (GPU forward + chained argmax + CPU input building) / 8B decode step")\n    log(f"S7 head latency: {th.get(\'t_head_gpu\', 0) * 1000:.2f} ms GPU + {th.get(\'t_inputs_cpu\', 0) * 1000:.2f} ms CPU "\n        f"-> c_head = {c_meas:.4f} (used {c_head:.4f}); params {th.get(\'params\')}")\n    small = [k for k in F if k != "llama3-8b"]\n\n    def sources(keys, part="train"):\n        return [(F[k][part], F[k][part].trainable()) for k in keys if part in F[k] and F[k][part].n_pos]\n\n    # ---- S8 leave-one-model-out on the small models\n    if cfg["RUN_FOLDS"] and len(small) >= 3:\n        for h in small:\n            others = [k for k in small if k != h]\n            ev = Evaluator(h, R[h]["test"], T[h], V, verify, c_head, cfg)\n            variants = [("zeroshot", others, True, pre, cfg["FT_STEPS"]),\n                        ("textonly", others, False, pre, cfg["FT_STEPS"]),\n                        ("inmodel", [h], True, pre, cfg["INMODEL_STEPS"])]\n            preds = {}\n            for name, keys, uc, init, steps in variants:\n                path = hd / f"lomo_{h}_{name}.pt"\n                m, ex = _train_or_load(path, init, V, cfg, sources(keys), sources(keys, "val"), uc, steps, cfg["LR"],\n                                       device, log, f"S8 fold {h} / {name}")\n                W, P = _pred(pd_ / f"lomo_{h}_{name}_test.npz", m, F[h]["test"], device, uc)\n                tau = _val_tau(m, keys, F, R, T, V, verify, c_head, cfg, device, uc)\n                preds[name] = (W, P, tau)\n                res["lomo"].append(dict(heldout=h, variant=name, train_models="+".join(keys), tau=tau,\n                                        **_head_acc(W, F[h]["test"]), **{f"val_{k}": v for k, v in (ex.get("val") or {}).items()}))\n                del m\n            truthW = _truth_W(F[h]["test"])\n            for name, (W, P, tau) in list(preds.items()) + [("truth_units", (truthW, np.ones_like(truthW, np.float32), 0.0))]:\n                Rr, _, st = ev.run(W, P, tau)\n                res["lomo_sim"].append(dict(heldout=h, method=name, tau=tau, speedup_8b_cost=Rr["speedup"],\n                                            tokens_per_round=Rr["tokens_per_round"], mean_draft_len=Rr["mean_draft_len"],\n                                            acceptance=Rr["acceptance"]))\n                res["conv"].append(dict(target=h, method=name, **st))\n            for bname, dr in (("pld", U.pld_drafts(ev.seqs)), ("logitspec", U.logitspec_drafts(ev.seqs))):\n                Rb = U.simulate(ev.seqs, dr, U.ones_like_probs(dr), 0.0, 0.0, verify, cfg["GMAX"])\n                res["lomo_sim"].append(dict(heldout=h, method=bname, tau=0.0, speedup_8b_cost=Rb["speedup"],\n                                            tokens_per_round=Rb["tokens_per_round"], mean_draft_len=Rb["mean_draft_len"],\n                                            acceptance=Rb["acceptance"]))\n            res["nn_cov"].append(dict(target=h, **U.next_next_coverage(ev.seqs)))\n            log(f"S8 fold {h}: " + "; ".join(f"{r[\'method\']} {r[\'speedup_8b_cost\']:.3f}x ({r[\'tokens_per_round\']:.2f} tok/round)"\n                                            for r in res["lomo_sim"] if r["heldout"] == h))\n    elif cfg["RUN_FOLDS"]:\n        log(f"S8: only {len(small)} small models loaded (< 3): leave-one-model-out skipped")\n\n    # ---- S9 Llama-3-8B\n    if cfg["RUN_L8"] and L8 is not None and "llama3-8b" in F and "test" in F["llama3-8b"]:\n        res.update(stage_l8(cfg, root, F, R, T, V, L8, small, sources, pre, verify, c_head, log, device))\n    return res\n\n\ndef stage_l8(cfg, root, F, R, T, V, L8, small, sources, pre, verify, c_head, log, device) -> dict:\n    hd, pd_ = root / "heads", root / "preds"\n    key = "llama3-8b"\n    out = dict(l8=[], l8_tau=[], l8_sens=[], conv=[], nn_cov=[], l8_heads=[])\n    ev = Evaluator(key, R[key]["test"], T[key], V, verify, c_head, cfg)\n    ev_dev = Evaluator(key, R[key]["train"], T[key], V, verify, c_head, cfg) if "train" in R[key] else None\n    families = {s["key"]: s.get("family", "") for s in cfg["MODELS"]}\n    nollama = [k for k in small if families.get(k) != "llama3"]\n    variants = [("zeroshot_all", small, True, pre, cfg["FT_STEPS"], "train"),\n                ("textonly_all", small, False, pre, cfg["FT_STEPS"], "train")]\n    if len(nollama) >= 2 and len(nollama) < len(small):\n        variants.append(("zeroshot_nollama", nollama, True, pre, cfg["FT_STEPS"], "train"))\n    preds = {}\n    for name, keys, uc, init, steps, part in variants:\n        path = hd / f"l8_{name}.pt"\n        m, ex = _train_or_load(path, init, V, cfg, sources(keys), sources(keys, "val"), uc, steps, cfg["LR"], device, log,\n                               f"S9 {name}")\n        W, P = _pred(pd_ / f"l8_{name}_test.npz", m, F[key]["test"], device, uc)\n        tau_zs = _val_tau(m, keys, F, R, T, V, verify, c_head, cfg, device, uc)\n        tau_dev = None\n        if ev_dev is not None:\n            Wd, Pd = _pred(pd_ / f"l8_{name}_dev.npz", m, F[key]["train"], device, uc)\n            tau_dev = _pick_tau([ev_dev.tau_grid(Wd, Pd)])\n        preds[name] = dict(W=W, P=P, tau=tau_zs, tau_dev=tau_dev, use_cands=uc, train=keys)\n        out["l8_heads"].append(dict(variant=name, train_models="+".join(keys), tau_zeroshot=tau_zs, tau_8b_dev=tau_dev,\n                                    **_head_acc(W, F[key]["test"])))\n        del m\n    # in-model references: need 8B dev traces; trained on 90% of dev, threshold chosen on the other 10%\n    if "train" in F[key] and F[key]["train"].n_pos:\n        fs8 = F[key]["train"]\n        val_seq = np.random.default_rng(cfg["SEED"]).random(len(fs8.prompt_u)) < 0.1\n        if not val_seq.any():\n            val_seq[0] = True\n        trn = fs8.trainable()\n        tr8 = [(fs8, trn[~val_seq[fs8.pos_seq[trn]]])]\n        va8 = [(fs8, trn[val_seq[fs8.pos_seq[trn]]])]\n        va_pos = np.flatnonzero(val_seq[fs8.pos_seq])\n        ev_dval = Evaluator(key, [r for r, v in zip(R[key]["train"], val_seq) if v], T[key], V, verify, c_head, cfg)\n        for name, init, steps in (("inmodel_ft", hd / "l8_zeroshot_all.pt", cfg["INMODEL_STEPS"]),\n                                  ("inmodel_own", pre, cfg["FT_STEPS"])):\n            path = hd / f"l8_{name}.pt"\n            m, ex = _train_or_load(path, init, V, cfg, tr8, va8, True, steps, cfg["LR"] * (0.5 if name == "inmodel_ft" else 1),\n                                   device, log, f"S9 {name}")\n            W, P = _pred(pd_ / f"l8_{name}_test.npz", m, F[key]["test"], device, True)\n            Wv, Pv = predict(m, fs8, device, True, idx=va_pos)\n            tau_dev = _pick_tau([ev_dval.tau_grid(Wv, Pv)])\n            preds[name] = dict(W=W, P=P, tau=tau_dev, tau_dev=tau_dev, use_cands=True, train=[key])\n            out["l8_heads"].append(dict(variant=name, train_models=key + " dev (90%)", tau_zeroshot=None, tau_8b_dev=tau_dev,\n                                        **_head_acc(W, F[key]["test"])))\n            del m\n    # ---- baselines on the same 8B devtest sentences\n    sims = {}\n    ent = None\n    if L8["source"] == "phase1" and "devtest" in L8["traces"] and cfg["PHASE1_SELECTED"]:\n        from .policies import Entropy, FixedK\n        from .simulator import GAMMA_MAX, MeasuredVerify, simulate as p1sim\n        t = L8["timing"]\n        c1 = t.get("c") or t["t_draft"] / t["t_target"]\n        mv = MeasuredVerify(t)\n        sel = cfg["PHASE1_SELECTED"]\n        for nm, pol, gm in (("Entropy (Phase 1, 1B draft)", Entropy(**sel["Entropy"]), GAMMA_MAX),\n                            ("FixedK (Phase 1, 1B draft)", FixedK(**sel["FixedK"]), sel["FixedK"]["k"])):\n            r1 = p1sim(L8["traces"]["devtest"], pol, gm, c1, mv, record_rounds=False)\n            em, co = r1.sent_emitted.astype(float), r1.sent_cost\n            sims[nm] = dict(sent_emitted=em, sent_cost=co, speedup=float(em.sum() / co.sum()),\n                            tokens_per_round=r1.emitted / max(r1.rounds, 1), mean_draft_len=r1.drafted / max(r1.rounds, 1),\n                            acceptance=r1.accepted / max(r1.drafted, 1), rounds=r1.rounds)\n        ent = sims["Entropy (Phase 1, 1B draft)"]\n    for bname, dr in (("Prompt lookup (training-free)", U.pld_drafts(ev.seqs)),\n                      ("LogitSpec-style chain (training-free)", U.logitspec_drafts(ev.seqs))):\n        sims[bname] = U.simulate(ev.seqs, dr, U.ones_like_probs(dr), 0.0, 0.0, verify, cfg["GMAX"])\n    out["nn_cov"].append(dict(target=key, **U.next_next_coverage(ev.seqs)))\n    truthW = _truth_W(F[key]["test"])\n    sims["Truth units (upper bound of this head design)"], _, st = ev.run(truthW, np.ones_like(truthW, np.float32), 0.0)\n    out["conv"].append(dict(target=key, method="truth_units", **st))\n    for name, pr in preds.items():\n        R_, _, st = ev.run(pr["W"], pr["P"], pr["tau"])\n        label = {"zeroshot_all": "Universal head, zero-shot (all small LLMs)",\n                 "zeroshot_nollama": "Universal head, zero-shot (no Llama-family data)",\n                 "textonly_all": "Text-only head, zero-shot (ablation: no distribution input)",\n                 "inmodel_ft": "In-model head (zero-shot head fine-tuned on 8B dev)",\n                 "inmodel_own": "In-model head (8B dev only)"}[name]\n        sims[label] = R_\n        pr["label"] = label\n        out["conv"].append(dict(target=key, method=name, **st))\n        R0, _, _ = ev.run(pr["W"], pr["P"], 0.0)\n        sims[label + " [tau=0]"] = R0\n        Ro, _, _ = ev.run(pr["W"], pr["P"], 0.0, oracle_level=True)\n        sims[label + " [oracle length, not a method]"] = Ro\n        for t_row in ev.tau_grid(pr["W"], pr["P"]):\n            out["l8_tau"].append(dict(variant=name, **t_row))\n        for cs in cfg["C_SENS"]:\n            Rs, _, _ = ev.run(pr["W"], pr["P"], pr["tau"], c_head=cs)\n            out["l8_sens"].append(dict(variant=name, c_head=cs, speedup=Rs["speedup"]))\n    ref = ent if ent is not None else None\n    for nm, R_ in sims.items():\n        tau = next((p["tau"] for p in preds.values() if p.get("label") == nm), None)\n        out["l8"].append(U.summary_row(nm, R_, cfg["N_BOOT"], cfg["SEED"], ref=ref if nm != "Entropy (Phase 1, 1B draft)" else None,\n                                       tau=tau, c_head=c_head if nm.startswith(("Universal", "Text-only", "In-model", "Truth")) else None))\n    # dist vs text-only paired comparison (the novelty signal)\n    if "zeroshot_all" in preds and "textonly_all" in preds:\n        a, b = sims[preds["zeroshot_all"]["label"]], sims[preds["textonly_all"]["label"]]\n        out["dist_vs_text"] = paired_bootstrap(a["sent_emitted"], a["sent_cost"], b["sent_emitted"], b["sent_cost"],\n                                               cfg["N_BOOT"], cfg["SEED"])\n    out["preds_meta"] = {k: dict(tau=v["tau"], tau_dev=v["tau_dev"], train=v["train"]) for k, v in preds.items()}\n    out["phase1_reference"] = cfg["PHASE1_MAIN"]\n    log("S9 Llama-3-8B devtest:\\n" + _md_table(out["l8"], ["method", "speedup", "ci_lo", "ci_hi", "tokens_per_round"]))\n    return out\n\n\n# --------------------------------------------------------------------------- #\n# S10: verdict and report\n# --------------------------------------------------------------------------- #\n\ndef verdict(cfg, res: dict) -> dict:\n    rows = {r["method"]: r for r in res.get("l8", [])}\n    zs = rows.get("Universal head, zero-shot (all small LLMs)")\n    if zs is None:\n        return dict(decision="INCOMPLETE", reason="the Llama-3-8B fold did not run (attach the Phase-1 traces or allow collection)")\n    ent = rows.get("Entropy (Phase 1, 1B draft)")\n    ent_s = ent["speedup"] if ent else (cfg["PHASE1_MAIN"] or {}).get("Entropy", {}).get("speedup")\n    inm = rows.get("In-model head (zero-shot head fine-tuned on 8B dev)") or rows.get("In-model head (8B dev only)")\n    txt = rows.get("Text-only head, zero-shot (ablation: no distribution input)")\n    rec = (zs["speedup"] - 1) / (inm["speedup"] - 1) if inm and inm["speedup"] > 1 else None\n    beats = zs.get("diff_lo", -1) > 0 if ent else (ent_s is not None and zs["speedup"] > ent_s)\n    dist_helps = res.get("dist_vs_text") and res["dist_vs_text"]["lo"] > 0\n    if zs["speedup"] >= cfg["GO_SPEEDUP"] and (rec is None or rec >= cfg["GO_RECOVERY"]):\n        dec = "GO"\n    elif beats and (rec is None or rec >= cfg["GO_RECOVERY"]):\n        dec = "WEAK GO"\n    else:\n        dec = "NO GO"\n    return dict(decision=dec, zeroshot_speedup=zs["speedup"], zeroshot_ci=[zs["ci_lo"], zs["ci_hi"]],\n                entropy_speedup=ent_s, beats_entropy_ci=bool(beats), inmodel_speedup=inm["speedup"] if inm else None,\n                recovery=rec, textonly_speedup=txt["speedup"] if txt else None,\n                distribution_input_helps=bool(dist_helps), dist_vs_text=res.get("dist_vs_text"),\n                criteria=dict(go_speedup=cfg["GO_SPEEDUP"], go_recovery=cfg["GO_RECOVERY"]))\n\n\ndef write_report(cfg, root: Path, res: dict, agree: list, log) -> dict:\n    rd = root / "results"\n    for name in ("lomo", "lomo_sim", "l8", "l8_tau", "l8_sens", "conv", "nn_cov", "l8_heads"):\n        if res.get(name):\n            _write_csv(rd / f"{name}.csv", res[name])\n    (rd / "timing_head.json").write_text(json.dumps(res.get("timing", {}), indent=1, default=float))\n    V = verdict(cfg, res)\n    (rd / "verdict.json").write_text(json.dumps(V, indent=1, default=float))\n    L = [f"# Week-1 kill test: universal Hindi MTP head\\n", f"Build `{cfg[\'GIT_HASH\']}`. Generated by `lgmtp.week1`.\\n",\n         f"## Verdict: **{V[\'decision\']}**\\n", "```", json.dumps(V, indent=1, default=float), "```\\n"]\n    if res.get("l8"):\n        L += ["## Llama-3-8B, FLORES devtest (exact trace simulation)\\n",\n              _md_table(res["l8"], ["method", "speedup", "ci_lo", "ci_hi", "tokens_per_round", "mean_draft_len",\n                                    "acceptance", "diff_vs_ref", "diff_lo", "diff_hi", "tau"]), "",\n              "Paired differences are against Entropy (Phase 1, 1B draft) on the same sentences. "\n              "Rows marked [oracle length] pick the best draft length with hindsight and are diagnostics only.\\n"]\n    if res.get("l8_heads"):\n        L += ["## Head accuracy on 8B devtest (teacher forced, per unit)\\n",\n              _md_table(res["l8_heads"], ["variant", "train_models", "acc1", "acc2", "acc3", "chain2", "tau_zeroshot", "tau_8b_dev"]), ""]\n    if res.get("lomo_sim"):\n        L += ["## Leave-one-model-out on the small LLMs (speed-up under the 8B cost model)\\n",\n              _md_table(res["lomo_sim"], ["heldout", "method", "speedup_8b_cost", "tokens_per_round", "mean_draft_len", "acceptance", "tau"]), ""]\n    if res.get("lomo"):\n        L += ["## Leave-one-model-out head accuracy\\n",\n              _md_table(res["lomo"], ["heldout", "variant", "train_models", "acc1", "acc2", "acc3", "chain2"]), ""]\n    if agree:\n        L += ["## Cross-LLM agreement on word units (teacher forced on another model\'s output)\\n",\n              _md_table(agree, ["text_from", "model", "p_start", "p_cont_token", "p_unit", "p_unit_given_start"]), ""]\n    if res.get("nn_cov"):\n        L += ["## Next-next token in the top-k of the last distribution (LogitSpec observation)\\n",\n              _md_table(res["nn_cov"], ["target", "top1", "top2", "top5", "top10", "top20"]), ""]\n    if res.get("conv"):\n        L += ["## Draft conversion to target tokens\\n",\n              _md_table(res["conv"], ["target", "method", "positions", "levels", "anchored", "fallback", "failed", "empty"]), ""]\n    t = res.get("timing", {})\n    if t:\n        L += ["## Head cost\\n", "```", json.dumps(t, indent=1, default=float), "```"]\n    (rd / "SUMMARY.md").write_text("\\n".join(L), encoding="utf-8")\n    log(f"S10 verdict: {V[\'decision\']}")\n    return V\n\n\ndef _resume_copy(run: Path, root: Path, log):\n    """Copy an earlier run folder (the one holding data/sources.json) into `root`, never overwriting."""\n    n_new = n_had = 0\n    for f in sorted(Path(run).rglob("*")):\n        if f.is_file():\n            dst = root / f.relative_to(run)\n            if dst.exists():\n                n_had += 1\n                continue\n            dst.parent.mkdir(parents=True, exist_ok=True)\n            shutil.copy2(f, dst)\n            n_new += 1\n    models = sorted(p.name for p in (Path(run) / "data").iterdir() if p.is_dir())\n    log(f"resume: copied {n_new} files ({n_had} already present) from {run}; model folders: {models or \'none\'}")\n\n\ndef _unzip_safe(z: zipfile.ZipFile, dst: Path) -> None:\n    """extractall that also accepts backslash separators (Windows PowerShell\'s Compress-Archive)."""\n    for info in z.infolist():\n        name = info.filename.replace("\\\\", "/")\n        parts = [p for p in name.split("/") if p not in ("", ".")]\n        if name.endswith("/") or not parts or ".." in parts or ":" in parts[0]:\n            continue\n        out = dst.joinpath(*parts)\n        out.parent.mkdir(parents=True, exist_ok=True)\n        with z.open(info) as fi, open(out, "wb") as fo:\n            shutil.copyfileobj(fi, fo)\n\n\ndef locate_run(paths, run_name: str) -> Path | None:\n    """An earlier run among `paths` (folders searched recursively, or .zip files): the folder holding\n    data/sources.json, whatever it is called. Handles a notebook output attached as input, a downloaded\n    output re-uploaded as a Dataset (unzipped by Kaggle or kept as a .zip) and zips without the run folder.\n    A zip is extracted to a temp folder. Prefers a folder named `run_name`, then the shallowest."""\n    import tempfile\n\n    def is_hit(n: str) -> bool:\n        n = n.replace("\\\\", "/")\n        return n == "data/sources.json" or n.endswith("/data/sources.json")\n\n    for d in paths:\n        if not d or not Path(d).exists():\n            continue\n        d = Path(d)\n        if d.is_dir():\n            runs = [p.parent.parent for p in d.rglob("sources.json") if p.parent.name == "data"]\n            if runs:\n                return min(runs, key=lambda r: (r.name != run_name, len(r.parts), str(r)))\n            zips = sorted(d.rglob("*.zip"))\n        else:\n            zips = [d] if d.suffix.lower() == ".zip" else []\n        for zp in zips:\n            try:\n                with zipfile.ZipFile(zp) as z:\n                    hits = [n.replace("\\\\", "/") for n in z.namelist() if is_hit(n)]\n                    if not hits:\n                        continue\n                    tmp = Path(tempfile.mkdtemp(prefix="lgmtp_resume_"))\n                    _unzip_safe(z, tmp)\n            except (zipfile.BadZipFile, OSError):\n                continue\n            best = min(hits, key=lambda n: (n.split("/")[-3:-2] != [run_name], n.count("/"), n))\n            return tmp / Path(best).parent.parent\n    return None\n\n\ndef week1_main(cfg: dict) -> dict:\n    cfg = {**DEFAULT_W1, **cfg}\n    root = Path(cfg["OUT_ROOT"]) / cfg["RUN_NAME"]\n    for sub in ("data", "features", "heads", "preds", "results"):\n        (root / sub).mkdir(parents=True, exist_ok=True)\n    log = Log(root / "results" / "log.txt")\n    src = cfg["RESUME_FROM"]\n    if src:\n        where = cfg["PHASE1_SEARCH"] if src == "auto" else [src]\n        run = locate_run(where, root.name)\n        if run is None:\n            if src != "auto":\n                raise FileNotFoundError(f"RESUME_FROM={src!r}: no earlier run (data/sources.json, as a folder or "\n                                        f"inside a .zip) found there")\n            log(f"resume: RESUME_FROM=\'auto\' found no earlier run (data/sources.json) in {where}; starting fresh")\n        elif run.resolve() != root.resolve():\n            _resume_copy(run, root, log)\n    try:\n        return _week1_run(cfg, root, log)\n    except BaseException as e:\n        import traceback\n        log(f"FAILED: {type(e).__name__}: {e} ({_mem()})\\n" + traceback.format_exc())\n        try:\n            _zip_tree(root / "results", Path(cfg["OUT_ROOT"]) / "week1_results.zip", root)\n            log("partial results zipped to week1_results.zip; attach this version\'s output and rerun to resume")\n        except Exception:  # noqa: BLE001\n            pass\n        raise\n\n\ndef _week1_run(cfg: dict, root: Path, log) -> dict:\n    device = "cpu" if cfg["SMOKE"] else _device()\n    log(f"week1 start | device {device} | build {cfg[\'GIT_HASH\']}")\n    (root / "results" / "config.json").write_text(json.dumps({k: v for k, v in cfg.items() if k != "SOURCES"},\n                                                             indent=1, default=str))\n    S = stage_sources(cfg, root, log)\n    V = stage_vocab(cfg, root, S, log)\n    M = stage_collect(cfg, root, S, log, device)\n    if len(M) < min(cfg["MIN_MODELS"], len(cfg["MODELS"])):\n        raise RuntimeError(f"only {len(M)} models collected (need {cfg[\'MIN_MODELS\']}); see collection_summary.csv")\n    L8 = stage_phase1(cfg, root, S, log, device) if cfg["RUN_L8"] else None\n    F, R, T, mono = stage_features(cfg, root, M, L8, V, S, log)\n    agree = stage_agree(cfg, root, M, L8, {k: T[k]["table"] for k in M}, log, device) if cfg["RUN_AGREE"] else []\n    res = stage_heads(cfg, root, F, R, T, mono, V, L8, log, device)\n    Vd = write_report(cfg, root, res, agree, log)\n    info = run_info(cfg["GIT_HASH"], models={k: m["info"] for k, m in M.items()}, l8_source=(L8 or {}).get("source"),\n                    verdict=Vd["decision"])\n    (root / "results" / "run_info.json").write_text(json.dumps(info, indent=1, default=str))\n    z1 = Path(cfg["OUT_ROOT"]) / "week1_results.zip"\n    _zip_tree(root / "results", z1, root)\n    z2 = Path(cfg["OUT_ROOT"]) / "week1_data.zip"\n    _zip_tree([root / "data", root / "heads"], z2, root)\n    log("=" * 70)\n    log(f"DONE. Verdict: {Vd[\'decision\']}")\n    log(f"Download {z1.name} (results, ~small) and unzip into results/week1/ in the repo.")\n    log(f"{z2.name} holds the collected outputs + top-20 and the trained heads (for later weeks); keep it on Kaggle "\n        f"or download into data/week1/.")\n    log("=" * 70)\n    return dict(verdict=Vd, results_zip=str(z1), data_zip=str(z2), res=res)\n\n\ndef _zip_tree(srcs, zip_path: Path, base: Path) -> None:\n    srcs = srcs if isinstance(srcs, list) else [srcs]\n    tmp = zip_path.with_name(zip_path.name + ".part")\n    with zipfile.ZipFile(tmp, "w", compression=zipfile.ZIP_DEFLATED) as z:\n        for s in srcs:\n            for f in sorted(Path(s).rglob("*")):\n                if f.is_file():\n                    z.write(f, arcname=str(f.relative_to(base)))\n    tmp.replace(zip_path)\n',
}
for _n, _s in LGMTP_SOURCES.items():
    with open(os.path.join(PKG, _n + '.py'), 'w', encoding='utf-8') as _f:
        _f.write(_s)
sys.path.insert(0, os.path.dirname(PKG))
for _m in [m for m in list(sys.modules) if m == 'lgmtp' or m.startswith('lgmtp.')]:
    del sys.modules[_m]
import lgmtp
LGMTP_BUILD = 'git=9f1203decd0b src+consts_sha256=d875cf2683fc'
print('lgmtp build', LGMTP_BUILD)

## CONFIG

In [ ]:
from lgmtp.collect import DEFAULT_MODELS, OPTIONAL_MODELS
CONFIG = dict(
    MODELS=DEFAULT_MODELS,     # + OPTIONAL_MODELS to add Qwen3-1.7B
    N_EXTRA=3000,              # extra IITB English sources per model (training data); lower for a faster run
    N_DEV=None, N_DEVTEST=None,  # None = all FLORES sentences (997 / 1012)
    MONO_N=200000,             # Hindi sentences for the unit vocabulary + text-only pretraining
    MAX_NEW=256, GEN_BATCH=32,
    PRE_STEPS=3000, FT_STEPS=4000, INMODEL_STEPS=3000, BATCH=256,
    RUN_AGREE=True, RUN_FOLDS=True, RUN_L8=True,
    COLLECT_8B_IF_MISSING=True,  # only used if the Phase-1 traces are not attached
    PHASE1_DIR=None,           # optional explicit path to the Phase-1 trace files
    C_HEAD=None,               # None = measured head latency / 8B step; or a number to override
    OUT_ROOT="/kaggle/working",
    RESUME_FROM="auto",        # "auto" = continue from an earlier run's output attached as input (if any); None = fresh
    GIT_HASH=LGMTP_BUILD,
)
# Phase-1 numbers embedded at build time (results/phase1/selected_params.json, main_table.csv, REPORT.md)
import json
CONFIG.update(json.loads('{"PHASE1_SELECTED": {"Entropy": {"lam": -0.683, "beta": 4.0}, "FixedK": {"k": 2}}, "PHASE1_MAIN": {"FixedK": {"speedup": 1.3574108550387403, "ci_lo": 1.3499349532907536, "ci_hi": 1.365743221571691, "tokens_per_round": 2.2591073747522383}, "Entropy": {"speedup": 1.4518454191605557, "ci_lo": 1.443550863328387, "ci_hi": 1.4610082986779356, "tokens_per_round": 2.478572232012298}, "Oracle": {"speedup": 1.9039539381560673, "ci_lo": 1.8948879142501218, "ci_hi": 1.9137577712129188, "tokens_per_round": 3.5773580821473026}, "MorphEntropy_ext1": {"speedup": 1.4270224453823808, "ci_lo": 1.4183201789280704, "ci_hi": 1.436941550636568, "tokens_per_round": 2.640121410599465}}, "PHASE1_TIMING_FALLBACK": {"t_target": 0.0706, "t_draft": 0.02281305201278851, "c": 0.3231310483397806, "t_target_verify": {"1": 0.07201199999999999, "2": 0.07236499999999998, "3": 0.0730004, "4": 0.07363579999999999, "5": 0.074836, "6": 0.0750478, "7": 0.0752596, "8": 0.0758244}, "source": "Phase-1 REPORT.md (fallback)"}}'))

In [ ]:
from huggingface_hub import login
try:
    from kaggle_secrets import UserSecretsClient
    login(UserSecretsClient().get_secret("HF_TOKEN"))
    print("Logged in with Kaggle secret HF_TOKEN")
except Exception as e:
    print(f"No HF_TOKEN ({e}); continuing with ungated sources and mirrors")
import torch, transformers
print("torch", torch.__version__, "| transformers", transformers.__version__, "| GPUs", torch.cuda.device_count())

In [ ]:
from lgmtp.week1 import week1_main
RESULT = week1_main(CONFIG)

In [ ]:
from pathlib import Path
from IPython.display import Markdown, display
display(Markdown(Path(CONFIG["OUT_ROOT"], "week1", "results", "SUMMARY.md").read_text(encoding="utf-8")))